# Archive of superseded cells

Kept for reference only — **you do not need to run these.** They failed, are exact duplicates, only mounted or searched Drive, or were replaced by a later version. Each one says why it is here.

**[orig #1]** archived — failed (FileNotFoundError); superseded by A1 cell 6  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import re

BASE = Path("/content/drive/MyDrive/gw_framework/data")
RAIN_DIR = BASE / "Rainfall_Rajasthan"
GW_DIR = BASE / "Rajasthan_gw_data"
EXTRACT_DIR = RAIN_DIR / "Extracted_Daily_Data"
OUT_DIR = RAIN_DIR / "Matched_Block_Rainfall"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def normalize_name(value):
    if pd.isna(value):
        return None
    value = str(value).strip().lower()
    value = re.sub(r"[\u200b\u00a0]", " ", value)
    value = re.sub(r"[_\-]+", " ", value)
    value = re.sub(r"[^\w\s]", "", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value or None


# 1. Load groundwater Excel files
gw_parts = []

for file in GW_DIR.glob("*.xlsx"):
    df = pd.read_excel(file)
    df["Source_File"] = file.name
    gw_parts.append(df)

if not gw_parts:
    raise FileNotFoundError(f"No groundwater Excel files found in {GW_DIR}")

gw = pd.concat(gw_parts, ignore_index=True, sort=False)

block_cols = [
    c for c in gw.columns
    if re.sub(r"[^a-z]", "", str(c).lower())
    in {"block", "blockname", "tehsil", "tehsilname"}
]

if not block_cols:
    raise ValueError(f"No block-name column found. Columns: {gw.columns.tolist()}")

GW_BLOCK_COL = block_cols[0]
gw["Block_original"] = gw[GW_BLOCK_COL]
gw["Block_key"] = gw["Block_original"].map(normalize_name)

valid_blocks = set(gw["Block_key"].dropna().unique())

# 2. Load extracted rainfall
rain_file = EXTRACT_DIR / "daily_rainfall_extracted.csv"

if not rain_file.exists():
    raise FileNotFoundError(
        f"Run the PDF extraction first. Missing: {rain_file}"
    )

rain = pd.read_csv(rain_file)

if "station" not in rain.columns:
    raise ValueError(
        f"'station' column not found. Available columns: {rain.columns.tolist()}"
    )

rain["Station_original"] = rain["station"]
rain["Block_key"] = rain["Station_original"].map(normalize_name)

# 3. Exact normalized name matching only
rain["Block_match"] = rain["Block_key"].isin(valid_blocks)

matched = rain[rain["Block_match"]].copy()
unmatched = rain[~rain["Block_match"]].copy()

# 4. Save matching and exclusion outputs
matched.to_csv(OUT_DIR / "matched_daily_rainfall.csv", index=False)
unmatched.to_csv(OUT_DIR / "unmatched_rainfall_excluded.csv", index=False)

# Unique station match report
station_report = (
    rain[["Station_original", "Block_key", "Block_match"]]
    .drop_duplicates()
    .sort_values(["Block_match", "Station_original"])
)

station_report.to_csv(
    OUT_DIR / "station_block_match_report.csv", index=False
)

print("Groundwater block names:", len(valid_blocks))
print("Matched rainfall rows:", len(matched))
print("Excluded rainfall rows:", len(unmatched))

print("\nStation matching report:")
display(station_report)

print("\nMatched rainfall preview:")
display(matched.head(10))

print("\nExcluded station names:")
display(
    unmatched[["Station_original"]]
    .drop_duplicates()
    .sort_values("Station_original")
)

print("\nOutputs saved in:", OUT_DIR)


**[orig #2]** archived — earlier rainfall extraction; superseded by A1 cell 6  
<sub>✅ ran OK</sub>

In [ ]:

!pip -q install pdfplumber

from pathlib import Path
import pandas as pd
import numpy as np
import pdfplumber
import re

BASE = Path("/content/drive/MyDrive/gw_framework/data")
RAIN_DIR = BASE / "Rainfall_Rajasthan"
GW_DIR = BASE / "Rajasthan_gw_data"
OUT_DIR = RAIN_DIR / "Matched_Block_Rainfall"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

def norm(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = re.sub(r"[\u200b\u00a0]", " ", x)
    x = re.sub(r"[_\-]+", " ", x)
    x = re.sub(r"[^\w\s]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x or None

def get_metadata(text):
    # Extract values from common PDF text layouts.
    def field(label, stop_labels):
        stop = "|".join(stop_labels)
        pattern = rf"{label}\s*:?\s*(.+?)(?=\s+(?:{stop})\s*:|[\r\n]|$)"
        m = re.search(pattern, text, flags=re.I)
        return m.group(1).strip(" :\t") if m else None

    station = field(
        r"STATION", ["DISTRICT", "STATE", "CATCHMENT", "LATITUDE", "LONGITUDE"]
    )
    district = field(
        r"DISTRICT", ["STATION", "STATE", "CATCHMENT", "LATITUDE", "LONGITUDE"]
    )
    catchment = field(r"CATCHMENT\s*NO\.?", ["STATION", "DISTRICT", "STATE",
                                             "LATITUDE", "LONGITUDE"])
    latitude = field(r"LATITUDE", ["STATION", "DISTRICT", "STATE",
                                   "CATCHMENT", "LONGITUDE"])
    longitude = field(r"LONGITUDE", ["STATION", "DISTRICT", "STATE",
                                     "CATCHMENT", "LATITUDE"])

    return station, district, catchment, latitude, longitude

def parse_page(page, filename, year, page_num):
    text = page.extract_text() or ""
    words = page.extract_words(x_tolerance=2, y_tolerance=3)

    station, district, catchment, lat, lon = get_metadata(text)

    # Find month-column x positions.
    month_x = {}
    for w in words:
        tok = w["text"].strip().upper().strip(".")
        if tok in MONTHS:
            month_x[MONTHS[tok]] = (w["x0"] + w["x1"]) / 2

    if len(month_x) < 10:
        return [], {
            "file": filename, "page": page_num,
            "issue": "Month headers not detected",
            "station": station
        }

    # Group PDF words by horizontal text line.
    lines = {}
    for w in words:
        lines.setdefault(round(w["top"], 1), []).append(w)

    records = []
    first_month_x = min(month_x.values())
    typical_gap = np.median(np.diff(sorted(month_x.values())))

    for y, line in sorted(lines.items()):
        line = sorted(line, key=lambda w: w["x0"])

        day_words = [
            w for w in line
            if w["x0"] < first_month_x - 5
            and re.fullmatch(r"\d{1,2}", w["text"].strip())
        ]
        if not day_words:
            continue

        day = int(day_words[0]["text"])
        if day < 1 or day > 31:
            continue

        for w in line:
            center = (w["x0"] + w["x1"]) / 2
            if center < first_month_x - 5:
                continue

            token = w["text"].strip().replace(",", "")
            if not re.fullmatch(r"-?\d+(?:\.\d+)?", token):
                continue

            month = min(month_x, key=lambda m: abs(month_x[m] - center))
            if abs(month_x[month] - center) > typical_gap * 0.48:
                continue

            try:
                date = pd.Timestamp(year=year, month=month, day=day)
            except ValueError:
                continue

            records.append({
                "date": date,
                "year": year,
                "month": month,
                "day": day,
                "station": station,
                "district": district,
                "catchment_no": catchment,
                "latitude_raw": lat,
                "longitude_raw": lon,
                "rainfall_mm": float(token),
                "source_file": filename,
                "page_number": page_num
            })

    # Flag duplicate day/month entries before keeping one.
    frame = pd.DataFrame(records)
    issue = None
    if not frame.empty and frame.duplicated(["date"]).any():
        issue = {
            "file": filename,
            "page": page_num,
            "issue": "Duplicate date values on this page",
            "station": station
        }
        frame = frame.drop_duplicates(["date"], keep="first")

    return frame.to_dict("records"), issue

# ------------------------------------------------------------
# Load groundwater block names
# ------------------------------------------------------------
gw_files = list(GW_DIR.glob("*.xlsx"))
if not gw_files:
    raise FileNotFoundError(f"No groundwater Excel files found: {GW_DIR}")

gw_list = []
for f in gw_files:
    df = pd.read_excel(f)
    df["source_groundwater_file"] = f.name
    gw_list.append(df)

gw = pd.concat(gw_list, ignore_index=True, sort=False)

block_cols = [
    c for c in gw.columns
    if re.sub(r"[^a-z]", "", str(c).lower())
    in {"block", "blockname", "tehsil", "tehsilname"}
]
if not block_cols:
    raise ValueError(f"Block column not found. Columns: {gw.columns.tolist()}")

block_col = block_cols[0]
gw["Block_key"] = gw[block_col].map(norm)
valid_blocks = set(gw["Block_key"].dropna().unique())

print("Groundwater block field:", block_col)
print("Unique normalized groundwater blocks:", len(valid_blocks))

# ------------------------------------------------------------
# Extract all annual PDFs
# ------------------------------------------------------------
pdf_files = sorted(
    RAIN_DIR.glob("*.pdf"),
    key=lambda p: int(re.search(r"\d{4}", p.stem).group())
    if re.search(r"\d{4}", p.stem) else 9999
)

all_records, all_issues = [], []

for pdf_path in pdf_files:
    match = re.search(r"\d{4}", pdf_path.stem)
    if not match:
        print("Skipping file without year in name:", pdf_path.name)
        continue

    year = int(match.group())
    print("Reading:", pdf_path.name)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                records, issue = parse_page(
                    page, pdf_path.name, year, page_num
                )
                all_records.extend(records)
                if issue:
                    all_issues.append(issue)
    except Exception as e:
        all_issues.append({
            "file": pdf_path.name, "page": None,
            "issue": str(e), "station": None
        })

rain = pd.DataFrame(all_records)

if rain.empty:
    raise ValueError(
        "No daily records extracted. We need to inspect the PDF text layout "
        "or use OCR before matching."
    )

rain["Station_key"] = rain["station"].map(norm)
rain["Block_match"] = rain["Station_key"].isin(valid_blocks)

# Save complete extraction before exclusion.
rain.to_csv(OUT_DIR / "all_extracted_daily_rainfall.csv", index=False)

# Exact station-name/block-name match only.
matched = rain[rain["Block_match"]].copy()
unmatched = rain[~rain["Block_match"]].copy()

matched.to_csv(OUT_DIR / "matched_daily_rainfall.csv", index=False)
unmatched.to_csv(OUT_DIR / "unmatched_rainfall_excluded.csv", index=False)

station_report = (
    rain[["station", "Station_key", "Block_match"]]
    .drop_duplicates()
    .sort_values(["Block_match", "station"])
)
station_report.to_csv(OUT_DIR / "station_block_match_report.csv", index=False)

pd.DataFrame(all_issues).to_csv(
    OUT_DIR / "extraction_issues.csv", index=False
)

print("\n========== RESULTS ==========")
print("Extracted daily records:", len(rain))
print("Matched daily records:", len(matched))
print("Excluded daily records:", len(unmatched))
print("Matched stations:", matched["station"].nunique())
print("Excluded stations:", unmatched["station"].nunique())

print("\nStation match report:")
display(station_report)

print("\nMatched data preview:")
display(matched.head(20))

print("\nExtraction issues:")
display(pd.DataFrame(all_issues).head(30))

print("\nAll outputs saved in:", OUT_DIR)


**[orig #4]** archived — failed; superseded by A1 cell 6  
<sub>⚠ last run ended with AttributeError</sub>

In [ ]:

!pip -q install pdfplumber

from pathlib import Path
import pandas as pd
import numpy as np
import pdfplumber
import re
import calendar

BASE = Path("/content/drive/MyDrive/gw_framework/data")
RAIN_DIR = BASE / "Rainfall_Rajasthan"
GW_DIR = BASE / "Rajasthan_gw_data"
OUT_DIR = RAIN_DIR / "Corrected_Rainfall_Extraction"
OUT_DIR.mkdir(parents=True, exist_ok=True)

MONTHS = {
    "JAN": 1, "FEB": 2, "MAR": 3, "APR": 4,
    "MAY": 5, "JUN": 6, "JUL": 7, "AUG": 8,
    "SEP": 9, "OCT": 10, "NOV": 11, "DEC": 12
}

def norm_name(x):
    if pd.isna(x):
        return None
    x = str(x).strip().lower()
    x = re.sub(r"[\u200b\u00a0]", " ", x)
    x = re.sub(r"[_\-]+", " ", x)
    x = re.sub(r"[^\w\s]", "", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x or None

def get_field(text, label, stop_labels):
    stops = "|".join(stop_labels)
    pattern = rf"{label}\s*:?\s*(.+?)(?=\s+(?:{stops})\s*:|[\r\n]|$)"
    m = re.search(pattern, text, re.I)
    return m.group(1).strip(" :\t") if m else None

def extract_page(text, filename, year, page_no):
    records = []
    issues = []

    station = get_field(
        text, r"STATION", ["DISTRICT", "STATE", "CATCHMENT", "LATITUDE", "LONGITUDE"]
    )
    district = get_field(
        text, r"DISTRICT", ["STATION", "STATE", "CATCHMENT", "LATITUDE", "LONGITUDE"]
    )
    catchment = get_field(
        text, r"CATCHMENT\s*NO\.?", ["STATION", "DISTRICT", "STATE", "LATITUDE", "LONGITUDE"]
    )
    latitude = get_field(
        text, r"LATITUDE", ["STATION", "DISTRICT", "STATE", "CATCHMENT", "LONGITUDE"]
    )
    longitude = get_field(
        text, r"LONGITUDE", ["STATION", "DISTRICT", "STATE", "CATCHMENT", "LATITUDE"]
    )

    # Extract the printed monthly rainfall summary for validation.
    monthly_reported = {}
    monthly_match = re.search(
        r"MONTHLY\s+RAINFALL\s+(.+?)(?=\s+RAINYDAYS\b)",
        text, flags=re.I | re.S
    )

    if monthly_match:
        vals = re.findall(r"-?\d+(?:\.\d+)?", monthly_match.group(1))
        if len(vals) == 12:
            monthly_reported = {
                m: float(v) for m, v in zip(range(1, 13), vals)
            }

    annual_match = re.search(
        r"ANNUAL\s+RAINFALL\s*=\s*(-?\d+(?:\.\d+)?)",
        text, flags=re.I
    )
    annual_reported = float(annual_match.group(1)) if annual_match else np.nan

    # Read the daily table between the header and the MONTHLY summary.
    table_match = re.search(
        r"\bDATE\s+JAN\s+FEB\s+MAR\s+APR\s+MAY\s+JUN\s+JUL\s+"
        r"AUG\s+SEP\s+OCT\s+NOV\s+DEC\b(.*?)\bMONTHLY\b",
        text, flags=re.I | re.S
    )

    if not table_match:
        return [], {
            "file": filename, "page": page_no, "station": station,
            "issue": "Daily table section not detected",
            "monthly_reported": monthly_reported,
            "annual_reported": annual_reported
        }

    table_text = table_match.group(1)

    for line in table_text.splitlines():
        line = line.strip()
        if not line:
            continue

        # A daily line must start with day number followed by rainfall values.
        match = re.match(r"^(\d{1,2})\s+(.+)$", line)
        if not match:
            continue

        day = int(match.group(1))
        if not 1 <= day <= 31:
            continue

        vals = re.findall(r"-?\d+(?:\.\d+)?", match.group(2))
        valid_months = [
            m for m in range(1, 13)
            if day <= calendar.monthrange(year, m)[1]
        ]

        if len(vals) != len(valid_months):
            issues.append({
                "file": filename,
                "page": page_no,
                "station": station,
                "issue": f"Day {day}: found {len(vals)} values; expected {len(valid_months)}",
                "details": line
            })
            continue

        for month, val in zip(valid_months, vals):
            records.append({
                "date": pd.Timestamp(year=year, month=month, day=day),
                "year": year,
                "month": month,
                "day": day,
                "station": station,
                "district": district,
                "catchment_no": catchment,
                "latitude_raw": latitude,
                "longitude_raw": longitude,
                "rainfall_mm": float(val),
                "source_file": filename,
                "page_number": page_no
            })

    # Validate monthly and annual rainfall summaries.
    if records:
        temp = pd.DataFrame(records)
        monthly_extracted = temp.groupby("month")["rainfall_mm"].sum()

        for month, reported in monthly_reported.items():
            extracted = float(monthly_extracted.get(month, 0.0))
            if not np.isclose(extracted, reported, atol=0.15):
                issues.append({
                    "file": filename,
                    "page": page_no,
                    "station": station,
                    "issue": f"Month {month} total mismatch",
                    "details": f"Extracted={extracted}; PDF={reported}"
                })

        annual_extracted = temp["rainfall_mm"].sum()
        if pd.notna(annual_reported) and not np.isclose(
            annual_extracted, annual_reported, atol=0.2
        ):
            issues.append({
                "file": filename,
                "page": page_no,
                "station": station,
                "issue": "Annual total mismatch",
                "details": f"Extracted={annual_extracted}; PDF={annual_reported}"
            })

    return records, issues


# ------------------------------------------------------------
# 1. Read groundwater block names
# ------------------------------------------------------------
gw_files = sorted(GW_DIR.glob("*.xlsx"))
if not gw_files:
    raise FileNotFoundError(f"No groundwater Excel files found: {GW_DIR}")

gw_parts = []
for f in gw_files:
    g = pd.read_excel(f)
    gw_parts.append(g)

gw = pd.concat(gw_parts, ignore_index=True, sort=False)

block_candidates = [
    c for c in gw.columns
    if re.sub(r"[^a-z]", "", str(c).lower())
    in {"block", "blockname", "tehsil", "tehsilname"}
]
if not block_candidates:
    raise ValueError(f"No block column found: {gw.columns.tolist()}")

block_col = block_candidates[0]
valid_blocks = set(gw[block_col].map(norm_name).dropna().unique())
print("Groundwater block column:", block_col)
print("Unique normalized blocks:", len(valid_blocks))


# ------------------------------------------------------------
# 2. Extract all annual PDFs
# ------------------------------------------------------------
pdf_files = sorted(
    RAIN_DIR.glob("*.pdf"),
    key=lambda p: int(re.search(r"\d{4}", p.stem).group())
    if re.search(r"\d{4}", p.stem) else 9999
)

all_records, all_issues = [], []

for pdf_path in pdf_files:
    year_match = re.search(r"\d{4}", pdf_path.stem)
    if not year_match:
        continue

    year = int(year_match.group())
    print("Processing", pdf_path.name)

    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_no, page in enumerate(pdf.pages, start=1):
                text = page.extract_text() or ""
                records, issues = extract_page(
                    text, pdf_path.name, year, page_no
                )
                all_records.extend(records)
                all_issues.extend(issues)

    except Exception as e:
        all_issues.append({
            "file": pdf_path.name,
            "page": None,
            "station": None,
            "issue": "PDF error",
            "details": str(e)
        })

rain = pd.DataFrame(all_records)
if rain.empty:
    raise ValueError(
        "No daily rainfall records extracted. Check the PDF table format."
    )

# Save all extracted records before any station exclusion.
rain["Station_key"] = rain["station"].map(norm_name)
rain["Block_match"] = rain["Station_key"].isin(valid_blocks)

rain.to_csv(OUT_DIR / "all_daily_rainfall.csv", index=False)

matched = rain[rain["Block_match"]].copy()
excluded = rain[~rain["Block_match"]].copy()

matched.to_csv(OUT_DIR / "matched_daily_rainfall.csv", index=False)
excluded.to_csv(OUT_DIR / "unmatched_stations_excluded.csv", index=False)

pd.DataFrame(all_issues).to_csv(
    OUT_DIR / "extraction_validation_issues.csv", index=False
)

station_report = (
    rain[["station", "Station_key", "Block_match"]]
    .drop_duplicates()
    .sort_values(["Block_match", "station"])
)
station_report.to_csv(OUT_DIR / "station_block_match_report.csv", index=False)

print("\n========== FINAL EXTRACTION SUMMARY ==========")
print("Extracted daily rows:", len(rain))
print("Matched daily rows:", len(matched))
print("Excluded daily rows:", len(excluded))
print("Matched stations:", matched["station"].nunique())
print("Excluded stations:", excluded["station"].nunique())
print("Validation issues:", len(all_issues))

print("\nMatched stations:")
display(station_report[station_report["Block_match"]].head(50))

print("\nValidation issue sample:")
display(pd.DataFrame(all_issues).head(30))

print("\nOutput folder:", OUT_DIR)


**[orig #5]** archived — failed (IndentationError); superseded by A1 cell 6  
<sub>⚠ last run ended with IndentationError</sub>

In [ ]:

# ============================================================
# RAJASTHAN RAINFALL PDF EXTRACTION FOR GROUNDWATER BLOCKS
# Complete Google Colab script
#
# Rainfall source:
#   gw_framework/data/Rainfall_Rajasthan/*.pdf
#
# Groundwater source:
#   gw_framework/data/Rajasthan_gw_data/*.xlsx
#
# RULES:
# 1. Station name must match groundwater block name.
# 2. Matching is case-insensitive.
# 3. Asterisks and punctuation are ignored for matching only.
# 4. District names/numbers are NOT used for matching.
# 5. Unmatched station pages are skipped before daily extraction.
# 6. Original station names and rainfall values are preserved.
# 7. Daily data are validated against PDF monthly/annual totals.
# 8. Original source files are never modified.
# ============================================================

!pip -q install pdfplumber openpyxl

from pathlib import Path
import pandas as pd
import numpy as np
import pdfplumber
import re
import calendar
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. DEFINE FOLDERS
# ------------------------------------------------------------

BASE = Path("/content/drive/MyDrive/gw_framework/data")

RAIN_DIR = BASE / "Rainfall_Rajasthan"
GW_DIR = BASE / "Rajasthan_gw_data"

OUT_DIR = RAIN_DIR / "Matched_Block_Rainfall"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAIN_DIR.exists():
    raise FileNotFoundError(f"Rainfall folder not found: {RAIN_DIR}")

if not GW_DIR.exists():
    raise FileNotFoundError(f"Groundwater folder not found: {GW_DIR}")

print("Rainfall folder:", RAIN_DIR)
print("Groundwater folder:", GW_DIR)
print("Output folder:", OUT_DIR)

# ------------------------------------------------------------
# 2. NORMALIZE NAMES FOR MATCHING ONLY
# ------------------------------------------------------------

def norm_name(value):
    if pd.isna(value):
        return None

    value = str(value).strip().lower()

    # Ignore asterisks, including station names such as ABU ROAD*
    value = value.replace("*", " ")

    # Normalize unusual spaces
    value = re.sub(r"[\u200b\u00a0]", " ", value)

    # Treat hyphens and underscores as spaces
    value = re.sub(r"[_\-]+", " ", value)

    # Ignore other punctuation
    value = re.sub(r"[^\w\s]", " ", value)

    # Collapse repeated spaces
    value = re.sub(r"\s+", " ", value).strip()

    return value or None


# ------------------------------------------------------------
# 3. EXTRACT A FIELD FROM PDF HEADER TEXT
# ------------------------------------------------------------

def get_field(text, label, stop_labels):
    stops = "|".join(stop_labels)

    pattern = (
        rf"{label}\s*:?\s*(.+?)"
        rf"(?=\s+(?:{stops})\s*:|[\r\n]|$)"
    )

    match = re.search(pattern, text, flags=re.I)

    return match.group(1).strip(" :\t") if match else None


# ------------------------------------------------------------
# 4. EXTRACT DAILY RAINFALL FROM A MATCHED PAGE
# ------------------------------------------------------------

def extract_matched_page(text, filename, year, page_number):

    records = []
    issues = []

    # Station and descriptive metadata
    station = get_field(
        text, r"STATION",
        ["DISTRICT", "STATE", "CATCHMENT", "LATITUDE", "LONGITUDE"]
    )

    district = get_field(
        text, r"DISTRICT",
        ["STATION", "STATE", "CATCHMENT", "LATITUDE", "LONGITUDE"]
    )

    state = get_field(
        text, r"STATE",
        ["STATION", "DISTRICT", "CATCHMENT", "LATITUDE", "LONGITUDE"]
    )

    catchment = get_field(
        text, r"CATCHMENT\s*NO\.?",
        ["STATION", "DISTRICT", "STATE", "LATITUDE", "LONGITUDE"]
    )

    latitude = get_field(
        text, r"LATITUDE",
        ["STATION", "DISTRICT", "STATE", "CATCHMENT", "LONGITUDE"]
    )

    longitude = get_field(
        text, r"LONGITUDE",
        ["STATION", "DISTRICT", "STATE", "CATCHMENT", "LATITUDE"]
    )

    # --------------------------------------------------------
    # Read monthly summary printed in the PDF
    # --------------------------------------------------------

    monthly_reported = {}

    monthly_match = re.search(
        r"MONTHLY\s+RAINFALL\s+(.+?)(?=\s+RAINYDAYS\b)",
        text,
        flags=re.I | re.S
    )

    if monthly_match:
        values = re.findall(
            r"-?\d+(?:\.\d+)?",
            monthly_match.group(1)
        )

        if len(values) == 12:
            monthly_reported = {
                month: float(value)
                for month, value in zip(range(1, 13), values)
            }
        else:
            issues.append({
                "source_file": filename,
                "page_number": page_number,
                "station": station,
                "issue": "Monthly summary does not contain 12 values",
                "details": str(values)
            })

    # Annual rainfall total printed in the PDF
    annual_match = re.search(
        r"ANNUAL\s+RAINFALL\s*=\s*(-?\d+(?:\.\d+)?)",
        text,
        flags=re.I
    )

    annual_reported = (
        float(annual_match.group(1))
        if annual_match else np.nan
    )

    # --------------------------------------------------------
    # Locate the daily rainfall table
    # --------------------------------------------------------

    table_match = re.search(
        r"\bDATE\s+JAN\s+FEB\s+MAR\s+APR\s+MAY\s+JUN\s+JUL\s+"
        r"AUG\s+SEP\s+OCT\s+NOV\s+DEC\b"
        r"(.*?)\bMONTHLY\b",
        text,
        flags=re.I | re.S
    )

    if not table_match:
        issues.append({
            "source_file": filename,
            "page_number": page_number,
            "station": station,
            "issue": "Daily table not detected",
            "details": ""
        })

        return records, issues, {
            "station": station,
            "district": district,
            "state": state,
            "catchment_no": catchment,
            "latitude_raw": latitude,
            "longitude_raw": longitude,
            "year": year,
            "source_file": filename,
            "page_number": page_number,
            "daily_record_count": 0,
            "daily_total_mm": np.nan,
            "pdf_annual_total_mm": annual_reported,
            "annual_difference_mm": np.nan
        }

    table_text = table_match.group(1)

    # Each line contains day number followed by the monthly values
    for line in table_text.splitlines():

        line = line.strip()

        if not line:
            continue

        row_match = re.match(r"^(\d{1,2})\s+(.+)$", line)

        if not row_match:
            continue

        day = int(row_match.group(1))

        if not 1 <= day <= 31:
            continue

        values = re.findall(
            r"-?\d+(?:\.\d+)?",
            row_match.group(2)
        )

        # Only valid calendar months have a value for this day.
        valid_months = [
            month for month in range(1, 13)
            if day <= calendar.monthrange(year, month)[1]
        ]

        # Never shift values if a row has an unexpected number.
        if len(values) != len(valid_months):

            issues.append({
                "source_file": filename,
                "page_number": page_number,
                "station": station,
                "issue": f"Incorrect value count for day {day}",
                "details": (
                    f"Found {len(values)} values; "
                    f"expected {len(valid_months)}. "
                    f"Row: {line}"
                )
            })

            continue

        for month, value in zip(valid_months, values):

            records.append({
                "date": pd.Timestamp(
                    year=year, month=month, day=day
                ),
                "year": year,
                "month": month,
                "day": day,

                # Preserve the station name exactly as extracted.
                "station": station,

                # Informational metadata; not matching keys.
                "district": district,
                "state": state,
                "catchment_no": catchment,
                "latitude_raw": latitude,
                "longitude_raw": longitude,

                "rainfall_mm": float(value),
                "source_file": filename,
                "page_number": page_number
            })

    # --------------------------------------------------------
    # Validate daily totals against the printed summaries
    # --------------------------------------------------------

    annual_extracted = np.nan

    if records:

        page_df = pd.DataFrame(records)

        annual_extracted = page_df["rainfall_mm"].sum()

        monthly_extracted = (
            page_df.groupby("month")["rainfall_mm"].sum()
        )

        for month, reported in monthly_reported.items():

            extracted = float(
                monthly_extracted.get(month, 0.0)
            )

            if not np.isclose(extracted, reported, atol=0.15):

                issues.append({
                    "source_file": filename,
                    "page_number": page_number,
                    "station": station,
                    "issue": f"Month {month} total mismatch",
                    "details": (
                        f"Extracted={extracted}; "
                        f"PDF={reported}"
                    )
                })

        if (
            pd.notna(annual_reported)
            and not np.isclose(
                annual_extracted, annual_reported, atol=0.2
            )
        ):

            issues.append({
                "source_file": filename,
                "page_number": page_number,
                "station": station,
                "issue": "Annual rainfall total mismatch",
                "details": (
                    f"Extracted={annual_extracted}; "
                    f"PDF={annual_reported}"
                )
            })

    page_summary = {
        "station": station,
        "district": district,
        "state": state,
        "catchment_no": catchment,
        "latitude_raw": latitude,
        "longitude_raw": longitude,
        "year": year,
        "source_file": filename,
        "page_number": page_number,
        "daily_record_count": len(records),
        "daily_total_mm": annual_extracted,
        "pdf_annual_total_mm": annual_reported,
        "annual_difference_mm": (
            annual_extracted - annual_reported
            if pd.notna(annual_extracted)
            and pd.notna(annual_reported)
            else np.nan
        )
    }

    return records, issues, page_summary


# ------------------------------------------------------------
# 5. LOAD GROUNDWATER BLOCK NAMES
# ------------------------------------------------------------

gw_files = sorted(GW_DIR.glob("*.xlsx"))

if not gw_files:
    raise FileNotFoundError(
        f"No groundwater Excel files found in {GW_DIR}"
    )

gw_parts = []

for file in gw_files:

    # These files are read-only; no source files are changed.
    df = pd.read_excel(file)
    df["Groundwater_Source_File"] = file.name
    gw_parts.append(df)

gw = pd.concat(
    gw_parts,
    ignore_index=True,
    sort=False
)

block_candidates = [
    col for col in gw.columns
    if re.sub(r"[^a-z]", "", str(col).lower())
    in {"block", "blockname", "tehsil", "tehsilname"}
]

if not block_candidates:
    raise ValueError(
        "Could not identify groundwater block column. "
        f"Available columns: {gw.columns.tolist()}"
    )

block_col = block_candidates[0]

gw["Block_original"] = gw[block_col]
gw["Block_key"] = gw["Block_original"].map(norm_name)

valid_blocks = set(
    gw["Block_key"].dropna().unique()
)

print("\nGroundwater block column:", block_col)
print("Unique normalized groundwater blocks:", len(valid_blocks))


# ------------------------------------------------------------
# 6. SCAN PDF METADATA FIRST; EXTRACT ONLY MATCHING STATIONS
# ------------------------------------------------------------

pdf_files = sorted(
    RAIN_DIR.glob("*.pdf"),
    key=lambda p: (
        int(re.search(r"\d{4}", p.stem).group())
        if re.search(r"\d{4}", p.stem)
        else 9999
    )
)

if not pdf_files:
    raise FileNotFoundError(
        f"No annual PDF files found in {RAIN_DIR}"
    )

 all_records = []


**[orig #29]** archived — MODFLOW install failed; replaced by A6 install cell  
<sub>⚠ last run ended with RuntimeError</sub>

In [ ]:

# ================================================================
# ALWAR GROUNDWATER MODEL — FIRST-PASS MODFLOW 6 / FLOPY
# Google Colab | One-layer equivalent porous-medium model
#
# IMPORTANT:
# This is a numerical prototype, not a calibrated Alwar model.
# Clearly marked provisional values must be replaced by verified
# GIS, aquifer, recharge, pumping and groundwater-level data.
# ================================================================

import os
import sys
import json
import subprocess
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------- 1. INSTALLATION -------------------------

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "flopy", "modflow-devtools"
])

import flopy

print("FloPy version:", flopy.__version__)

# Download MODFLOW 6 executable.
# The official MODFLOW 6 release is obtained through get-modflow.
install_result = subprocess.run(
    [
        sys.executable, "-m", "get_modflow",
        "--repo", "modflow6",
        "--release", "latest",
        "--ostag", "linux",
        "--bindir", "/content/modflow6_bin"
    ],
    capture_output=True,
    text=True
)

if install_result.returncode != 0:
    print(install_result.stdout[-3000:])
    print(install_result.stderr[-3000:])
    raise RuntimeError(
        "MODFLOW 6 installation failed. Check the installation "
        "output above and retry the installation cell."
    )

mf6_candidates = list(Path("/content/modflow6_bin").rglob("mf6"))
if not mf6_candidates:
    mf6_candidates = list(Path("/content/modflow6_bin").rglob("mf6.exe"))

if not mf6_candidates:
    raise FileNotFoundError(
        "MODFLOW 6 executable was not found after installation."
    )

mf6_exe = str(mf6_candidates[0])
print("MODFLOW executable:", mf6_exe)

# ----------------------- 2. OUTPUT FOLDER ------------------------

# Creates a new folder; does not overwrite the source datasets.
OUT = Path("/content/Alwar_MODFLOW6_FirstPass")
OUT.mkdir(parents=True, exist_ok=True)

# ----------------------- 3. MODEL SETTINGS -----------------------

# Numerical grid: illustrative rectangular domain, NOT a GIS boundary.
nlay = 1
nrow = 20
ncol = 20
delr = 1000.0       # cell width, m
delc = 1000.0       # cell height, m

# Illustrative vertical geometry.
top_elevation = 100.0
bottom_elevation = 50.0
aquifer_thickness = top_elevation - bottom_elevation

# CGWB Alwar report: alluvial aquifer transmissivity range.
T_min = 27.0
T_max = 662.0
T_initial = (T_min + T_max) / 2.0

# Derived K assumes the full 50 m model layer is the transmissive
# thickness. This is a simplifying assumption, not a measured K.
K_initial = T_initial / aquifer_thickness
K_min_scenario = T_min / aquifer_thickness
K_max_scenario = T_max / aquifer_thickness

# Provisional parameters, not site-calibrated.
Sy_initial = 0.10
Ss_initial = 1.0e-5     # 1/m; provisional and not report-derived
recharge_mm_year = 100.0
recharge_m_day = recharge_mm_year / 1000.0 / 365.25

# Illustrative hydraulic-head boundary conditions.
west_head = 100.0
east_head = 95.0
initial_head = 97.5

# Twelve monthly-like periods of 30 days each: 360-day model year.
# Replace with actual calendar-based periods for research simulations.
nper = 12
period_length_days = 30.0
nstp = 1
tsmult = 1.0

# ----------------------- 4. SAVE PARAMETER REGISTER --------------

parameter_rows = [
    ["Transmissivity_min", T_min, "m2/day",
     "CGWB Alwar aquifer-mapping report", "Reported range"],
    ["Transmissivity_max", T_max, "m2/day",
     "CGWB Alwar aquifer-mapping report", "Reported range"],
    ["Transmissivity_initial", T_initial, "m2/day",
     "Midpoint of reported range", "Initial scenario only"],
    ["Aquifer_thickness", aquifer_thickness, "m",
     "Illustrative model geometry", "Assumed; replace with mapped geometry"],
    ["Hydraulic_conductivity_initial", K_initial, "m/day",
     "T_initial / assumed thickness", "Derived; not measured"],
    ["Hydraulic_conductivity_min_scenario", K_min_scenario, "m/day",
     "T_min / assumed thickness", "Scenario bound; not measured"],
    ["Hydraulic_conductivity_max_scenario", K_max_scenario, "m/day",
     "T_max / assumed thickness", "Scenario bound; not measured"],
    ["Specific_yield", Sy_initial, "dimensionless",
     "No site-specific value assigned", "Provisional sensitivity value"],
    ["Specific_storage", Ss_initial, "1/m",
     "No site-specific value assigned", "Provisional sensitivity value"],
    ["Recharge", recharge_mm_year, "mm/year",
     "No calibrated recharge rate assigned", "Provisional sensitivity value"],
    ["West_boundary_head", west_head, "m model datum",
     "Illustrative boundary", "Assumed; replace with observed or justified head"],
    ["East_boundary_head", east_head, "m model datum",
     "Illustrative boundary", "Assumed; replace with observed or justified head"],
    ["Cell_width", delr, "m", "Numerical design", "Assumed"],
    ["Cell_height", delc, "m", "Numerical design", "Assumed"],
]

parameter_df = pd.DataFrame(
    parameter_rows,
    columns=["parameter", "value", "units", "basis", "status"]
)
parameter_df.to_csv(OUT / "parameter_register.csv", index=False)

print("\nInitial hydraulic parameters:")
print(parameter_df.to_string(index=False))

# ----------------------- 5. BUILD MODEL --------------------------

sim_name = "alwar_firstpass"
model_name = "alwar_gwf"
sim_ws = str(OUT / "simulation")
Path(sim_ws).mkdir(parents=True, exist_ok=True)

sim = flopy.mf6.MFSimulation(
    sim_name=sim_name,
    sim_ws=sim_ws,
    exe_name=mf6_exe
)

# Simulation timing
flopy.mf6.ModflowTdis(
    sim,
    time_units="DAYS",
    nper=nper,
    perioddata=[
        (period_length_days, nstp, tsmult)
        for _ in range(nper)
    ]
)

# Solver
ims = flopy.mf6.ModflowIms(
    sim,
    print_option="SUMMARY",
    complexity="MODERATE",
    outer_dvclose=1.0e-4,
    outer_maximum=100,
    inner_maximum=300,
    inner_dvclose=1.0e-5,
    linear_acceleration="BICGSTAB"
)

# Groundwater flow model
gwf = flopy.mf6.ModflowGwf(
    sim,
    modelname=model_name,
    save_flows=True,
    newtonoptions="NEWTON UNDER_RELAXATION"
)

sim.register_ims_package(ims, [gwf.name])

# Grid elevations
top = np.full((nrow, ncol), top_elevation, dtype=float)
botm = np.full((nlay, nrow, ncol), bottom_elevation, dtype=float)

flopy.mf6.ModflowGwfdis(
    gwf,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top=top,
    botm=botm,
    length_units="METERS"
)

# Unconfined initial head. Values are below model top and above bottom.
strt = np.full((nlay, nrow, ncol), initial_head, dtype=float)

flopy.mf6.ModflowGwfic(gwf, strt=strt)

# NPF: horizontal conductivity and convertible (unconfined) layer.
k_array = np.full((nlay, nrow, ncol), K_initial, dtype=float)

flopy.mf6.ModflowGwfnpf(
    gwf,
    icelltype=1,
    k=k_array,
    save_specific_discharge=True,
    save_flows=True
)

# Storage: use specific yield for unconfined behavior.
# Specific storage is retained as a provisional value for completeness.
flopy.mf6.ModflowGwfsto(
    gwf,
    iconvert=1,
    ss=Ss_initial,
    sy=Sy_initial,
    transient={kper: True for kper in range(nper)},
    steady_state={}
)

# Recharge: same provisional rate in all cells and periods.
# Replace this with spatially and temporally varying recharge
# calculated from observed climate and a defensible recharge method.
rch_data = {
    kper: recharge_m_day * np.ones((nrow, ncol), dtype=float)
    for kper in range(nper)
}

flopy.mf6.ModflowGwfrcha(
    gwf,
    recharge=rch_data,
    pname="RECHARGE"
)

# Constant-head boundaries on west and east edges.
# These are artificial numerical boundaries until justified by
# regional heads, rivers, basin boundaries, or other hydrogeology.
chd_spd = []
for kper in range(nper):
    for i in range(nrow):
        chd_spd.append([(0, i, 0), west_head])
        chd_spd.append([(0, i, ncol - 1), east_head])

flopy.mf6.ModflowGwfchd(
    gwf,
    stress_period_data=chd_spd,
    pname="ILLUSTRATIVE_CHD"
)

# No pumping wells are assigned because verified pumping rates,
# screen depths, coordinates and time periods are not yet available.

# Output control
flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=f"{model_name}.hds",
    budget_filerecord=f"{model_name}.cbc",
    saverecord=[
        ("HEAD", "ALL"),
        ("BUDGET", "ALL")
    ],
    printrecord=[
        ("HEAD", "LAST"),
        ("BUDGET", "LAST")
    ]
)

# ----------------------- 6. WRITE AND RUN ------------------------

sim.write_simulation()

success, report = sim.run_simulation(
    silent=False,
    report=True
)

print("\nMODFLOW 6 run successful:", success)
if report:
    print("\n".join(report[-20:]))

if not success:
    raise RuntimeError(
        "MODFLOW 6 did not converge or did not complete. "
        "Review the solver output in the cell above and the mfsim.lst file."
    )

# ----------------------- 7. READ OUTPUTS --------------------------

head_file = Path(sim_ws) / f"{model_name}.hds"
budget_file = Path(sim_ws) / f"{model_name}.cbc"

if not head_file.exists():
    raise FileNotFoundError(f"Head output not found: {head_file}")

hds = flopy.utils.HeadFile(str(head_file))
times = hds.get_times()
head_final = hds.get_data(totim=times[-1])

# Convert dry/inactive output values to NaN for plotting and summaries.
head_plot = np.where(
    np.isfinite(head_final) & (np.abs(head_final) < 1.0e20),
    head_final,
    np.nan
)

head_df = pd.DataFrame(head_plot[0])
head_df.to_csv(OUT / "final_head_grid.csv", index=False)

summary = {
    "model_name": model_name,
    "simulation_success": bool(success),
    "stress_periods": nper,
    "total_simulated_days": float(times[-1]),
    "final_head_min_m": float(np.nanmin(head_plot)),
    "final_head_max_m": float(np.nanmax(head_plot)),
    "final_head_mean_m": float(np.nanmean(head_plot)),
    "K_initial_m_day": float(K_initial),
    "recharge_mm_year": float(recharge_mm_year),
    "pumping_assigned": False,
    "calibrated": False,
    "domain_is_real_Alwar_GIS_boundary": False
}

with open(OUT / "run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nRun summary:")
print(json.dumps(summary, indent=2))

# ----------------------- 8. PLOT HEADS ----------------------------

plt.figure(figsize=(8, 6))
im = plt.imshow(
    head_plot[0],
    origin="upper",
    extent=[0, ncol * delr, nrow * delc, 0],
    aspect="equal"
)
plt.colorbar(im, label="Simulated hydraulic head (m; illustrative datum)")
plt.contour(
    np.linspace(delr/2, ncol*delr-delr/2, ncol),
    np.linspace(delc/2, nrow*delc-delc/2, nrow),
    head_plot[0],
    colors="black",
    linewidths=0.5
)
plt.xlabel("Model X distance (m; local grid)")
plt.ylabel("Model Y distance (m; local grid)")
plt.title("Alwar First-Pass Model: Final Simulated Head")
plt.tight_layout()
plt.savefig(OUT / "final_head_map.png", dpi=200)
plt.show()

# ----------------------- 9. SENSITIVITY SCENARIO REGISTER ---------

# This table defines useful first tests; it does not automatically
# claim these scenarios have been calibrated.
sensitivity = pd.DataFrame([
    ["Low T-derived K", K_min_scenario, recharge_mm_year,
     "K = 27 / 50; provisional thickness assumption"],
    ["Initial T-derived K", K_initial, recharge_mm_year,
     "K = midpoint(27, 662) / 50"],
    ["High T-derived K", K_max_scenario, recharge_mm_year,
     "K = 662 / 50; provisional thickness assumption"],
    ["Low recharge test", K_initial, 50.0,
     "Recharge sensitivity test; not a measured recharge rate"],
    ["High recharge test", K_initial, 200.0,
     "Recharge sensitivity test; not a measured recharge rate"],
], columns=[
    "scenario", "K_m_day", "recharge_mm_year", "interpretation"
])

sensitivity.to_csv(OUT / "sensitivity_scenarios_not_run.csv", index=False)

# ----------------------- 10. SAVE MODEL INPUTS --------------------

# Parameter register, final head grid, summary, and model input files
# remain in OUT. Zip everything for download.
import zipfile

zip_path = Path("/content/Alwar_MODFLOW6_FirstPass.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in OUT.rglob("*"):
        if path.is_file():
            zf.write(path, arcname=str(path.relative_to(OUT.parent)))

print("\nOUTPUT FOLDER:", OUT)
print("DOWNLOADABLE ARCHIVE:", zip_path)
print("\nMain outputs:")
for path in [
    OUT / "parameter_register.csv",
    OUT / "final_head_grid.csv",
    OUT / "final_head_map.png",
    OUT / "run_summary.json",
    OUT / "sensitivity_scenarios_not_run.csv",
    zip_path
]:
    print(" -", path)


**[orig #32]** archived — exact duplicate of original cell 33  
<sub>✅ ran OK</sub>

In [ ]:

# =====================================================================
# ALWAR GROUNDWATER CALIBRATION PREPARATION — MODFLOW 6 / FLOPY
# Google Colab
#
# Stage 1: Audit cleaned groundwater observations
# Stage 2: Prepare observed-head and well-to-cell templates
# Stage 3: Calibrate K and recharge only when valid targets are ready
#
# IMPORTANT:
# The existing 20x20 model has an illustrative local grid.
# Geographic coordinates cannot be mapped scientifically to it until
# a real GIS grid / coordinate transformation is supplied.
# =====================================================================

import os
import sys
import json
import shutil
import subprocess
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------- 1. INSTALL / IMPORT ------------------------

try:
    import flopy
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "flopy"
    ])
    import flopy

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print("FloPy version:", flopy.__version__)

# ------------------------- 2. PATHS ----------------------------------

DRIVE_ROOT = Path("/content/drive/MyDrive")

GW_FOLDER = (
    DRIVE_ROOT / "gw_framework" / "data" /
    "Rajasthan_gw_data" /
    "Groundwater_Cleaning_According_to_User_Rules"
)

# New output folder. Source data are never edited.
OUT = DRIVE_ROOT / "Alwar_Groundwater_Calibration"
OUT.mkdir(parents=True, exist_ok=True)

CLEANED_LONG = GW_FOLDER / "groundwater_cleaned_long_seasonal.csv"
CLEANED_WIDE = GW_FOLDER / "groundwater_cleaned_wide_source_preserved.csv"

# The existing prototype model may still exist in the current runtime.
MODEL_DIR = Path("/content/Alwar_MODFLOW6_FirstPass/simulation")
MF6_EXE = Path("/content/modflow6_bin/mf6")

print("\nCalibration output:", OUT)
print("Long groundwater file exists:", CLEANED_LONG.exists())
print("Wide groundwater file exists:", CLEANED_WIDE.exists())
print("Existing simulation folder exists:", MODEL_DIR.exists())

# ------------------------- 3. LOCATE SOURCE DATA ----------------------

if CLEANED_LONG.exists():
    source_path = CLEANED_LONG
elif CLEANED_WIDE.exists():
    source_path = CLEANED_WIDE
else:
    raise FileNotFoundError(
        "The expected cleaned groundwater files were not found.\n"
        "Expected folder:\n"
        f"{GW_FOLDER}\n"
        "Check the Google Drive path and filenames."
    )

df = pd.read_csv(source_path, low_memory=False)
print("\nSource file:", source_path)
print("Rows:", len(df))
print("Columns:", list(df.columns))

# Column names are normalized only for matching.
# Original source columns and source files are preserved.
def norm_col(c):
    return (
        str(c).strip().lower()
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )

normalized = {c: norm_col(c) for c in df.columns}

def find_col(candidates):
    # Exact alias match first
    for candidate in candidates:
        for original, normed in normalized.items():
            if normed == candidate:
                return original

    # Then substring match
    for candidate in candidates:
        for original, normed in normalized.items():
            if candidate in normed:
                return original

    return None

well_col = find_col([
    "well_id", "wellid", "well_code", "site_id",
    "unique_id", "station_id", "id"
])

lat_col = find_col(["latitude", "lat"])
lon_col = find_col(["longitude", "long", "lon"])

date_col = find_col([
    "date", "observation_date", "measurement_date",
    "reading_date", "sample_date"
])

season_col = find_col([
    "season", "season_name", "monitoring_season"
])

year_col = find_col(["year", "observation_year"])

status_col = find_col([
    "status", "reading_status", "observation_status",
    "water_level_status"
])

water_level_col = find_col([
    "water_level_m_bgl", "water_level_bgl",
    "depth_to_water", "depth_to_water_m",
    "water_level_m", "water_level", "wl_value",
    "numeric_reading", "reading_value"
])

block_col = find_col(["block", "groundwater_block"])

print("\nDetected column mapping:")
for name, value in {
    "well_id": well_col,
    "latitude": lat_col,
    "longitude": lon_col,
    "date": date_col,
    "season": season_col,
    "year": year_col,
    "status": status_col,
    "water_level": water_level_col,
    "block": block_col
}.items():
    print(f"{name}: {value}")

# Save source-column audit.
pd.DataFrame({
    "source_column": list(df.columns),
    "normalized_for_matching": [normalized[c] for c in df.columns],
    "dtype": [str(df[c].dtype) for c in df.columns]
}).to_csv(OUT / "01_source_column_audit.csv", index=False)

# ------------------------- 4. OBSERVATION STATUS AUDIT ---------------

if status_col:
    status = df[status_col].astype("string").str.strip().str.lower()
else:
    status = pd.Series("unknown", index=df.index, dtype="string")

# Find likely numeric values. Do not convert "Dry" into a number.
if water_level_col:
    numeric_value = pd.to_numeric(
        df[water_level_col].astype("string")
        .str.replace("*", "", regex=False)
        .str.strip(),
        errors="coerce"
    )
else:
    numeric_value = pd.Series(np.nan, index=df.index)

dry_mask = status.eq("dry") | (
    water_level_col is not None and
    df[water_level_col].astype("string").str.strip().str.lower().eq("dry")
)

missing_mask = numeric_value.isna() & ~dry_mask

audit = pd.DataFrame({
    "record_number": np.arange(1, len(df) + 1),
    "source_row_index": df.index,
    "numeric_candidate": numeric_value,
    "status_detected": status,
    "is_dry": dry_mask,
    "is_missing_or_nonnumeric": missing_mask
})

audit.to_csv(OUT / "02_observation_status_audit.csv", index=False)

print("\nObservation audit:")
print("Total rows:", len(df))
print("Numeric candidates:", int(numeric_value.notna().sum()))
print("Dry observations:", int(dry_mask.sum()))
print("Missing/nonnumeric observations:", int(missing_mask.sum()))

# ------------------------- 5. PREPARE OBSERVATION TABLE ---------------

obs = df.copy()

# Preserve source columns; append standardized helper columns.
obs["calibration_numeric_candidate"] = numeric_value
obs["calibration_is_dry"] = dry_mask
obs["calibration_is_missing"] = missing_mask

if well_col:
    obs["calibration_well_id"] = df[well_col].astype("string").str.strip()
else:
    obs["calibration_well_id"] = pd.NA

if lat_col:
    obs["calibration_latitude"] = pd.to_numeric(
        df[lat_col], errors="coerce"
    )
else:
    obs["calibration_latitude"] = np.nan

if lon_col:
    obs["calibration_longitude"] = pd.to_numeric(
        df[lon_col], errors="coerce"
    )
else:
    obs["calibration_longitude"] = np.nan

if date_col:
    obs["calibration_date"] = pd.to_datetime(
        df[date_col], errors="coerce", dayfirst=True
    )
elif year_col:
    # Year alone is not a precise observation date.
    obs["calibration_date"] = pd.to_datetime(
        pd.to_numeric(df[year_col], errors="coerce"),
        format="%Y", errors="coerce"
    )
else:
    obs["calibration_date"] = pd.NaT

if season_col:
    obs["calibration_season"] = df[season_col].astype("string")
else:
    obs["calibration_season"] = pd.NA

if block_col:
    obs["calibration_block"] = df[block_col].astype("string")
else:
    obs["calibration_block"] = pd.NA

# The detected numeric column may represent depth below ground,
# depth to water, or another convention. We do NOT assume it is head.
obs["observed_head_m"] = np.nan
obs["model_row"] = np.nan
obs["model_col"] = np.nan
obs["stress_period"] = np.nan
obs["ground_elevation_m"] = np.nan

# Export every source record with helper columns for inspection.
obs.to_csv(
    OUT / "03_groundwater_observations_with_calibration_fields.csv",
    index=False
)

# Export numeric candidates separately for manual selection.
numeric_obs = obs.loc[obs["calibration_numeric_candidate"].notna()].copy()
numeric_obs.to_csv(
    OUT / "04_numeric_observation_candidates.csv",
    index=False
)

# ------------------------- 6. CALIBRATION TEMPLATE --------------------

# This template must be completed using verified coordinate-to-grid
# mapping and a known vertical datum.
#
# Required columns:
# calibration_well_id, calibration_date, observed_head_m,
# model_row, model_col, stress_period
#
# For depth-to-water readings in m bgl:
# observed_head_m = surveyed_ground_elevation_m - depth_to_water_m
# ONLY if the elevation and water-level reference conventions match.

template_cols = [
    "calibration_well_id",
    "calibration_date",
    "calibration_season",
    "calibration_block",
    "calibration_latitude",
    "calibration_longitude",
    "calibration_numeric_candidate",
    "ground_elevation_m",
    "observed_head_m",
    "model_row",
    "model_col",
    "stress_period"
]

template = obs[template_cols].copy()

template_path = OUT / "05_CALIBRATION_TARGETS_TEMPLATE.csv"
template.to_csv(template_path, index=False)

print("\nCreated calibration target template:")
print(template_path)

# ------------------------- 7. MODEL READINESS CHECK -------------------

if not MODEL_DIR.exists():
    print(
        "\nExisting model files are not present in this runtime. "
        "Upload/extract your downloaded model ZIP before calibration."
    )

required_model_files = [
    "mfsim.nam",
    "alwar_gwf.nam"
]

if MODEL_DIR.exists():
    for fname in required_model_files:
        print(fname, "exists:", (MODEL_DIR / fname).exists())

# ------------------------- 8. CALIBRATION TARGET FILE -----------------
# The script will only calibrate if you fill the template and save a
# separate file named 06_CALIBRATION_TARGETS_READY.csv in OUT.
#
# Do not simply rename the blank template. It needs:
# - real model row and column indices (zero-based)
# - observed hydraulic head in metres
# - stress_period 0 to 11 for the current 12-period model
#
# If observations are only depth-to-water, ground elevations are needed
# to derive hydraulic head. If the model grid is still the illustrative
# 20x20 grid, geographic coordinates cannot be mapped validly yet.

ready_path = OUT / "06_CALIBRATION_TARGETS_READY.csv"

def calibration_readiness_check(path):
    if not path.exists():
        return False, "Ready target file has not been created."

    targets = pd.read_csv(path)

    required = [
        "observed_head_m", "model_row", "model_col", "stress_period"
    ]

    missing_cols = [c for c in required if c not in targets.columns]
    if missing_cols:
        return False, f"Missing required columns: {missing_cols}"

    for c in required:
        targets[c] = pd.to_numeric(targets[c], errors="coerce")

    targets = targets.dropna(subset=required)

    if len(targets) < 3:
        return False, "At least 3 valid targets are required for this check."

    if not targets["model_row"].between(0, 19).all():
        return False, "model_row must be 0–19 for the current prototype."

    if not targets["model_col"].between(0, 19).all():
        return False, "model_col must be 0–19 for the current prototype."

    if not targets["stress_period"].between(0, 11).all():
        return False, "stress_period must be 0–11 for the current model."

    if not np.allclose(
        targets["stress_period"],
        np.round(targets["stress_period"])
    ):
        return False, "stress_period must contain integers."

    if targets["observed_head_m"].nunique() < 2:
        return False, "Observed heads have no meaningful variation."

    return True, targets

ready, result = calibration_readiness_check(ready_path)

# ------------------------- 9. RUN CALIBRATION IF READY ----------------
# Parameters optimized: uniform K and uniform recharge.
# Objective: minimize RMSE between simulated and observed heads.
#
# This is an initial parameter-fitting exercise, not a substitute for
# checking conceptual model validity, boundary conditions, or uncertainty.
#
# Calibration is skipped unless valid targets and existing MODFLOW files
# are available.

if not ready:
    print("\nCALIBRATION NOT RUN.")
    print("Reason:", result)
    print("\nNext steps:")
    print("1. Inspect 04_numeric_observation_candidates.csv.")
    print("2. Verify whether readings are depth-to-water or hydraulic head.")
    print("3. Obtain ground elevations if readings are depth below ground.")
    print("4. Build a real GIS model grid and map each well to a cell.")
    print("5. Fill 05_CALIBRATION_TARGETS_TEMPLATE.csv.")
    print("6. Save the completed file as:")
    print(ready_path)

else:
    if not MODEL_DIR.exists():
        raise FileNotFoundError(
            "Calibration targets are ready but model files are absent. "
            "Restore the model simulation folder first."
        )

    if not MF6_EXE.exists():
        raise FileNotFoundError(
            f"MODFLOW executable not found: {MF6_EXE}"
        )

    targets = result.copy()
    targets["model_row"] = targets["model_row"].astype(int)
    targets["model_col"] = targets["model_col"].astype(int)
    targets["stress_period"] = targets["stress_period"].astype(int)

    # Load existing model.
    sim = flopy.mf6.MFSimulation.load(
        sim_ws=str(MODEL_DIR),
        exe_name=str(MF6_EXE),
        verbosity_level=0
    )

    gwf = sim.get_model("alwar_gwf")
    if gwf is None:
        raise RuntimeError(
            "Model 'alwar_gwf' was not found in the simulation."
        )

    npf = gwf.get_package("npf")
    rcha = gwf.get_package("rcha")

    if npf is None or rcha is None:
        raise RuntimeError(
            "Required NPF or RCHA package is missing."
        )

    # Initial K bounds derived from the CGWB T range divided by the
    # assumed 50 m thickness. These bounds remain provisional.
    k_lo = 27.0 / 50.0
    k_hi = 662.0 / 50.0

    # Recharge search bounds are deliberately broad sensitivity bounds,
    # not measured Alwar recharge rates.
    rch_lo = 0.0
    rch_hi = 0.0008  # m/day, about 292 mm/year

    # Calibrate log10(K) and log10(recharge) when recharge is positive.
    # Zero recharge is handled as a separate candidate.
    from scipy.optimize import differential_evolution

    head_file = MODEL_DIR / "alwar_gwf.hds"

    def run_candidate(log_params):
        K = 10.0 ** float(log_params[0])
        recharge = 10.0 ** float(log_params[1])

        # Update spatially uniform K.
        npf.k.set_data(
            np.full((1, 20, 20), K, dtype=float)
        )

        # Update recharge for all stress periods.
        recharge_data = {
            kper: np.full((20, 20), recharge, dtype=float)
            for kper in range(12)
        }
        rcha.recharge.set_data(recharge_data)

        sim.write_simulation()

        ok, run_report = sim.run_simulation(
            silent=True,
            report=True
        )

        if not ok or not head_file.exists():
            return 1.0e6

        hds = flopy.utils.HeadFile(str(head_file))
        model_times = hds.get_times()

        residuals = []

        for _, row in targets.iterrows():
            sp = int(row["stress_period"])
            if sp >= len(model_times):
                continue

            head = hds.get_data(totim=model_times[sp])
            simulated = head[
                0, int(row["model_row"]), int(row["model_col"])
            ]
            observed = float(row["observed_head_m"])

            if not np.isfinite(simulated) or abs(simulated) > 1.0e20:
                continue

            residuals.append(simulated - observed)

        if len(residuals) < 3:
            return 1.0e6

        return float(np.sqrt(np.mean(np.square(residuals))))

    bounds = [
        (np.log10(k_lo), np.log10(k_hi)),
        (np.log10(1.0e-7), np.log10(rch_hi))
    ]

    print("\nStarting calibration search. This may take time.")

    result_opt = differential_evolution(
        run_candidate,
        bounds=bounds,
        seed=42,
        maxiter=12,
        popsize=6,
        polish=False,
        workers=1,
        updating="immediate"
    )

    best_K = 10.0 ** result_opt.x[0]
    best_recharge = 10.0 ** result_opt.x[1]
    best_rmse = float(result_opt.fun)

    # Run final best parameter combination and save final heads.
    final_rmse = run_candidate(result_opt.x)

    hds = flopy.utils.HeadFile(str(head_file))
    model_times = hds.get_times()

    residual_rows = []
    for _, row in targets.iterrows():
        sp = int(row["stress_period"])
        if sp >= len(model_times):
            continue

        head = hds.get_data(totim=model_times[sp])
        simulated = float(
            head[0, int(row["model_row"]), int(row["model_col"])]
        )
        observed = float(row["observed_head_m"])

        residual_rows.append({
            "well_id": row.get("calibration_well_id", ""),
            "stress_period": sp,
            "model_row": int(row["model_row"]),
            "model_col": int(row["model_col"]),
            "observed_head_m": observed,
            "simulated_head_m": simulated,
            "residual_sim_minus_obs_m": simulated - observed
        })

    residual_df = pd.DataFrame(residual_rows)
    residual_df.to_csv(
        OUT / "07_calibration_residuals.csv",
        index=False
    )

    calibration_summary = {
        "calibration_method": "Differential evolution; uniform K and recharge",
        "successful_optimizer_run": bool(result_opt.success),
        "optimizer_message": str(result_opt.message),
        "number_of_targets": int(len(residual_df)),
        "best_K_m_day": float(best_K),
        "best_recharge_m_day": float(best_recharge),
        "best_recharge_mm_year": float(best_recharge * 365.25 * 1000),
        "RMSE_m": float(best_rmse),
        "interpretation": (
            "Conditional initial fit only. Results are not defensible as "
            "district calibration unless the model grid, observation heads, "
            "well-to-cell mapping, boundaries and conceptual model are valid."
        )
    }

    with open(OUT / "08_calibration_summary.json", "w") as f:
        json.dump(calibration_summary, f, indent=2)

    print("\nCalibration result:")
    print(json.dumps(calibration_summary, indent=2))

    # Plot residuals
    if not residual_df.empty:
        plt.figure(figsize=(8, 4))
        plt.scatter(
            residual_df["observed_head_m"],
            residual_df["simulated_head_m"]
        )
        minv = min(
            residual_df["observed_head_m"].min(),
            residual_df["simulated_head_m"].min()
        )
        maxv = max(
            residual_df["observed_head_m"].max(),
            residual_df["simulated_head_m"].max()
        )
        plt.plot([minv, maxv], [minv, maxv], linestyle="--")
        plt.xlabel("Observed hydraulic head (m)")
        plt.ylabel("Simulated hydraulic head (m)")
        plt.title("Calibration: Observed vs Simulated Heads")
        plt.tight_layout()
        plt.savefig(
            OUT / "09_observed_vs_simulated.png",
            dpi=200,
            bbox_inches="tight"
        )
        plt.show()

# ------------------------- 10. ZIP REPORTS ---------------------------

zip_path = Path("/content/Alwar_Groundwater_Calibration_Reports.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as zf:
    for p in OUT.rglob("*"):
        if p.is_file():
            zf.write(p, arcname=str(p.relative_to(OUT.parent)))

print("\nFinished.")
print("Calibration workspace:", OUT)
print("Report archive:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print("Download the archive from the path shown above.")


**[orig #35]** archived — exact duplicate of original cell 27  
<sub>✅ ran OK</sub>

In [ ]:

import requests
import subprocess
import os
import json

domain = "cgwb.gov.in"

# Query Google Public DNS over HTTPS.
try:
    r = requests.get(
        "https://dns.google/resolve",
        params={"name": domain, "type": "A"},
        timeout=20
    )
    r.raise_for_status()
    dns_result = r.json()

    ips = [
        answer["data"]
        for answer in dns_result.get("Answer", [])
        if answer.get("type") == 1
    ]

    print("DNS status:", dns_result.get("Status"))
    print("Google DNS A records:", ips)

except Exception as e:
    ips = []
    print("Google DNS lookup failed:", repr(e))

# Test each returned address while retaining the domain as the
# TLS hostname. curl still verifies the server certificate.
test_url = "https://cgwb.gov.in/sites/default/files/2022-10/alwar.pdf"

for ip in ips:
    print("\nTesting CGWB with resolved IP:", ip)

    result = subprocess.run(
        [
            "curl", "--fail", "--location",
            "--connect-timeout", "15",
            "--max-time", "60",
            "--resolve", f"cgwb.gov.in:443:{ip}",
            "--output", "/tmp/cgwb_test.pdf",
            "--write-out",
            "HTTP=%{http_code} IP=%{remote_ip}\n",
            test_url
        ],
        capture_output=True,
        text=True
    )

    print(result.stdout)
    print(result.stderr[-1000:] if result.stderr else "")

    if os.path.exists("/tmp/cgwb_test.pdf"):
        with open("/tmp/cgwb_test.pdf", "rb") as f:
            valid_pdf = f.read(5) == b"%PDF-"

        if valid_pdf:
            print("Verified PDF downloaded from:", ip)
            break
        else:
            os.remove("/tmp/cgwb_test.pdf")


**[orig #37]** archived — failed; file was found by a later cell  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:
import pandas as pd

# Use the uploaded audit CSV in your Google Drive project
df = pd.read_csv(
    "/content/drive/MyDrive/gw_framework/data/"
    "gwl/diagnostics/manual_observations_audit.csv"
)

df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("Negative DTW observations:", (df["dtw"] < 0).sum())

print("\nNegative DTW counts by well:")
print(
    df.loc[df["dtw"] < 0]
      .groupby("well_id")
      .agg(
          negative_count=("dtw", "size"),
          minimum_dtw=("dtw", "min"),
          first_date=("date", "min"),
          last_date=("date", "max")
      )
      .sort_values("negative_count", ascending=False)
)

print("\nLargest positive DTW observations:")
print(
    df.nlargest(15, "dtw")[["well_id", "date", "dtw"]]
      .to_string(index=False)
)

**[orig #38]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

matches = list(
    Path("/content/drive/MyDrive").rglob(
        "manual_observations_audit.csv"
    )
)

if matches:
    for path in matches:
        print("Found:", path)
else:
    print("Audit CSV not found in Google Drive.")
    print("We will need to regenerate it or upload it.")

**[orig #42]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

my_drive = Path('/content/drive/MyDrive')

# Find shortcut/folder names containing "INDRA" or "downscaled"
for p in my_drive.rglob("*"):
    if p.name.lower() in {"indra-cmip6", "downscaled_data"}:
        print("Found:", p)
        print("Is directory:", p.is_dir())

**[orig #44]** archived — exact duplicate of original cell 43  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/INDRA-CMIP6"
)

print("Root exists:", ROOT.exists())
print("\nFolder structure:\n")

for p in sorted(ROOT.rglob("*")):
    relative = p.relative_to(ROOT)
    if p.is_dir():
        print(f"[DIR]  {relative}/")
    else:
        print(f"[FILE] {relative}  |  {p.stat().st_size / 1e6:.2f} MB")

**[orig #49]** archived — failed (NameError) INDRA test  
<sub>⚠ last run ended with NameError</sub>

In [ ]:
import numpy as np

# Replace these with your actual column names.
ID_COL = "well_id"
DATE_COL = "date"
LAT_COL = "latitude"
LON_COL = "longitude"

wells = (
    gw[[ID_COL, LAT_COL, LON_COL]]
    .dropna()
    .drop_duplicates(ID_COL)
    .copy()
)

wells.columns = ["well_id", "latitude", "longitude"]

# Use one historical model and future SSP2-4.5.
climate_files = {
    "pr_historical": CLIMATE / "pr/ACCESS-CM2/historical_DBCCA.nc",
    "pr_ssp245": CLIMATE / "pr/ACCESS-CM2/ssp245_DBCCA.nc",
    "tasmax_historical": CLIMATE / "tasmax/ACCESS-CM2/historical_DBCCA.nc",
    "tasmax_ssp245": CLIMATE / "tasmax/ACCESS-CM2/ssp245_DBCCA.nc",
    "tasmin_historical": CLIMATE / "tasmin/ACCESS-CM2/historical_DBCCA.nc",
    "tasmin_ssp245": CLIMATE / "tasmin/ACCESS-CM2/ssp245_DBCCA.nc",
}

for label, path in climate_files.items():
    if not path.exists():
        raise FileNotFoundError(path)

    with xr.open_dataset(path) as ds:
        da = ds[next(iter(ds.data_vars))]
        print(
            label,
            "| variable:", da.name,
            "| units:", da.attrs.get("units"),
            "| time:", str(ds.time.values[0]),
            "to", str(ds.time.values[-1])
        )

        # Show nearest grid-cell coordinates for each well.
        sampled = da.sel(
            lat=xr.DataArray(wells["latitude"].values, dims="well"),
            lon=xr.DataArray(wells["longitude"].values, dims="well"),
            method="nearest"
        )

        grid_lat = sampled["lat"].values
        grid_lon = sampled["lon"].values

        if label == "pr_historical":
            wells["grid_lat"] = grid_lat
            wells["grid_lon"] = grid_lon

print("\nWELL-TO-GRID MATCH")
display(wells)

**[orig #53]** archived — failed (NameError)  
<sub>⚠ last run ended with NameError</sub>

In [ ]:
# Check whether the combined dataset 'gw' has been loaded
print("Columns available:", gw.columns.tolist())

# Inspect well coordinates
display(
    gw[[
        "District",
        "WIMS_ID",
        "Latitude",
        "Longitude"
    ]].head(10)
)

# Check missing and invalid coordinates
print("\nMissing latitude:", gw["Latitude"].isna().sum())
print("Missing longitude:", gw["Longitude"].isna().sum())

valid = (
    gw["Latitude"].between(-90, 90)
    & gw["Longitude"].between(-180, 180)
)

print("Rows with valid coordinate ranges:", valid.sum(), "of", len(gw))

**[orig #60]** archived — failed; replaced by A1 inspection cells  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:

# ============================================================
# RAINFALL DATA INSPECTION FOR JAIPUR CIRCLE GROUNDWATER MODEL
# Read-only inspection: original files will not be modified.
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import warnings

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Locate rainfall and groundwater folders
# ------------------------------------------------------------
BASE = Path("/content/drive/MyDrive")
RAINFALL_DIR = BASE / "Rainfall_Rajasthan"
GW_DIR = BASE / "gw_framework/data/Rajasthan_gw_data"

OUT_DIR = GW_DIR / "Rainfall_Inspection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

if not RAINFALL_DIR.exists():
    raise FileNotFoundError(
        f"Folder not found: {RAINFALL_DIR}\n"
        "Check the exact Google Drive folder name and path."
    )

# ------------------------------------------------------------
# 2. List all rainfall files, including subfolders
# ------------------------------------------------------------
SUPPORTED = {
    ".csv", ".xlsx", ".xls", ".xlsm",
    ".txt", ".parquet", ".json", ".nc", ".nc4"
}

files = sorted(
    p for p in RAINFALL_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in SUPPORTED
)

print(f"Rainfall folder: {RAINFALL_DIR}")
print(f"Supported files found: {len(files)}")
print("-" * 80)

for i, f in enumerate(files, 1):
    print(f"{i:03d}. {f.relative_to(RAINFALL_DIR)} "
          f"| {f.stat().st_size / 1024:.1f} KB")

if not files:
    raise ValueError(
        "No supported data files found. Check the folder contents."
    )

# ------------------------------------------------------------
# 3. Read tabular files and inspect Excel sheet names
# ------------------------------------------------------------
def read_tabular_file(path):
    ext = path.suffix.lower()
    tables = {}

    if ext in [".xlsx", ".xls", ".xlsm"]:
        xls = pd.ExcelFile(path)
        for sheet in xls.sheet_names:
            try:
                tables[sheet] = pd.read_excel(
                    path, sheet_name=sheet
                )
            except Exception as e:
                print(f"Could not read sheet {sheet}: {e}")

    elif ext == ".csv":
        # Try common encodings and delimiters
        for encoding in ["utf-8", "cp1252", "latin1"]:
            try:
                tables["data"] = pd.read_csv(
                    path, encoding=encoding, sep=None,
                    engine="python", on_bad_lines="warn"
                )
                break
            except Exception:
                continue

    elif ext == ".txt":
        for encoding in ["utf-8", "cp1252", "latin1"]:
            try:
                tables["data"] = pd.read_csv(
                    path, encoding=encoding, sep=None,
                    engine="python", on_bad_lines="warn"
                )
                break
            except Exception:
                continue

    elif ext == ".parquet":
        tables["data"] = pd.read_parquet(path)

    elif ext == ".json":
        try:
            tables["data"] = pd.read_json(path)
        except Exception:
            with open(path, encoding="utf-8") as fp:
                obj = json.load(fp)
            if isinstance(obj, list):
                tables["data"] = pd.DataFrame(obj)

    return tables

tabular_data = {}
file_errors = []

for f in files:
    if f.suffix.lower() in [".nc", ".nc4"]:
        print(f"\nNETCDF FILE DETECTED: {f.name}")
        print("Will inspect separately in Section 8.")
        continue

    try:
        tables = read_tabular_file(f)
        for sheet, df in tables.items():
            key = f"{f.relative_to(RAINFALL_DIR)}::{sheet}"
            tabular_data[key] = df
    except Exception as e:
        file_errors.append({
            "file": str(f.relative_to(RAINFALL_DIR)),
            "error": str(e)
        })

print(f"\nReadable tabular datasets: {len(tabular_data)}")
print(f"Read errors: {len(file_errors)}")

# ------------------------------------------------------------
# 4. Inspect columns, dimensions, data types and sample records
# ------------------------------------------------------------
inventory = []
column_report = []

for key, df in tabular_data.items():
    print("\n" + "=" * 90)
    print("DATASET:", key)
    print("Shape:", df.shape)
    print("\nColumns and data types:")
    print(df.dtypes.to_string())
    print("\nFirst 5 records:")
    display(df.head())

    for col in df.columns:
        s = df[col]
        column_report.append({
            "dataset": key,
            "column": str(col),
            "dtype": str(s.dtype),
            "non_null": int(s.notna().sum()),
            "missing": int(s.isna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "unique_values": int(s.nunique(dropna=True)),
            "examples": " | ".join(
                s.dropna().astype(str).head(3).tolist()
            )
        })

    inventory.append({
        "dataset": key,
        "rows": len(df),
        "columns": len(df.columns),
        "duplicate_full_rows": int(df.duplicated().sum()),
        "column_names": " | ".join(map(str, df.columns))
    })

inventory_df = pd.DataFrame(inventory)
columns_df = pd.DataFrame(column_report)

# ------------------------------------------------------------
# 5. Automatically identify date, rainfall and location columns
# ------------------------------------------------------------
def identify_columns(df):
    result = {
        "date_columns": [],
        "rainfall_columns": [],
        "latitude_columns": [],
        "longitude_columns": [],
        "station_columns": [],
        "district_columns": [],
        "block_columns": []
    }

    for col in df.columns:
        name = str(col).strip().lower()
        compact = re.sub(r"[^a-z0-9]", "", name)

        if (
            compact in ["date", "datetime", "time", "observationdate"]
            or "date" in compact
            or compact in ["day"]
        ):
            result["date_columns"].append(col)

        if (
            "rain" in compact
            or compact in ["rf", "precip", "precipitation", "pr"]
        ):
            result["rainfall_columns"].append(col)

        if compact in ["lat", "latitude", "latitudedegrees"]:
            result["latitude_columns"].append(col)

        if compact in ["lon", "long", "longitude", "longitudedegrees"]:
            result["longitude_columns"].append(col)

        if any(x in compact for x in [
            "station", "raingauge", "gaugeid", "stationid",
            "stationcode", "observatory"
        ]):
            result["station_columns"].append(col)

        if "district" in compact:
            result["district_columns"].append(col)

        if "block" in compact or "tehsil" in compact:
            result["block_columns"].append(col)

    return result

# Use a broad date parser to handle common date formats.
def find_date_column(df, candidates):
    ranked = []

    for col in candidates:
        try:
            parsed = pd.to_datetime(df[col], errors="coerce", dayfirst=True)
            valid_fraction = parsed.notna().mean() if len(df) else 0
            ranked.append((valid_fraction, col, parsed))
        except Exception:
            pass

    if not ranked:
        return None, None

    ranked.sort(key=lambda x: x[0], reverse=True)
    fraction, col, parsed = ranked[0]

    if fraction >= 0.50:
        return col, parsed

    return None, None

dataset_findings = []
daily_checks = []

for key, df in tabular_data.items():
    identified = identify_columns(df)
    date_col, parsed_dates = find_date_column(
        df, identified["date_columns"]
    )

    print("\n" + "-" * 90)
    print("DATASET:", key)
    print("Potential date columns:", identified["date_columns"])
    print("Potential rainfall columns:", identified["rainfall_columns"])
    print("Potential station columns:", identified["station_columns"])
    print("Potential district columns:", identified["district_columns"])
    print("Potential block columns:", identified["block_columns"])

    date_min = date_max = None
    duplicate_dates = None
    missing_calendar_days = None

    if date_col is not None:
        valid_dates = parsed_dates.dropna()
        if len(valid_dates):
            date_min = valid_dates.min()
            date_max = valid_dates.max()

            # This simple calendar check is most appropriate for
            # daily data from one station or one station-date table.
            duplicate_dates = int(valid_dates.duplicated().sum())
            expected_days = (date_max.normalize() -
                             date_min.normalize()).days + 1
            observed_unique_days = valid_dates.dt.normalize().nunique()
            missing_calendar_days = max(
                0, expected_days - observed_unique_days
            )

        print("Selected date column:", date_col)
        print("Date range:", date_min, "to", date_max)
        print("Duplicate date values in entire table:", duplicate_dates)
        print("Missing calendar days in overall date span:",
              missing_calendar_days)

    rainfall_summaries = []

    for col in identified["rainfall_columns"]:
        values = pd.to_numeric(
            df[col].astype(str).str.replace(",", "", regex=False),
            errors="coerce"
        )

        rainfall_summaries.append({
            "column": str(col),
            "numeric_count": int(values.notna().sum()),
            "missing_or_nonnumeric": int(values.isna().sum()),
            "minimum": values.min(),
            "median": values.median(),
            "maximum": values.max(),
            "zero_count": int((values == 0).sum()),
            "negative_count": int((values < 0).sum()),
            "above_300_mm_count": int((values > 300).sum())
        })

    if rainfall_summaries:
        print("\nRainfall value checks:")
        display(pd.DataFrame(rainfall_summaries))

    dataset_findings.append({
        "dataset": key,
        "rows": len(df),
        "date_column": str(date_col) if date_col is not None else None,
        "date_start": date_min,
        "date_end": date_max,
        "date_duplicates_in_table": duplicate_dates,
        "missing_calendar_days_overall": missing_calendar_days,
        "potential_rainfall_columns":
            ", ".join(map(str, identified["rainfall_columns"])),
        "potential_station_columns":
            ", ".join(map(str, identified["station_columns"])),
        "potential_district_columns":
            ", ".join(map(str, identified["district_columns"])),
        "potential_block_columns":
            ", ".join(map(str, identified["block_columns"]))
    })

# ------------------------------------------------------------
# 6. Inspect Jaipur groundwater wells and measurement coverage
# ------------------------------------------------------------
gw_files = sorted(GW_DIR.glob("*.xlsx")) if GW_DIR.exists() else []

print("\n" + "=" * 90)
print("GROUNDWATER DATA FILES:", len(gw_files))

gw_frames = []
for f in gw_files:
    try:
        temp = pd.read_excel(f)
        temp["Source_File"] = f.name
        gw_frames.append(temp)
    except Exception as e:
        print("Groundwater file error:", f.name, e)

if gw_frames:
    gw = pd.concat(gw_frames, ignore_index=True, sort=False)

    print("Groundwater records:", len(gw))
    print("Groundwater columns:")
    print(gw.columns.tolist())

    # Identify likely location fields without assuming exact names.
    col_map = {
        re.sub(r"[^a-z0-9]", "", str(c).lower()): c
        for c in gw.columns
    }

    for target in ["district", "block", "latitude", "longitude"]:
        matches = [v for k, v in col_map.items() if target in k]
        print(f"Potential {target} fields:", matches)

    district_cols = [
        c for c in gw.columns
        if "district" in str(c).lower()
    ]

    if district_cols:
        dc = district_cols[0]
        print("\nGroundwater records by district:")
        display(
            gw[dc].astype("string").value_counts(dropna=False)
            .rename_axis("District")
            .reset_index(name="Number_of_records")
        )

    print("\nSample groundwater records:")
    display(gw.head())

else:
    print("No groundwater Excel files found at:", GW_DIR)

# ------------------------------------------------------------
# 7. Save an inspection report
# ------------------------------------------------------------
if not inventory_df.empty:
    inventory_df.to_csv(
        OUT_DIR / "rainfall_file_inventory.csv", index=False
    )

if not columns_df.empty:
    columns_df.to_csv(
        OUT_DIR / "rainfall_column_report.csv", index=False
    )

findings_df = pd.DataFrame(dataset_findings)
if not findings_df.empty:
    findings_df.to_csv(
        OUT_DIR / "rainfall_date_and_value_checks.csv", index=False
    )

if file_errors:
    pd.DataFrame(file_errors).to_csv(
        OUT_DIR / "rainfall_read_errors.csv", index=False
    )

print("\n" + "=" * 90)
print("INSPECTION COMPLETE")
print("Reports saved in:", OUT_DIR)
print(" - rainfall_file_inventory.csv")
print(" - rainfall_column_report.csv")
print(" - rainfall_date_and_value_checks.csv")
if file_errors:
    print(" - rainfall_read_errors.csv")

# ------------------------------------------------------------
# 8. Optional: inspect NetCDF rainfall files
# ------------------------------------------------------------
nc_files = [
    f for f in files if f.suffix.lower() in [".nc", ".nc4"]
]

if nc_files:
    print("\nNetCDF files found:", len(nc_files))
    try:
        import xarray as xr

        for f in nc_files:
            print("\nNETCDF:", f.name)
            try:
                ds = xr.open_dataset(f)
                print(ds)
                print("Variables:", list(ds.data_vars))
                print("Coordinates:", list(ds.coords))
                ds.close()
            except Exception as e:
                print("Could not inspect NetCDF:", e)

    except ImportError:
        print("Install xarray/netCDF4 if needed:")
        print("!pip install xarray netCDF4")


**[orig #61]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path

DRIVE = Path("/content/drive")

# Check the mounted Drive and list MyDrive contents
MYDRIVE = DRIVE / "MyDrive"

print("MyDrive exists:", MYDRIVE.exists())

if MYDRIVE.exists():
    print("\nFolders and files in MyDrive:")
    for p in sorted(MYDRIVE.iterdir(), key=lambda x: x.name.lower()):
        print(("📁 " if p.is_dir() else "📄 ") + p.name)

# Search for folders whose names contain 'rainfall'
print("\nSearching for rainfall folders...")
matches = []

for root in [MYDRIVE, DRIVE / "Shareddrives"]:
    if root.exists():
        for p in root.rglob("*"):
            if p.is_dir() and "rainfall" in p.name.lower():
                matches.append(p)
                if len(matches) >= 100:
                    break

print("\nMatching folders:")
if matches:
    for p in matches:
        print(p)
else:
    print("No matching folders found.")
    print("Check whether the folder is in a Shared Drive, shortcut,")
    print("or a different Google account.")


**[orig #64]** archived — DTW check placed in the rainfall flow (used the wrong `df`)  
<sub>✅ ran OK</sub>

In [ ]:
required = {"well_id", "date", "dtw"}

if required.issubset(df.columns):
    df["date"] = pd.to_datetime(
        df["date"], errors="coerce", dayfirst=True
    )
    df["dtw"] = pd.to_numeric(df["dtw"], errors="coerce")

    print("Total observations:", len(df))
    print("Negative DTW values:", (df["dtw"] < 0).sum())

    print("\nNegative values by well:")
    display(
        df[df["dtw"] < 0]
        .groupby("well_id")
        .agg(
            count=("dtw", "size"),
            minimum=("dtw", "min"),
            first_date=("date", "min"),
            last_date=("date", "max")
        )
        .sort_values("count", ascending=False)
    )

    print("\n15 largest DTW readings:")
    display(df.nlargest(15, "dtw")[["well_id", "date", "dtw"]])

else:
    print("This CSV does not contain all required observation columns.")
    print("Available columns:", df.columns.tolist())

**[orig #74]** archived — failed (ValueError)  
<sub>⚠ last run ended with ValueError</sub>

In [ ]:

from pathlib import Path
import pandas as pd

root = Path("/content/drive/MyDrive/gw_framework/data/gwl")

files = {
    "original_wells_meta": root / "processed" / "wells_meta.csv",
    "validation_report": root / "diagnostics" / "well_metadata_validation.csv",
    "seasonal_observations": root / "processed" / "seasonal.csv",
}

for label, path in files.items():
    print("\n" + "=" * 70)
    print(label, ":", path)
    print("=" * 70)

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    df = pd.read_csv(path, low_memory=False)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if label != "seasonal_observations":
        print("\nFirst 10 rows:")
        display(df.head(10))

        for col in ["state", "district", "block", "source", "file",
                    "well_id", "lat", "lon", "_merge"]:
            if col in df.columns:
                print(f"\n{col} value counts (including missing):")
                print(df[col].fillna("[MISSING]").astype(str)
                      .value_counts().head(20))

# Compare well IDs and coordinates between the two metadata files
original_path = files["original_wells_meta"]
validation_path = files["validation_report"]

if original_path.exists() and validation_path.exists():
    original = pd.read_csv(original_path, low_memory=False)
    validation = pd.read_csv(validation_path, low_memory=False)

    if "well_id" in original.columns and "well_id" in validation.columns:
        original["well_id"] = original["well_id"].astype(str).str.strip()
        validation["well_id"] = validation["well_id"].astype(str).str.strip()

        comparison = original.merge(
            validation,
            on="well_id",
            how="outer",
            suffixes=("_original", "_validation"),
            indicator=True
        )

        print("\nID comparison:")
        print(comparison["_merge"].value_counts())

        coordinate_cols = [
            c for c in comparison.columns
            if c.startswith(("lat_", "lon_"))
        ]
        print("\nCoordinate comparison columns:", coordinate_cols)

        display(comparison.head(30))

        out = root / "diagnostics" / "Study_Area_Verification"
        out.mkdir(parents=True, exist_ok=True)
        comparison.to_csv(out / "08_original_vs_validation_metadata.csv",
                          index=False)
        print("\nComparison saved to:", out / "08_original_vs_validation_metadata.csv")


**[orig #76]** archived — failed (FileNotFoundError)  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:

import pandas as pd
from pathlib import Path
from IPython.display import display

folder = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Study_Area_Verification"
)

file = folder / "09_project_source_file_inventory.csv"

if not file.exists():
    raise FileNotFoundError(
        "Inventory not found. Rerun the Groundwater Source Recovery Audit first."
    )

df = pd.read_csv(file)

print("TABLE 1 — LARGEST NON-EMPTY DATA FILES")
display(
    df[df["size_bytes"] > 0]
    .sort_values("size_bytes", ascending=False)
    [["filename", "path", "size_bytes", "rows", "columns", "read_status"]]
    .head(30)
)

print("\nTABLE 2 — LIKELY GROUNDWATER FILES")
keywords = [
    "groundwater", "water_level", "waterlevel", "well",
    "observation", "seasonal", "dtw", "gwl", "manual",
    "cgwb", "dwlr", "piezometer"
]

mask = df["path"].str.lower().apply(
    lambda p: any(k in p for k in keywords)
)

display(
    df.loc[mask, [
        "filename", "path", "size_bytes",
        "rows", "columns", "read_status"
    ]].head(50)
)


**[orig #77]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd
from google.colab import drive
from IPython.display import display

# Mount Drive if necessary
drive.mount("/content/drive", force_remount=False)

MYDRIVE = Path("/content/drive/MyDrive")
filename = "09_project_source_file_inventory.csv"

print("Searching Google Drive for:", filename)

matches = list(MYDRIVE.rglob(filename))

if matches:
    print(f"\nFound {len(matches)} matching file(s).")

    for i, path in enumerate(matches, start=1):
        print(f"\nMatch {i}: {path}")

    inventory_path = matches[0]
    df = pd.read_csv(inventory_path)

    print("\nTABLE 1 — LARGEST NON-EMPTY DATA FILES")
    display(
        df[df["size_bytes"] > 0]
        .sort_values("size_bytes", ascending=False)
        .head(30)
    )

    print("\nTABLE 2 — LIKELY GROUNDWATER FILES")

    keywords = [
        "groundwater", "water_level", "waterlevel", "well",
        "observation", "seasonal", "dtw", "gwl", "manual",
        "cgwb", "dwlr", "piezometer"
    ]

    mask = df["path"].astype(str).str.lower().apply(
        lambda p: any(k in p for k in keywords)
    )

    display(df.loc[mask].head(50))

else:
    print("\nInventory file was not found anywhere in My Drive.")
    print("\nChecking whether the original source folder exists...")

    root = MYDRIVE / "gw_framework" / "data" / "gwl"

    print("Groundwater folder exists:", root.exists())

    if root.exists():
        print("\nFiles currently in the groundwater folder:")
        for p in list(root.rglob("*"))[:100]:
            if p.is_file():
                print(p)
    else:
        print("The expected groundwater folder is also missing.")
        print("Check that you are signed into the correct Google account.")


**[orig #78]** archived — folder listing probe  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

root = Path("/content/drive/MyDrive/gw_framework/data/gwl")

print("Groundwater folder exists:", root.exists())

if root.exists():
    for folder in [
        root,
        root / "processed",
        root / "diagnostics",
        root / "diagnostics" / "Study_Area_Verification"
    ]:
        print(f"\n--- {folder} ---")
        if folder.exists():
            for p in sorted(folder.iterdir()):
                if p.is_file():
                    print(f"{p.name} | {p.stat().st_size:,} bytes")
                else:
                    print(f"{p.name}/")
        else:
            print("Folder not found")


**[orig #80]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/gw_framework")
MYDRIVE = Path("/content/drive/MyDrive")

# Search likely source folders first
search_roots = [
    BASE / "data" / "gwl" / "raw_wris",
    BASE / "data" / "gwl" / "processed" / "_parsed",
    BASE / "data" / "inputs",
    BASE / "data",
    BASE
]

extensions = {".csv", ".xlsx", ".xls", ".parquet", ".zip"}
seen = set()
results = []

for root in search_roots:
    if not root.exists():
        print("Folder missing:", root)
        continue

    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in extensions:
            continue

        key = str(p.resolve())
        if key in seen:
            continue
        seen.add(key)

        name = p.name.lower()
        try:
            size = p.stat().st_size
        except OSError:
            continue

        # Retain likely groundwater sources and all files in raw/parsed folders.
        path_lower = str(p).lower()
        relevant = (
            any(x in name for x in [
                "wells_levels", "groundwater", "water_level",
                "waterlevel", "manual", "dwlr", "well", "observation"
            ])
            or "/raw_wris/" in path_lower
            or "/_parsed/" in path_lower
        )

        if relevant:
            results.append({
                "filename": p.name,
                "path": str(p),
                "size_bytes": size,
                "extension": p.suffix.lower()
            })

found = pd.DataFrame(results)

print("\nLIKELY GROUNDWATER SOURCE FILES")
if found.empty:
    print("No likely source files found in the searched project folders.")
else:
    display(
        found.sort_values("size_bytes", ascending=False)
        .head(100)
    )

# Check the exact expected original filename across My Drive.
print("\nEXACT ORIGINAL FILENAME SEARCH")

exact_matches = [
    p for p in MYDRIVE.rglob("*")
    if p.is_file()
    and p.name.lower() == "wells_levels_user_original.csv"
]

if exact_matches:
    for p in exact_matches:
        print(f"FOUND: {p} | {p.stat().st_size:,} bytes")
        try:
            source = pd.read_csv(p, low_memory=False)
            print("Shape:", source.shape)
            print("Columns:", source.columns.tolist())
            display(source.head(5))
        except Exception as e:
            print("Could not read file:", repr(e))
else:
    print("Original filename not found in My Drive.")

# Save a source-location report
out = (
    BASE / "data" / "gwl" / "diagnostics"
    / "Study_Area_Verification"
)
out.mkdir(parents=True, exist_ok=True)

if not found.empty:
    found.to_csv(out / "12_likely_groundwater_source_files.csv", index=False)

print("\nSearch completed. No files were modified.")


**[orig #81]** archived — pasted output text, not code  
<sub>⚠ last run ended with SyntaxError</sub>

In [ ]:
 Folder missing: /content/drive/MyDrive/gw_framework/data/inputs

LIKELY GROUNDWATER SOURCE FILES




    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }





      filename
      path
      size_bytes
      extension




      23
      groundwater_measurements_with_estimated_head.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      33903033
      .csv


      4
      groundwater_long_seasonal.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      31941421
      .csv


      11
      groundwater_cleaned_long_seasonal.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      21853754
      .csv


      7
      groundwater_value_review.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      15762595
      .csv


      15
      groundwater_numeric_readings.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      10327514
      .csv


      12
      groundwater_measurement_review.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      9314211
      .csv


      13
      groundwater_no_reading_observations.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      9184896
      .csv


      14
      groundwater_dry_observations.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      2341982
      .csv


      10
      groundwater_cleaned_wide_source_preserved.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      1468514
      .csv


      3
      groundwater_combined_wide.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      1400742
      .csv


      22
      well_inventory_with_dem_elevation.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      909981
      .csv


      30
      manual_observations_audit.csv
      /content/drive/MyDrive/gw_framework/data/gwl/d...
      277585
      .csv


      39
      wells_levels_user_original.csv
      /content/drive/MyDrive/gw_framework/inputs/gwl...
      158911
      .csv


      34
      wells_levels.csv
      /content/drive/MyDrive/gw_framework/inputs/wel...
      158911
      .csv


      0
      895eea4cddcbde00_man.parquet
      /content/drive/MyDrive/gw_framework/data/gwl/p...
      27027
      .parquet


      20
      groundwater_water_level_column_report.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      16376
      .csv


      2
      895eea4cddcbde00_meta.parquet
      /content/drive/MyDrive/gw_framework/data/gwl/p...
      9851
      .parquet


      26
      well_id_coordinate_mapping_crosscheck.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      7205
      .csv


      28
      readings_manual_qc.parquet
      /content/drive/MyDrive/gw_framework/data/gwl/p...
      4474
      .parquet


      32
      02_well_location_audit.csv
      /content/drive/MyDrive/gw_framework/data/gwl/d...
      3791
      .csv


      36
      well_metrics.csv
      /content/drive/MyDrive/gw_framework/work/03_ob...
      3337
      .csv


      29
      dwlr_daily_qc.parquet
      /content/drive/MyDrive/gw_framework/data/gwl/p...
      2948
      .parquet


      8
      groundwater_value_summary_by_season.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      2344
      .csv


      18
      groundwater_seasonal_status_summary.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      2271
      .csv


      1
      895eea4cddcbde00_tel.parquet
      /content/drive/MyDrive/gw_framework/data/gwl/p...
      2241
      .parquet


      31
      well_metadata_validation.csv
      /content/drive/MyDrive/gw_framework/data/gwl/d...
      2182
      .csv


      19
      groundwater_processing_file_report.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      1017
      .csv


      33
      11_groundwater_source_inspection.csv
      /content/drive/MyDrive/gw_framework/data/gwl/d...
      908
      .csv


      6
      duplicate_well_IDs.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      654
      .csv


      38
      wells_meta.csv
      /content/drive/MyDrive/gw_framework/work/05_ca...
      630
      .csv


      37
      wells_meta.csv
      /content/drive/MyDrive/gw_framework/work/05_ca...
      630
      .csv


      35
      wells_meta.csv
      /content/drive/MyDrive/gw_framework/work/03_ob...
      630
      .csv


      25
      wells_without_dem_elevation.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      520
      .csv


      24
      wells_missing_or_invalid_coordinates.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      520
      .csv


      21
      groundwater_cleaning_QC_summary.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      475
      .csv


      27
      wells_meta.csv
      /content/drive/MyDrive/gw_framework/data/gwl/p...
      319
      .csv


      9
      groundwater_value_flag_summary.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      188
      .csv


      5
      groundwater_QC_summary.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      165
      .csv


      17
      groundwater_review_flag_summary.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      128
      .csv


      16
      groundwater_status_summary.csv
      /content/drive/MyDrive/gw_framework/data/Rajas...
      80
      .csv















    .colab-df-container {
      display:flex;
      gap: 12px;
    }

    .colab-df-convert {
      background-color: #E8F0FE;
      border: none;
      border-radius: 50%;
      cursor: pointer;
      display: none;
      fill: #1967D2;
      height: 32px;
      padding: 0 0 0 0;
      width: 32px;
    }

    .colab-df-convert:hover {
      background-color: #E2EBFA;
      box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);
      fill: #174EA6;
    }

    .colab-df-buttons div {
      margin-bottom: 4px;
    }

    [theme=dark] .colab-df-convert {
      background-color: #3B4455;
      fill: #D2E3FC;
    }

    [theme=dark] .colab-df-convert:hover {
      background-color: #434B5C;
      box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);
      filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));
      fill: #FFFFFF;
    }



      const buttonEl =
        document.querySelector('#df-39445f6f-950e-445b-9e26-db1f2e4f5734 button.colab-df-convert');
      buttonEl.style.display =
        google.colab.kernel.accessAllowed ? 'block' : 'none';

      async function convertToInteractive(key) {
        const element = document.querySelector('#df-39445f6f-950e-445b-9e26-db1f2e4f5734');
        const dataTable =
          await google.colab.kernel.invokeFunction('convertToInteractive',
                                                    [key], {});
        if (!dataTable) return;

        const docLinkHtml = 'Like what you see? Visit the ' +
          '<a target="_blank" href=https://colab.research.google.com/notebooks/data_table.ipynb>data table notebook</a>'
          + ' to learn more about interactive tables.';
        element.innerHTML = '';
        dataTable['output_type'] = 'display_data';
        await google.colab.output.renderOutput(dataTable, element);
        const docLink = document.createElement('div');
        docLink.innerHTML = docLinkHtml;
        element.appendChild(docLink);
      }







EXACT ORIGINAL FILENAME SEARCH
FOUND: /content/drive/MyDrive/gw_framework/inputs/gwl/wells_levels_user_original.csv | 158,911 bytes
Shape: (3813, 6)
Columns: ['well_id', 'lat', 'lon', 'date', 'dtw_mbgl', 'well_depth_m']




    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }





      well_id
      lat
      lon
      date
      dtw_mbgl
      well_depth_m




      0
      W01
      22.1307
      76.2838
      1980-01-15
      33.84
      15.0


      1
      W01
      22.1307
      76.2838
      1980-05-15
      31.83
      15.0


      2
      W01
      22.1307
      76.2838
      1981-01-15
      32.92
      15.0


      3
      W01
      22.1307
      76.2838
      1981-05-15
      32.54
      15.0


      4
      W01
      22.1307
      76.2838
      1981-11-15
      31.36
      15.0















    .colab-df-container {
      display:flex;
      gap: 12px;
    }

    .colab-df-convert {
      background-color: #E8F0FE;
      border: none;
      border-radius: 50%;
      cursor: pointer;
      display: none;
      fill: #1967D2;
      height: 32px;
      padding: 0 0 0 0;
      width: 32px;
    }

    .colab-df-convert:hover {
      background-color: #E2EBFA;
      box-shadow: 0px 1px 2px rgba(60, 64, 67, 0.3), 0px 1px 3px 1px rgba(60, 64, 67, 0.15);
      fill: #174EA6;
    }

    .colab-df-buttons div {
      margin-bottom: 4px;
    }

    [theme=dark] .colab-df-convert {
      background-color: #3B4455;
      fill: #D2E3FC;
    }

    [theme=dark] .colab-df-convert:hover {
      background-color: #434B5C;
      box-shadow: 0px 1px 3px 1px rgba(0, 0, 0, 0.15);
      filter: drop-shadow(0px 1px 2px rgba(0, 0, 0, 0.3));
      fill: #FFFFFF;
    }



      const buttonEl =
        document.querySelector('#df-bc452494-0984-42c9-9ece-00f487640c2b button.colab-df-convert');
      buttonEl.style.display =
        google.colab.kernel.accessAllowed ? 'block' : 'none';

      async function convertToInteractive(key) {
        const element = document.querySelector('#df-bc452494-0984-42c9-9ece-00f487640c2b');
        const dataTable =
          await google.colab.kernel.invokeFunction('convertToInteractive',
                                                    [key], {});
        if (!dataTable) return;

        const docLinkHtml = 'Like what you see? Visit the ' +
          '<a target="_blank" href=https://colab.research.google.com/notebooks/data_table.ipynb>data table notebook</a>'
          + ' to learn more about interactive tables.';
        element.innerHTML = '';
        dataTable['output_type'] = 'display_data';
        await google.colab.output.renderOutput(dataTable, element);
        const docLink = document.createElement('div');
        docLink.innerHTML = docLinkHtml;
        element.appendChild(docLink);
      }







Search completed. No files were modified.


**[orig #84]** archived — empty probe  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from pathlib import Path

folder = Path("/content/drive/MyDrive/gw_framework/inputs/gwl")

for path in sorted(folder.rglob("*")):
    if path.is_file():
        print(path)

**[orig #87]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/gw_framework")

targets = [
    "Alwar.xlsx",
    "Alwar NHP.xlsx",
    "groundwater_measurements_with_estimated_head.csv",
    "groundwater_cleaned_long_seasonal.csv",
]

for name in targets:
    print(f"\n{'='*70}\nSearching for: {name}")
    matches = [
        p for p in ROOT.rglob("*")
        if p.is_file() and p.name.lower() == name.lower()
    ]

    if matches:
        for p in matches:
            print("FOUND:", p)
    else:
        print("NOT FOUND anywhere under:", ROOT)

print("\nSearch complete.")


**[orig #89]** archived — Drive check probe  
<sub>✅ ran OK</sub>

In [ ]:
from pathlib import Path
import os
import pandas as pd

DRIVE = Path("/content/drive/MyDrive")

print("=" * 70)
print("GOOGLE DRIVE CHECK")
print("=" * 70)
print("MyDrive exists:", DRIVE.exists())
print("MyDrive path:", DRIVE.resolve() if DRIVE.exists() else "NOT FOUND")

if not DRIVE.exists():
    print("\nDrive is not mounted correctly.")
    print("Run the following in a separate cell:")
    print("from google.colab import drive")
    print("drive.mount('/content/drive')")
else:
    print("\nTop-level MyDrive folders:")
    for p in sorted(DRIVE.iterdir(), key=lambda x: x.name.lower()):
        print(("DIR " if p.is_dir() else "FILE"), p.name)

    print("\n" + "=" * 70)
    print("SEARCHING FOR GROUNDWATER SOURCE FILES")
    print("=" * 70)

    keywords = (
        "alwar",
        "groundwater",
        "ground_water",
        "water_level",
        "waterlevel",
        "well_level",
        "head_obs",
        "heads.csv",
        "observation",
        "seasonal",
        "wris",
        "nhp",
        "gwl",
    )

    extensions = {".csv", ".xlsx", ".xls", ".parquet", ".zip", ".shp"}

    results = []
    errors = []

    for root, dirs, files in os.walk(DRIVE):
        # Avoid common cache folders that are unlikely to contain source data.
        dirs[:] = [
            d for d in dirs
            if d not in {".ipynb_checkpoints", "__pycache__"}
        ]

        for filename in files:
            path = Path(root) / filename

            if path.suffix.lower() not in extensions:
                continue

            if any(k in filename.lower() for k in keywords):
                try:
                    results.append({
                        "filename": filename,
                        "extension": path.suffix.lower(),
                        "size_MB": round(path.stat().st_size / (1024**2), 3),
                        "full_path": str(path),
                    })
                except OSError as e:
                    errors.append({"path": str(path), "error": str(e)})

    if results:
        inventory = pd.DataFrame(results).sort_values(
            ["filename", "full_path"]
        )
        display(inventory)
        print(f"\nMatching files found: {len(inventory)}")
    else:
        print("No files matching the search terms were found.")

    if errors:
        print("\nFiles that could not be inspected:", len(errors))
        display(pd.DataFrame(errors))

    output = DRIVE / "groundwater_source_recovery_inventory.csv"

    if results:
        pd.DataFrame(results).to_csv(output, index=False)
        print("\nInventory saved to:", output)

**[orig #90]** archived — Drive mount only  
<sub>✅ ran OK</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**[orig #91]** archived — failed (MergeError); fixed in Step 2 cells  
<sub>⚠ last run ended with MergeError</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path("/content/drive/MyDrive/gw_framework")

ALWAR_FILE = ROOT / "data/Rajasthan_gw_data/Alwar.xlsx"
NHP_FILE = ROOT / "data/BDR/Alwar/nhpbdr 2023/Alwar NHP.xlsx"

SEASONAL_FILE = (
    ROOT / "data/Rajasthan_gw_data/"
    "Groundwater_Cleaning_According_to_User_Rules/"
    "groundwater_cleaned_long_seasonal.csv"
)

HEAD_FILE = (
    ROOT / "data/Rajasthan_gw_data/Well_Elevation_and_Head_Estimates/"
    "groundwater_measurements_with_estimated_head.csv"
)

OUT = ROOT / "data/gwl/diagnostics/Alwar_Source_Recovery/Step_2_QC"
OUT.mkdir(parents=True, exist_ok=True)

def read_csv(path):
    return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)

def save_csv(df, name):
    path = OUT / name
    df.to_csv(path, index=False)
    print("Saved:", path, "| rows:", len(df))

def numeric(series):
    return pd.to_numeric(series, errors="coerce")

def norm_text(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", " ", regex=True)
    )

# ---------------------------------------------------------
# 1. Load source datasets
# ---------------------------------------------------------
for path in [ALWAR_FILE, NHP_FILE, SEASONAL_FILE, HEAD_FILE]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

wells = pd.read_excel(ALWAR_FILE, sheet_name="Alwar")
seasonal = read_csv(SEASONAL_FILE)
heads = read_csv(HEAD_FILE)

print("\nSOURCE SIZES")
print("Alwar wells:", wells.shape)
print("Seasonal observations:", seasonal.shape)
print("Estimated-head records:", heads.shape)

# ---------------------------------------------------------
# 2. Well coordinate and metadata audit
# ---------------------------------------------------------
wells.columns = wells.columns.astype(str).str.strip()

for col in ["Latitude", "Longitude", "RL (m)", "Total_Depth_bgl"]:
    if col in wells.columns:
        wells[col] = numeric(wells[col])

lat = wells["Latitude"]
lon = wells["Longitude"]

wells["coordinate_missing"] = lat.isna() | lon.isna()
wells["coordinate_out_of_global_range"] = (
    lat.notna() & lon.notna() &
    ~lat.between(-90, 90) |
    lon.notna() & ~lon.between(-180, 180)
)

# Broad screening envelope only; NOT a legal district boundary.
wells["inside_provisional_alwar_envelope"] = (
    lat.between(27.0, 28.2) & lon.between(76.0, 77.3)
)

wells["well_id_check"] = (
    wells["WIMS_ID"].astype("string").str.strip()
    if "WIMS_ID" in wells.columns
    else wells["ROLTA_ID/UID"].astype("string").str.strip()
)

print("\nCOORDINATE SUMMARY")
print("Total well records:", len(wells))
print("Missing coordinates:", int(wells["coordinate_missing"].sum()))
print(
    "Inside broad screening envelope:",
    int(wells["inside_provisional_alwar_envelope"].sum())
)
print(
    "Outside broad screening envelope:",
    int((~wells["inside_provisional_alwar_envelope"]).sum())
)

save_csv(
    wells.loc[
        wells["coordinate_missing"] |
        ~wells["inside_provisional_alwar_envelope"]
    ],
    "01_coordinate_review.csv"
)

# ---------------------------------------------------------
# 3. Historical observation QC
# ---------------------------------------------------------
seasonal["WL_bgl_num"] = numeric(seasonal["WL_bgl"])
seasonal["Year_num"] = numeric(seasonal["Year"])

status = (
    seasonal.groupby(["WL_status"], dropna=False)
    .size().reset_index(name="records")
)
save_csv(status, "02_observation_status_counts.csv")

numeric_obs = seasonal[seasonal["WL_bgl_num"].notna()].copy()

numeric_obs["negative_dtw"] = numeric_obs["WL_bgl_num"] < 0
numeric_obs["zero_dtw"] = numeric_obs["WL_bgl_num"] == 0
numeric_obs["over_100m_dtw"] = numeric_obs["WL_bgl_num"] > 100

depth_col = "Total_Depth_numeric"
if depth_col in numeric_obs.columns:
    numeric_obs[depth_col] = numeric(numeric_obs[depth_col])
    numeric_obs["deeper_than_reported_well_depth"] = (
        numeric_obs[depth_col].notna() &
        (numeric_obs["WL_bgl_num"] > numeric_obs[depth_col])
    )
else:
    numeric_obs["deeper_than_reported_well_depth"] = np.nan

print("\nNUMERIC OBSERVATION QC")
for col in [
    "negative_dtw", "zero_dtw", "over_100m_dtw",
    "deeper_than_reported_well_depth"
]:
    print(col, "=", int(numeric_obs[col].fillna(False).sum()))

save_csv(numeric_obs, "03_numeric_observations_for_review.csv")

# ---------------------------------------------------------
# 4. Reconcile seasonal observations against estimated-head file
# ---------------------------------------------------------
key_cols = [
    "Source_file", "Source_sheet", "Source_row_number",
    "Year", "Season", "Original_WL_Column"
]

required_seasonal = key_cols + ["WL_bgl", "WL_status"]
required_heads = [
    "Source_file", "Source_sheet", "Source_row_number",
    "Year", "Season", "Original_WL_Column",
    "Depth_to_water_bgl_m", "Estimated_head_m_AMSL",
    "DEM_ground_elevation_m", "Head_estimate_quality",
    "Elevation_status", "Elevation_review_flag"
]

missing_s = [c for c in required_seasonal if c not in seasonal.columns]
missing_h = [c for c in required_heads if c not in heads.columns]

if missing_s or missing_h:
    print("\nCannot perform full reconciliation.")
    print("Missing seasonal columns:", missing_s)
    print("Missing estimated-head columns:", missing_h)
else:
    left = seasonal[required_seasonal].copy()
    right = heads[required_heads].copy()

    for df in [left, right]:
        for c in key_cols:
            df[c] = df[c].astype("string").str.strip()

    left["WL_bgl_num"] = numeric(left["WL_bgl"])
    right["DTW_num"] = numeric(right["Depth_to_water_bgl_m"])
    right["Head_num"] = numeric(right["Estimated_head_m_AMSL"])
    right["DEM_num"] = numeric(right["DEM_ground_elevation_m"])

    merged = left.merge(
        right,
        on=key_cols,
        how="outer",
        indicator=True,
        validate="one_to_one"
    )

    merged["dtw_difference_m"] = (
        merged["WL_bgl_num"] - merged["DTW_num"]
    )

    merged["head_equation_difference_m"] = (
        merged["Head_num"] - (merged["DEM_num"] - merged["DTW_num"])
    )

    merged["dtw_mismatch_over_0_01m"] = (
        merged["dtw_difference_m"].abs() > 0.01
    )

    merged["head_equation_mismatch_over_0_01m"] = (
        merged["head_equation_difference_m"].abs() > 0.01
    )

    print("\nRECONCILIATION")
    print(merged["_merge"].value_counts(dropna=False))
    print(
        "DTW mismatches > 0.01 m:",
        int(merged["dtw_mismatch_over_0_01m"].fillna(False).sum())
    )
    print(
        "Head equation mismatches > 0.01 m:",
        int(merged["head_equation_mismatch_over_0_01m"].fillna(False).sum())
    )

    save_csv(merged, "04_seasonal_vs_estimated_head_reconciliation.csv")

    quality_summary = (
        heads.groupby(
            ["Head_estimate_quality", "Elevation_status",
             "Elevation_review_flag"],
            dropna=False
        ).size().reset_index(name="records")
    )
    save_csv(quality_summary, "05_head_quality_summary.csv")

# ---------------------------------------------------------
# 5. Historical observation counts by well
# ---------------------------------------------------------
id_col = "Well_ID_source"
if id_col in seasonal.columns:
    well_summary = (
        seasonal.groupby(id_col, dropna=False)
        .agg(
            records=("WL_status", "size"),
            numeric_readings=("WL_bgl_num", "count"),
            first_year=("Year_num", "min"),
            last_year=("Year_num", "max"),
            blocks=("Block", "nunique")
        )
        .reset_index()
    )
    save_csv(well_summary, "06_historical_well_coverage.csv")

# ---------------------------------------------------------
# 6. NHP workbook: concise sheet inventory
# ---------------------------------------------------------
nhp_xls = pd.ExcelFile(NHP_FILE)
nhp_inventory = []

for sheet in nhp_xls.sheet_names:
    df = pd.read_excel(NHP_FILE, sheet_name=sheet, header=None)
    nhp_inventory.append({
        "sheet": sheet,
        "rows": len(df),
        "columns": len(df.columns),
        "first_row": " | ".join(
            str(v) for v in df.iloc[0].tolist()
            if pd.notna(v)
        ) if len(df) else ""
    })

save_csv(pd.DataFrame(nhp_inventory), "07_nhp_sheet_inventory.csv")

# ---------------------------------------------------------
# 7. Summary report
# ---------------------------------------------------------
summary = {
    "source_file": str(ALWAR_FILE),
    "well_metadata_rows": int(len(wells)),
    "coordinate_missing": int(wells["coordinate_missing"].sum()),
    "inside_provisional_envelope": int(
        wells["inside_provisional_alwar_envelope"].sum()
    ),
    "seasonal_records": int(len(seasonal)),
    "numeric_seasonal_readings": int(seasonal["WL_bgl_num"].notna().sum()),
    "estimated_head_records": int(len(heads)),
    "nhp_sheets": nhp_xls.sheet_names,
    "important_note": (
        "The geographic envelope is only a screening check. "
        "Use an authoritative Alwar district boundary for final inclusion. "
        "DEM-derived heads are provisional and should not automatically "
        "be treated as observed calibration heads."
    )
}

summary_path = OUT / "00_audit_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, default=str)

print("\n" + "=" * 70)
print("AUDIT COMPLETE")
print("=" * 70)
print(json.dumps(summary, indent=2, default=str))
print("\nOutputs saved under:", OUT)
print("Original source files were not modified.")


**[orig #98]** archived — failed (KeyError); redone in Step 4 cell  
<sub>⚠ last run ended with KeyError</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

INPUT = Path(
    "/content/drive/MyDrive/gw_framework/data/"
    "Rajasthan_gw_data/Groundwater_Cleaning_According_to_User_Rules/"
    "groundwater_cleaned_long_seasonal.csv"
)

OUT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_4_Observation_Coverage"
)
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT, low_memory=False)

# Select Alwar rows using district labels
alwar = df[
    df["District"].astype("string").str.strip().str.casefold()
    == "alwar"
].copy()

alwar["WL_bgl"] = pd.to_numeric(alwar["WL_bgl"], errors="coerce")
alwar["Year"] = pd.to_numeric(alwar["Year"], errors="coerce")

print("Alwar records:", len(alwar))

# Find a stable well identifier available in the file
id_col = next(
    (c for c in ["Well_key", "Well_ID_source", "Well_ID", "Well_ID_Original"]
     if c in alwar.columns),
    None
)
if id_col is None:
    raise ValueError(
        "No recognized well identifier found. "
        f"Available columns: {alwar.columns.tolist()}"
    )

print("Well identifier:", id_col)

# Normalize status labels for reporting only
alwar["status_clean"] = (
    alwar["WL_status"].astype("string").str.strip().str.casefold()
)

# 1. Well-level summary
well = alwar.groupby(id_col, dropna=False).agg(
    records=(id_col, "size"),
    numeric_readings=("WL_bgl", "count"),
    first_year=("Year", "min"),
    last_year=("Year", "max"),
    number_of_seasons=("Season", "nunique")
).reset_index()

status_counts = pd.crosstab(
    alwar[id_col], alwar["status_clean"]
).reset_index()

well = well.merge(status_counts, on=id_col, how="left")

for col in ["dry", "no reading", "numeric reading"]:
    if col not in well.columns:
        well[col] = 0

well["well_class"] = np.select(
    [
        well["numeric_readings"].eq(0),
        well["numeric_readings"].gt(0)
        & well["dry"].gt(0),
        well["numeric_readings"].gt(0)
    ],
    [
        "NO_NUMERIC_DTW",
        "NUMERIC_DTW_AND_DRY_STATUS",
        "NUMERIC_DTW_PRESENT"
    ],
    default="REVIEW"
)

well["numeric_coverage_percent"] = (
    100 * well["numeric_readings"] / well["records"]
)

well.to_csv(OUT / "01_alwar_well_observation_summary.csv", index=False)

# 2. Annual and seasonal observation coverage
coverage = (
    alwar.groupby(["Year", "Season"], dropna=False)
    .agg(
        records=(id_col, "size"),
        wells=(id_col, "nunique"),
        numeric_readings=("WL_bgl", "count")
    )
    .reset_index()
)

coverage["numeric_coverage_percent"] = (
    100 * coverage["numeric_readings"] / coverage["records"]
)

coverage.to_csv(
    OUT / "02_alwar_year_season_coverage.csv", index=False
)

# 3. Annual status counts
annual = (
    alwar.groupby(["Year", "status_clean"], dropna=False)
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
annual.to_csv(OUT / "03_alwar_annual_status_counts.csv", index=False)

# 4. Summary
summary = {
    "alwar_records": int(len(alwar)),
    "unique_wells": int(alwar[id_col].nunique()),
    "wells_with_numeric_dtw": int(
        well.loc[well["numeric_readings"] > 0, id_col].nunique()
    ),
    "wells_without_numeric_dtw": int(
        well.loc[well["numeric_readings"] == 0, id_col].nunique()
    ),
    "wells_with_both_numeric_and_dry_status": int(
        ((well["numeric_readings"] > 0) & (well["dry"] > 0)).sum()
    ),
    "wells_with_only_one_numeric_reading": int(
        (well["numeric_readings"] == 1).sum()
    ),
    "wells_with_at_least_5_numeric_readings": int(
        (well["numeric_readings"] >= 5).sum()
    )
}

pd.Series(summary, name="value").to_csv(
    OUT / "04_alwar_coverage_summary.csv"
)

print("\nSUMMARY")
for k, v in summary.items():
    print(f"{k}: {v}")

print("\nSaved to:", OUT)


**[orig #105]** archived — Drive mount only  
<sub>✅ ran OK</sub>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

**[orig #106]** archived — patch snippet; Step 8 cell is self-contained  
<sub>✅ ran OK</sub>

In [ ]:
# ============================================================
# FIX FOR KEYERROR: 'Season_clean'
# Replace the old merge section with this code.
# ============================================================

# Normalize season names in BOTH DataFrames
trends["Season_clean"] = (
    trends["Season"].astype(str).str.strip().str.lower()
)

diag["Season_clean"] = (
    diag["Season"].astype(str).str.strip().str.lower()
)

# Normalize well keys to strings in both DataFrames
trends["Well_key"] = trends["Well_key"].astype(str).str.strip()
diag["Well_key"] = diag["Well_key"].astype(str).str.strip()

# Check for duplicate merge keys before merging
if trends.duplicated(["Well_key", "Season_clean"]).any():
    raise ValueError(
        "Duplicate Well_key + Season keys in trend results. "
        "Review these before merging."
    )

if diag.duplicated(["Well_key", "Season_clean"]).any():
    raise ValueError(
        "Duplicate Well_key + Season keys in diagnostics. "
        "Review these before merging."
    )

# Merge trend results with temporal/autocorrelation diagnostics
result = trends.merge(
    diag,
    on=["Well_key", "Season_clean"],
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Merge completed.")
print("Trend rows:", len(trends))
print("Diagnostic rows:", len(diag))
print("Merged rows:", len(result))
print("\nMerge status:")
print(result["_merge"].value_counts(dropna=False))

# Any unmatched series need investigation
unmatched = result.loc[result["_merge"] != "both"].copy()

if not unmatched.empty:
    print("\nWARNING: Some trend series did not match diagnostics.")
    display(unmatched[["Well_key", "Season", "Season_clean", "_merge"]].head(20))

# Remove the temporary merge indicator after checking
result.drop(columns=["_merge"], inplace=True)

**[orig #113]** archived — failed (KeyError)  
<sub>⚠ last run ended with KeyError</sub>

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json

ROOT = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/"
    "diagnostics/Alwar_Source_Recovery"
)

STEP8 = ROOT / "Step_8_Trend_Validation"
STEP10B = ROOT / "Step_10B_Candidate_TFPW_Recalculation"
STEP11 = ROOT / "Step_11_Trend_Robustness_Comparison"

F8 = STEP8 / "01_validated_well_season_trends.csv"
F10 = STEP10B / "01_candidate_tfpw_results.csv"
F11 = STEP11 / "01_step8_vs_step10b_comparison.csv"

OUT = ROOT / "Step_12_Trend_Direction_QC"
OUT.mkdir(parents=True, exist_ok=True)

for f in [F8, F10, F11]:
    if not f.exists():
        raise FileNotFoundError(f"Missing file: {f}")

s8 = pd.read_csv(F8)
s10 = pd.read_csv(F10)
s11 = pd.read_csv(F11)

def normalize(df):
    df = df.copy()
    df["Well_key"] = df["Well_key"].astype(str).str.strip()
    df["Season_clean"] = (
        df["Season"].astype(str).str.strip().str.lower()
    )
    return df

s8, s10, s11 = map(normalize, [s8, s10, s11])
keys = ["Well_key", "Season_clean"]

# Merge only the original 186 candidate series
d = s8.merge(
    s10,
    on=keys,
    how="left",
    suffixes=("_step8", "_step10b"),
    validate="one_to_one"
)

if len(d) != 186:
    raise ValueError(f"Expected 186 candidate series, found {len(d)}")

# Explicitly name the two original-scale Sen slopes
slope8 = pd.to_numeric(
    d["sen_slope_m_per_year_step8"], errors="coerce"
)
slope10 = pd.to_numeric(
    d["original_sen_slope_m_per_year"], errors="coerce"
)

d["step8_sen_slope"] = slope8
d["step10b_original_sen_slope"] = slope10

d["step8_direction"] = np.select(
    [slope8 > 0, slope8 < 0],
    ["DTW_INCREASE", "DTW_DECREASE"],
    default="ZERO_OR_MISSING"
)

d["step10b_direction"] = np.select(
    [slope10 > 0, slope10 < 0],
    ["DTW_INCREASE", "DTW_DECREASE"],
    default="ZERO_OR_MISSING"
)

d["direction_agreement"] = np.where(
    slope8.notna() & slope10.notna(),
    np.sign(slope8) == np.sign(slope10),
    np.nan
)

d["absolute_slope_difference_m_per_year"] = (
    slope8 - slope10
).abs()

# Identify candidates significant under both procedures
step8_fdr = pd.to_numeric(d["mk_p_fdr_bh"], errors="coerce")
tfpw_fdr = pd.to_numeric(d["tfpw_p_fdr_bh"], errors="coerce")

tfpw_applied = (
    d["tfpw_applied"].astype(str).str.lower().isin(["true", "1"])
)

d["step8_fdr_significant"] = step8_fdr < 0.05
d["tfpw_fdr_significant"] = (
    tfpw_applied & tfpw_fdr.notna() & (tfpw_fdr < 0.05)
)

d["significant_in_both"] = (
    d["step8_fdr_significant"] & d["tfpw_fdr_significant"]
)

# Flag slopes for manual review; these thresholds are screening flags,
# not automatic evidence that a slope is wrong.
d["direction_review_flag"] = "NONE"

d.loc[
    d["significant_in_both"] & (d["direction_agreement"] == False),
    "direction_review_flag"
] = "SIGNIFICANT_BOTH_DIRECTION_DISAGREEMENT"

d.loc[
    d["significant_in_both"] & d["direction_agreement"].isna(),
    "direction_review_flag"
] = "SIGNIFICANT_BOTH_DIRECTION_UNAVAILABLE"

d.loc[
    d["absolute_slope_difference_m_per_year"] > 0.5,
    "direction_review_flag"
] = "LARGE_SLOPE_DIFFERENCE_REVIEW"

d.to_csv(OUT / "01_candidate_direction_audit.csv", index=False)

review = d.loc[
    (d["significant_in_both"]) |
    (d["direction_review_flag"] != "NONE")
].copy()

review.to_csv(OUT / "02_manual_trend_review.csv", index=False)

summary = {
    "candidate_series": int(len(d)),
    "significant_in_both": int(d["significant_in_both"].sum()),
    "both_significant_direction_agrees": int(
        (
            d["significant_in_both"] &
            (d["direction_agreement"] == True)
        ).sum()
    ),
    "both_significant_direction_disagrees": int(
        (
            d["significant_in_both"] &
            (d["direction_agreement"] == False)
        ).sum()
    ),
    "both_significant_direction_unavailable": int(
        (
            d["significant_in_both"] &
            d["direction_agreement"].isna()
        ).sum()
    ),
    "large_slope_difference_flagged": int(
        (d["absolute_slope_difference_m_per_year"] > 0.5).sum()
    ),
    "note": (
        "Direction comparison is a QC screen. Investigate any mismatches "
        "and confirm slope units and source measurement conventions before "
        "final mapping or causal interpretation."
    )
}

with open(OUT / "03_step12_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 65)
print("STEP 12 COMPLETED")
print("=" * 65)
print(json.dumps(summary, indent=2))
print("\nOutput folder:", OUT)


**[orig #129]** archived — failed (ValueError); redone in Step 15A load cell  
<sub>⚠ last run ended with ValueError</sub>

In [ ]:

# ================================================================
# STEP 15A: SPATIALLY RESTRICTED GROUNDWATER TREND AUDIT
# Study: Selected 14-block Alwar analysis domain
#
# Uses:
#   1. Step 14F spatial inclusion list
#   2. Step 12 candidate direction audit
#   3. Step 10B TFPW sensitivity results
#   4. Deduplicated estimated-head observation data
#
# Original inputs are never modified.
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import numpy as np
import json
from datetime import datetime

# -------------------- 1. PATHS --------------------

BASE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery"
)

INCLUSION_FILE = (
    BASE / "Step_14F_Spatial_Analysis_Restriction"
    / "01_wells_in_14_block_analysis_domain.csv"
)

STEP12_FILE = (
    BASE / "Step_12_Trend_Direction_QC"
    / "01_candidate_direction_audit.csv"
)

TFPW_DIR = BASE / "Step_10B_Candidate_TFPW_Recalculation"

OBSERVATION_FILE = Path(
    "/content/drive/MyDrive/gw_framework/data/gwl/diagnostics/"
    "Alwar_Source_Recovery/Step_2_QC/"
    "estimated_heads_deduplicated_analysis_copy.csv"
)

OUT = BASE / "Step_15_Spatial_Trend_Restriction"
OUT.mkdir(parents=True, exist_ok=True)

# -------------------- 2. HELPER FUNCTIONS --------------------

def normalize_key(series):
    """Normalize well identifiers without changing source files."""
    return series.astype("string").str.strip()

def require_columns(df, columns, label):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise ValueError(
            f"{label}: required columns missing: {missing}\n"
            f"Available columns: {df.columns.tolist()}"
        )

def find_tfpw_csv(directory):
    """Find the TFPW result CSV conservatively."""
    if not directory.exists():
        raise FileNotFoundError(
            f"TFPW directory not found: {directory}"
        )

    files = sorted(
        p for p in directory.rglob("*.csv")
        if p.is_file()
    )

    print("\nCSV files in Step 10B:")
    for p in files:
        print(" -", p.relative_to(directory))

    # Prefer files whose names explicitly indicate the result table.
    candidates = [
        p for p in files
        if "tfpw" in p.name.lower()
        and any(term in p.name.lower()
                for term in ["result", "trend", "candidate", "comparison"])
    ]

    if len(candidates) == 1:
        return candidates[0]

    # Do not guess among multiple or unlabelled files.
    raise RuntimeError(
        "\nCould not uniquely identify the Step 10B TFPW results CSV.\n"
        "Review the filenames printed above, then set TFPW_FILE manually "
        "to the correct CSV path and rerun."
    )

# -------------------- 3. VALIDATE INPUT FILES --------------------

for p in [INCLUSION_FILE, STEP12_FILE, OBSERVATION_FILE]:
    if not p.exists():
        raise FileNotFoundError(f"Required input not found: {p}")

print("Input files found.")
print("Output directory:", OUT)

# -------------------- 4. READ INPUTS --------------------

included = pd.read_csv(INCLUSION_FILE, low_memory=False)
step12 = pd.read_csv(STEP12_FILE, low_memory=False)
observations = pd.read_csv(OBSERVATION_FILE, low_memory=False)

require_columns(included, ["Well_key"], "Step 14F inclusion file")
require_columns(step12, ["Well_key", "Season"], "Step 12 trend audit")
require_columns(observations, ["Well_key"], "Observation dataset")

# Locate the TFPW result file; the script stops if ambiguous.
TFPW_FILE = find_tfpw_csv(TFPW_DIR)
tfpw = pd.read_csv(TFPW_FILE, low_memory=False)

require_columns(tfpw, ["Well_key"], "Step 10B TFPW results")

# -------------------- 5. VALIDATE INCLUSION LIST --------------------

included["Well_key"] = normalize_key(included["Well_key"])
step12["Well_key"] = normalize_key(step12["Well_key"])
tfpw["Well_key"] = normalize_key(tfpw["Well_key"])
observations["Well_key"] = normalize_key(observations["Well_key"])

if included["Well_key"].isna().any() or (
    included["Well_key"].eq("").any()
):
    raise ValueError("Inclusion file contains blank Well_key values.")

if included["Well_key"].duplicated().any():
    duplicates = included.loc[
        included["Well_key"].duplicated(keep=False), "Well_key"
    ].unique().tolist()
    raise ValueError(
        f"Inclusion file has duplicate well keys: {duplicates[:20]}"
    )

allowed_keys = set(included["Well_key"].dropna())

print("\nINCLUSION LIST")
print("Unique included wells:", len(allowed_keys))

# -------------------- 6. AUDIT ORIGINAL TREND CANDIDATES --------------------

# Candidate series are defined by unique Well_key + Season.
step12["Season"] = step12["Season"].astype("string").str.strip()

series_key = ["Well_key", "Season"]

if step12[series_key].isna().any().any():
    raise ValueError("Step 12 contains missing Well_key or Season.")

if step12.duplicated(series_key).any():
    raise ValueError(
        "Step 12 has duplicate Well_key + Season rows. "
        "Resolve duplicates before continuing."
    )

step12["spatial_domain_status"] = np.where(
    step12["Well_key"].isin(allowed_keys),
    "INCLUDED_IN_14_BLOCK_DOMAIN",
    "OUTSIDE_SELECTED_14_BLOCK_DOMAIN"
)

restricted_step12 = step12.loc[
    step12["Well_key"].isin(allowed_keys)
].copy()

excluded_candidates = step12.loc[
    ~step12["Well_key"].isin(allowed_keys)
].copy()

# -------------------- 7. FILTER TFPW RESULTS --------------------

tfpw["spatial_domain_status"] = np.where(
    tfpw["Well_key"].isin(allowed_keys),
    "INCLUDED_IN_14_BLOCK_DOMAIN",
    "OUTSIDE_SELECTED_14_BLOCK_DOMAIN"
)

restricted_tfpw = tfpw.loc[
    tfpw["Well_key"].isin(allowed_keys)
].copy()

excluded_tfpw = tfpw.loc[
    ~tfpw["Well_key"].isin(allowed_keys)
].copy()

# -------------------- 8. FILTER OBSERVATIONS --------------------

observations["spatial_domain_status"] = np.where(
    observations["Well_key"].isin(allowed_keys),
    "INCLUDED_IN_14_BLOCK_DOMAIN",
    "OUTSIDE_SELECTED_14_BLOCK_DOMAIN"
)

restricted_observations = observations.loc[
    observations["Well_key"].isin(allowed_keys)
].copy()

excluded_observations = observations.loc[
    ~observations["Well_key"].isin(allowed_keys)
].copy()

# -------------------- 9. WELL RETENTION AUDIT --------------------

candidate_wells = set(step12["Well_key"].dropna())
retained_candidate_wells = candidate_wells & allowed_keys
excluded_candidate_wells = candidate_wells - allowed_keys

all_candidate_keys = step12[["Well_key"]].drop_duplicates().copy()
all_candidate_keys["spatial_domain_status"] = np.where(
    all_candidate_keys["Well_key"].isin(allowed_keys),
    "INCLUDED_IN_14_BLOCK_DOMAIN",
    "OUTSIDE_SELECTED_14_BLOCK_DOMAIN"
)

# Add series counts per well.
series_counts = (
    step12.groupby("Well_key", dropna=False)
    .size()
    .rename("original_candidate_series_count")
    .reset_index()
)
retained_counts = (
    restricted_step12.groupby("Well_key", dropna=False)
    .size()
    .rename("retained_candidate_series_count")
    .reset_index()
)

well_audit = all_candidate_keys.merge(
    series_counts, on="Well_key", how="left", validate="one_to_one"
).merge(
    retained_counts, on="Well_key", how="left", validate="one_to_one"
)

well_audit["retained_candidate_series_count"] = (
    well_audit["retained_candidate_series_count"].fillna(0).astype(int)
)
well_audit["original_candidate_series_count"] = (
    well_audit["original_candidate_series_count"].fillna(0).astype(int)
)

# -------------------- 10. SUMMARY COUNTS --------------------

def count_unique_wells(df):
    return int(df["Well_key"].nunique())

summary = {
    "step": "15A_Spatially_Restricted_Trend_Audit",
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "analysis_domain": "Union of selected 14 block polygons",
    "input_inclusion_file": str(INCLUSION_FILE),
    "input_step12_file": str(STEP12_FILE),
    "input_tfpw_file": str(TFPW_FILE),
    "input_observation_file": str(OBSERVATION_FILE),
    "original_spatially_included_wells": int(len(allowed_keys)),
    "original_step12_candidate_wells": int(len(candidate_wells)),
    "step12_candidate_wells_retained": int(len(retained_candidate_wells)),
    "step12_candidate_wells_excluded": int(len(excluded_candidate_wells)),
    "step12_original_well_season_series": int(len(step12)),
    "step12_retained_well_season_series": int(len(restricted_step12)),
    "step12_excluded_well_season_series": int(len(excluded_candidates)),
    "step10b_tfpw_input_rows": int(len(tfpw)),
    "step10b_tfpw_retained_rows": int(len(restricted_tfpw)),
    "step10b_tfpw_excluded_rows": int(len(excluded_tfpw)),
    "observation_input_rows": int(len(observations)),
    "observation_retained_rows": int(len(restricted_observations)),
    "observation_excluded_rows": int(len(excluded_observations)),
    "retained_observation_unique_wells": count_unique_wells(
        restricted_observations
    ),
    "original_source_data_modified": False,
    "trend_statistics_recalculated": False,
    "interpretation": (
        "Existing trend statistics were spatially filtered only. "
        "No trend tests or p-values were recalculated in Step 15A."
    )
}

# -------------------- 11. SAVE OUTPUTS --------------------

restricted_step12.to_csv(
    OUT / "01_spatially_restricted_candidate_trends.csv", index=False
)

restricted_tfpw.to_csv(
    OUT / "02_spatially_restricted_tfpw_results.csv", index=False
)

restricted_observations.to_csv(
    OUT / "03_spatially_restricted_groundwater_observations.csv",
    index=False
)

well_audit.to_csv(
    OUT / "04_candidate_well_retention_audit.csv", index=False
)

excluded_candidates.to_csv(
    OUT / "05_candidate_series_outside_domain.csv", index=False
)

excluded_tfpw.to_csv(
    OUT / "06_tfpw_rows_outside_domain.csv", index=False
)

excluded_observations.to_csv(
    OUT / "07_observations_outside_domain.csv", index=False
)

# Compact, transparent summary table.
summary_table = pd.DataFrame([
    {"metric": key, "value": value}
    for key, value in summary.items()
])

summary_table.to_csv(
    OUT / "08_step15a_summary.csv", index=False
)

with open(OUT / "09_step15a_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

# -------------------- 12. PRINT REPORT --------------------

print("\n" + "=" * 65)
print("STEP 15A — SPATIALLY RESTRICTED TREND AUDIT")
print("=" * 65)

print("\nCandidate wells:")
print("  Original:", len(candidate_wells))
print("  Retained:", len(retained_candidate_wells))
print("  Excluded:", len(excluded_candidate_wells))

print("\nWell-season series:")
print("  Original:", len(step12))
print("  Retained:", len(restricted_step12))
print("  Excluded:", len(excluded_candidates))

print("\nTFPW results:")
print("  Original rows:", len(tfpw))
print("  Retained rows:", len(restricted_tfpw))
print("  Excluded rows:", len(excluded_tfpw))

print("\nGroundwater observations:")
print("  Original rows:", len(observations))
print("  Retained rows:", len(restricted_observations))
print("  Excluded rows:", len(excluded_observations))

print("\nOutputs saved to:")
print(OUT)

print("\nSUMMARY JSON")
print(json.dumps(summary, indent=2, default=str))

print("\nStep 15A completed. No original input files were modified.")


**[orig #130]** archived — failed (ValueError)  
<sub>⚠ last run ended with ValueError</sub>

In [ ]:
require_columns(step12, ["Well_key", "Season"], "Step 12 trend audit")


**[orig #131]** archived — in-memory schema fix for failed cell 129  
<sub>✅ ran OK</sub>

In [ ]:
# STEP 15A SCHEMA FIX

print("Columns before fix:")
print(step12.columns.tolist())

if "Season" not in step12.columns and "Season_clean" in step12.columns:
    step12 = step12.rename(columns={"Season_clean": "Season"})

# Validate the corrected in-memory dataframe
require_columns(
    step12,
    ["Well_key", "Season"],
    "Step 12 trend audit"
)

print("\nSUCCESS: Step 12 columns are now compatible.")
print("Season values:", step12["Season"].dropna().unique())
print("Number of trend-audit rows:", len(step12))

**[orig #159]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

import os

ROOT = "/content/drive/MyDrive"

print("Searching Google Drive for the archive...")

matches = []

for current_dir, dirs, files in os.walk(ROOT):
    # Skip common folders that are unlikely to contain this archive
    dirs[:] = [
        d for d in dirs
        if d not in {
            ".ipynb_checkpoints",
            "node_modules",
            ".cache"
        }
    ]

    for filename in files:
        if filename.lower() in {
            "copy of india.tar.xz",
            "india.tar.xz"
        }:
            matches.append(os.path.join(current_dir, filename))

print("\nMatching files:")
for path in matches:
    print(path)
    print("Size (GiB):", round(os.path.getsize(path) / 1024**3, 2))

if not matches:
    print(
        "\nArchive not found under MyDrive. "
        "Check whether it is in a Shared Drive or another Google account."
    )


**[orig #160]** archived — interrupted tar listing; replaced by subprocess extraction  
<sub>⏸ interrupted (long run or waiting for upload)</sub>

In [ ]:

import os
import tarfile
import pandas as pd

ARCHIVE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/Copy of India.tar.xz"
)

records = []

with tarfile.open(ARCHIVE, mode="r:xz") as tar:
    for m in tar:
        records.append({
            "path": m.name,
            "is_directory": m.isdir(),
            "is_file": m.isfile(),
            "size_bytes": m.size,
            "size_MB": round(m.size / 1e6, 3),
            "size_GB": round(m.size / 1e9, 4),
            "type": (
                "directory" if m.isdir()
                else "file" if m.isfile()
                else "other"
            )
        })

members_df = pd.DataFrame(records)

# Show only actual files, not directories
files_df = members_df[members_df["is_file"]].copy()

print("Total archive members:", len(members_df))
print("Actual files:", len(files_df))
print("Directories:", int(members_df["is_directory"].sum()))

print("\nALL FILES AND SIZES")
display(
    files_df[["path", "size_MB", "size_GB", "type"]]
    .sort_values("size_bytes", ascending=False)
)

# Save the inventory for later analysis
OUT_DIR = os.path.dirname(ARCHIVE)
inventory_path = os.path.join(
    OUT_DIR, "India_archive_member_inventory.csv"
)
members_df.to_csv(inventory_path, index=False)
print("\nInventory saved:", inventory_path)

# Focus on ACCESS-CM2 SSP245
print("\nACCESS-CM2 / SSP245 MEMBERS")
target = files_df[
    files_df["path"].str.contains(
        r"India/ACCESS-CM2/ssp245/", regex=True
    )
]
display(target[["path", "size_MB", "size_GB"]])


**[orig #162]** archived — Drive mount only  
<sub>✅ ran OK</sub>

In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


**[orig #165]** archived — failed (ValueError)  
<sub>⚠ last run ended with ValueError</sub>

In [ ]:

import os
import pandas as pd
import geopandas as gpd

# --------------------------------------------------
# Paths
# --------------------------------------------------
BASE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

COORD_FILE = os.path.join(
    BASE, "ssp245_grid_coordinates.csv"
)

BLOCK_FILE = (
    "/content/drive/MyDrive/gw_framework/"
    "Shapefile/LGD_Blocks.geojson"
)

OUT_DIR = os.path.join(BASE, "Alwar_spatial_selection")
os.makedirs(OUT_DIR, exist_ok=True)

# --------------------------------------------------
# Read coordinates and block polygons
# --------------------------------------------------
coords = pd.read_csv(COORD_FILE)
blocks = gpd.read_file(BLOCK_FILE)

print("Coordinate records:", len(coords))
print("Polygon features:", len(blocks))
print("Polygon CRS:", blocks.crs)
print("Polygon columns:", blocks.columns.tolist())

# --------------------------------------------------
# Identify the 14 Alwar blocks
# --------------------------------------------------
# Your previously established LGD block codes are 574–587.
if "lgd_code" in blocks.columns:
    block_subset = blocks[
        pd.to_numeric(blocks["lgd_code"], errors="coerce")
        .between(574, 587)
    ].copy()
elif "LGD_CODE" in blocks.columns:
    block_subset = blocks[
        pd.to_numeric(blocks["LGD_CODE"], errors="coerce")
        .between(574, 587)
    ].copy()
else:
    raise ValueError(
        "Could not identify the LGD code field. "
        "Inspect the polygon columns printed above and set the field name."
    )

if block_subset.empty:
    raise ValueError(
        "No polygons selected. Check the LGD code field and values."
    )

block_subset = block_subset.to_crs("EPSG:4326")
study_area = block_subset.geometry.union_all()

print("Selected polygons:", len(block_subset))
print("Study-area bounds:", study_area.bounds)

# --------------------------------------------------
# Deduplicate the grid points for spatial selection
# --------------------------------------------------
grid = coords[
    ["model", "variable", "column_index_zero_based",
     "longitude", "latitude", "source_file"]
].drop_duplicates()

# A point within the union identifies a grid point inside the blocks.
points = gpd.GeoDataFrame(
    grid.copy(),
    geometry=gpd.points_from_xy(
        grid["longitude"], grid["latitude"]
    ),
    crs="EPSG:4326"
)

selected = points[points.geometry.intersects(study_area)].copy()

selected["longitude"] = selected.geometry.x
selected["latitude"] = selected.geometry.y
selected = pd.DataFrame(selected.drop(columns="geometry"))

# Save selected grid cells
selected_path = os.path.join(
    OUT_DIR, "Alwar_selected_grid_cells_inside_blocks.csv"
)
selected.to_csv(selected_path, index=False)

print("\nSelected coordinate records:", len(selected))
print("Saved:", selected_path)

print("\nSelected counts by model and variable:")
display(
    selected.groupby(["model", "variable"])
    .size().rename("selected_grid_cells").reset_index()
)

print("\nExample selected cells:")
display(selected.head(20))


**[orig #169]** archived — folder check probe  
<sub>✅ ran OK</sub>

In [ ]:

import os
import glob

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

base = "/content/drive/MyDrive/gw_framework/climate_forcing/Vimal_Mishra_CMIP6"

paths_to_check = [
    base,
    os.path.join(base, "SSP245_extracted"),
    os.path.join(base, "SSP245_extracted", "Alwar_grid_analysis"),
    os.path.join(base, "SSP245_extracted", "Alwar_grid_analysis",
                 "Alwar_daily_extracted"),
]

print("FOLDER CHECK")
for path in paths_to_check:
    print(f"\n{path}")
    print("Exists:", os.path.exists(path))
    if os.path.isdir(path):
        print("Items:", os.listdir(path)[:15])

print("\nSEARCHING FOR EXTRACTED ALWAR FILES...")
search_root = "/content/drive/MyDrive/gw_framework"

matches = glob.glob(
    os.path.join(search_root, "**", "*_ssp245_*_Alwar.csv"),
    recursive=True
)

print("Matching files found:", len(matches))
for path in matches[:50]:
    print(path)


**[orig #175]** archived — failed (FileNotFoundError)  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:

import os
import pandas as pd
import numpy as np

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

GRID_CSV = os.path.join(
    BASE,
    "Alwar_spatial_selection",
    "Alwar_selected_grid_cells_inside_blocks.csv"
)

grid = pd.read_csv(GRID_CSV)

cells = (
    grid[["longitude", "latitude"]]
    .drop_duplicates()
    .sort_values(["latitude", "longitude"])
    .reset_index(drop=True)
)

cells.insert(
    0, "grid_id",
    [f"ALW_{i:02d}" for i in range(1, len(cells) + 1)]
)

print("Unique cells:", len(cells))
print("\nSelected grid centres:")
print(cells.to_string(index=False))

print("\nUnique longitude spacing (degrees):")
print(
    np.sort(cells["longitude"].unique())
)
print("\nUnique latitude spacing (degrees):")
print(
    np.sort(cells["latitude"].unique())
)

cells.to_csv(
    os.path.join(BASE, "Spatial_validation",
                 "Alwar_unique_grid_centres_review.csv"),
    index=False
)


**[orig #176]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE = "/content/drive/MyDrive/gw_framework/climate_forcing/Vimal_Mishra_CMIP6"

# Search likely locations first
search_roots = [
    BASE,
    "/content/drive/MyDrive/gw_framework",
]

target = "Alwar_selected_grid_cells_inside_blocks.csv"
matches = []

for root in search_roots:
    if not os.path.exists(root):
        print("Folder not found:", root)
        continue

    for current_dir, dirs, files in os.walk(root):
        if target in files:
            matches.append(os.path.join(current_dir, target))

print("\nMatching files found:", len(matches))
for path in matches:
    print(path)

**[orig #177]** archived — placeholder path, failed  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:
import os
import pandas as pd

GRID_CSV = r"PASTE_THE_EXACT_PATH_PRINTED_ABOVE"

if not os.path.isfile(GRID_CSV):
    raise FileNotFoundError(
        f"File not found: {GRID_CSV}\n"
        "Check the printed path and spelling."
    )

grid = pd.read_csv(GRID_CSV)

print("File loaded successfully")
print("Rows:", len(grid))
print("Columns:", grid.columns.tolist())
print(grid.head())

**[orig #178]** archived — Drive check probe  
<sub>✅ ran OK</sub>

In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

root = "/content/drive/MyDrive"

print("Google Drive mounted:", os.path.isdir(root))
print("\nMyDrive folders:")

if os.path.isdir(root):
    for name in os.listdir(root):
        print(name)

project = os.path.join(root, "gw_framework")
print("\nProject folder exists:", os.path.isdir(project))

if os.path.isdir(project):
    for current, dirs, files in os.walk(project):
        depth = current.replace(project, "").count(os.sep)
        if depth <= 5:
            for filename in files:
                if "Alwar_selected_grid_cells_inside_blocks" in filename:
                    print("FOUND:", os.path.join(current, filename))


**[orig #182]** archived — failed (ValueError)  
<sub>⚠ last run ended with ValueError</sub>

In [ ]:

import os
import re
import numpy as np
import pandas as pd

BASE = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis"
)

MAPPING_CSV = os.path.join(
    BASE, "Spatial_validation",
    "Alwar_22cell_source_column_mapping.csv"
)

OUT_DIR = os.path.join(BASE, "Alwar_daily_22cell_extracted")
os.makedirs(OUT_DIR, exist_ok=True)

mapping = pd.read_csv(MAPPING_CSV)

required = {
    "model", "variable", "source_file", "grid_id",
    "longitude", "latitude", "column_index_zero_based"
}
missing = required - set(mapping.columns)
if missing:
    raise ValueError(f"Mapping is missing columns: {sorted(missing)}")

# Ensure every source file maps each cell to one unique source column
mapping["column_index_zero_based"] = pd.to_numeric(
    mapping["column_index_zero_based"], errors="raise"
).astype(int)

if mapping.duplicated(["source_file", "grid_id"]).any():
    raise ValueError("Duplicate source-file/grid-cell mappings detected.")

if mapping.duplicated(["source_file", "column_index_zero_based"]).any():
    raise ValueError("Two grid cells map to the same source column.")

files = mapping["source_file"].drop_duplicates().tolist()
print("Files to process:", len(files))
print("Output folder:", OUT_DIR)

file_log = []

for n, source in enumerate(files, start=1):
    if not os.path.isfile(source):
        raise FileNotFoundError(f"Source file missing: {source}")

    m = mapping[mapping["source_file"] == source].copy()
    m = m.sort_values("column_index_zero_based")

    # Validate mapped column positions against the first two metadata rows.
    # Source layout expected: row 1 = longitude, row 2 = latitude.
    metadata = pd.read_csv(source, header=None, nrows=2)

    indices = m["column_index_zero_based"].to_numpy()
    if indices.min() < 0 or indices.max() >= metadata.shape[1]:
        raise ValueError(
            f"Mapped column index outside file width: {source}; "
            f"metadata columns={metadata.shape[1]}, "
            f"mapped range={indices.min()}–{indices.max()}"
        )

    observed_lon = pd.to_numeric(
        metadata.iloc[0, indices], errors="coerce"
    ).to_numpy()
    observed_lat = pd.to_numeric(
        metadata.iloc[1, indices], errors="coerce"
    ).to_numpy()

    if not np.allclose(
        observed_lon, m["longitude"].to_numpy(),
        atol=1e-6, equal_nan=False
    ):
        raise ValueError(
            f"Longitude/index validation failed for {source}. "
            "Stop before extracting."
        )

    if not np.allclose(
        observed_lat, m["latitude"].to_numpy(),
        atol=1e-6, equal_nan=False
    ):
        raise ValueError(
            f"Latitude/index validation failed for {source}. "
            "Stop before extracting."
        )

    # Read date fields and only the mapped cell columns in chunks.
    usecols = sorted(set([0, 1, 2] + indices.tolist()))
    rename = {
        int(row.column_index_zero_based): row.grid_id
        for row in m.itertuples(index=False)
    }

    # pandas usecols returns selected columns; normalize them by position.
    pieces = []
    for chunk in pd.read_csv(
        source,
        header=None,
        skiprows=2,
        usecols=usecols,
        chunksize=2000
    ):
        # Reorder explicitly to the original source-column indices.
        chunk = chunk.reindex(columns=usecols)

        out = chunk[[0, 1, 2]].copy()
        out.columns = ["Year", "Month", "Day"]

        for col_idx in indices:
            out[rename[int(col_idx)]] = chunk[int(col_idx)].to_numpy()

        pieces.append(out)

    result = pd.concat(pieces, ignore_index=True)

    # Convert date fields to numeric; do not silently discard bad rows.
    for c in ["Year", "Month", "Day"]:
        result[c] = pd.to_numeric(result[c], errors="coerce")

    if result[["Year", "Month", "Day"]].isna().any().any():
        raise ValueError(f"Missing/invalid date fields in {source}")

    result[["Year", "Month", "Day"]] = (
        result[["Year", "Month", "Day"]].astype(int)
    )

    model = str(m["model"].iloc[0])
    variable = str(m["variable"].iloc[0])
    safe_model = re.sub(r"[^A-Za-z0-9_-]", "_", model)
    safe_var = re.sub(r"[^A-Za-z0-9_-]", "_", variable)

    output = os.path.join(
        OUT_DIR, f"{safe_model}_ssp245_{safe_var}_Alwar22.csv"
    )
    result.to_csv(output, index=False)

    file_log.append({
        "source_file": source,
        "output_file": output,
        "model": model,
        "variable": variable,
        "mapped_cells": m["grid_id"].nunique(),
        "rows": len(result),
        "coordinate_index_check": "PASS"
    })

    print(
        f"[{n}/{len(files)}] Saved {os.path.basename(output)} "
        f"| rows={len(result)} | cells={m['grid_id'].nunique()}"
    )

log = pd.DataFrame(file_log)
log_path = os.path.join(OUT_DIR, "extraction_log.csv")
log.to_csv(log_path, index=False)

print("\nEXTRACTION SUMMARY")
print("Files completed:", len(log))
print("All coordinate checks passed:",
      (log["coordinate_index_check"] == "PASS").all())
print("Row counts:")
print(log["rows"].value_counts().sort_index().to_string())
print("Log saved:", log_path)


**[orig #188]** archived — failed (FileNotFoundError)  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:
import pandas as pd
from pathlib import Path

outdir = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/Alwar_grid_analysis/"
    "Alwar_block_daily_forcing"
)

files = sorted(outdir.glob("*_AlwarBlocks.csv"))
print("Block-level output files:", len(files))

problems = []

for f in files:
    df = pd.read_csv(f, parse_dates=["Date"])

    if len(df) != 31411:
        problems.append((f.name, f"Unexpected row count: {len(df)}"))

    if df["Date"].duplicated().any():
        problems.append((f.name, "Duplicate dates"))

    if df["Date"].isna().any():
        problems.append((f.name, "Invalid dates"))

    if not df["Date"].is_monotonic_increasing:
        problems.append((f.name, "Dates are not increasing"))

print("Validation problems:", len(problems))
for p in problems[:30]:
    print(p)

audit = pd.read_csv(outdir / "processing_audit_log.csv")
print("\nRemaining missing block-day values:",
      audit["remaining_missing_block_daily_values"].sum())
print("Maximum output row count:", audit["expected_daily_rows"].max())

**[orig #191]** archived — exact duplicate of original cell 190  
<sub>✅ ran OK</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import traceback

base = Path(
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

source = base / "India/ACCESS-CM2/ssp245/PrecipData"

weights_file = (
    base / "Alwar_grid_analysis/Area_weighted_spatial_selection/"
    "Alwar_block_grid_intersection_weights.csv"
)

test_dir = base / "Alwar_grid_analysis/TEST_one_file"
test_dir.mkdir(parents=True, exist_ok=True)

try:
    print("1. Reading weights...")
    weights = pd.read_csv(weights_file)
    weights["column_index_zero_based"] = (
        weights["column_index_zero_based"].astype(int)
    )

    blocks = weights[["block_lgd", "block_name"]].drop_duplicates()
    grid_indices = sorted(
        weights["column_index_zero_based"].unique()
    )

    print("Blocks:", len(blocks))
    print("Selected grid cells:", len(grid_indices))

    # Only read the columns needed for this test.
    usecols = sorted(set([0, 1, 2] + grid_indices))

    print("2. Reading source file...")
    df = pd.read_csv(
        source,
        sep=r"\s+",
        header=None,
        usecols=usecols,
        engine="c",
        low_memory=False
    )
    df.columns = usecols
    print("Source rows:", len(df))
    print("Columns read:", len(df.columns))

    print("3. Constructing dates...")
    dates = pd.to_datetime({
        "year": pd.to_numeric(df[0], errors="coerce"),
        "month": pd.to_numeric(df[1], errors="coerce"),
        "day": pd.to_numeric(df[2], errors="coerce")
    }, errors="coerce")

    if dates.isna().any():
        raise ValueError("Invalid dates found in the source file.")

    climate = df[grid_indices].apply(pd.to_numeric, errors="coerce")
    climate.index = dates

    print("Date range:", climate.index.min(), "to", climate.index.max())
    print("Duplicate dates:", climate.index.duplicated().sum())

    print("4. Calculating block averages...")
    result = pd.DataFrame(index=climate.index)

    for _, block in blocks.iterrows():
        block_id = int(block["block_lgd"])
        block_name = block["block_name"]

        wb = weights[weights["block_lgd"] == block_id]
        cols = wb["column_index_zero_based"].tolist()
        w = wb.set_index("column_index_zero_based")[
            "weight_within_intersected_area"
        ].reindex(cols).to_numpy(dtype=float)

        values = climate[cols].to_numpy(dtype=float)
        valid = np.isfinite(values)
        denominator = (valid * w.reshape(1, -1)).sum(axis=1)
        numerator = np.nansum(values * w.reshape(1, -1), axis=1)

        result[f"{block_id}_{block_name}"] = np.divide(
            numerator,
            denominator,
            out=np.full(len(climate), np.nan),
            where=denominator > 0
        )

    print("5. Saving test output...")
    output = result.copy()
    output.insert(0, "Date", output.index.strftime("%Y-%m-%d"))
    output.insert(1, "Year", output.index.year)
    output.insert(2, "Month", output.index.month)
    output.insert(3, "Day", output.index.day)

    test_file = test_dir / "ACCESS-CM2_ssp245_PrecipData_TEST.csv"
    output.to_csv(test_file, index=False)

    print("\nTEST SUCCESSFUL")
    print("Output file:", test_file)
    print("Output shape:", output.shape)
    print("Missing values:", int(result.isna().sum().sum()))
    print("First five rows:")
    print(output.head().to_string(index=False))

except Exception:
    print("\nTEST FAILED — full error:")
    traceback.print_exc()

**[orig #202]** archived — Drive mount only  
<sub>✅ ran OK</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**[orig #206]** archived — file search probe  
<sub>✅ ran OK</sub>

In [ ]:
import os

SEARCH_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6"
)

targets = [
    "BCC-CSM2-MR",
    "EC-Earth3"
]

matches = []

for root, dirs, files in os.walk(SEARCH_DIR):
    for filename in files:
        if any(model in filename for model in targets) and (
            "TMaxData" in filename or "TMinData" in filename
        ):
            matches.append(os.path.join(root, filename))

print("Matching source files found:", len(matches))

for path in matches:
    print(path)

**[orig #211]** archived — failed (KeyError)  
<sub>⚠ last run ended with KeyError</sub>

In [ ]:
# ================================================================
# TRACE EXTREME Tmax < Tmin ANOMALIES TO SOURCE GRID CELLS
# Complete, self-contained Google Colab cell
# ================================================================

import os
import numpy as np
import pandas as pd

# ------------------------------------------------
# 1. Define directories
# ------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

ANALYSIS_DIR = os.path.join(
    BASE_DIR, "Alwar_grid_analysis"
)

WEIGHT_FILE = os.path.join(
    ANALYSIS_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

EXTRACTED_DIR = os.path.join(
    ANALYSIS_DIR, "Alwar_daily_extracted"
)

CORRECTED_DIR = os.path.join(
    ANALYSIS_DIR, "Alwar_daily_leapday_corrected"
)

BLOCK_DIR = os.path.join(
    ANALYSIS_DIR, "Alwar_block_daily_forcing"
)

OUTPUT_DIR = os.path.join(
    ANALYSIS_DIR, "Climate_Indicators"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. Check required files and directories
# ------------------------------------------------

for path in [
    WEIGHT_FILE,
    EXTRACTED_DIR,
    CORRECTED_DIR,
    BLOCK_DIR
]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing required path: {path}")

# ------------------------------------------------
# 3. Read spatial weights
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

required_weight_columns = [
    "block_lgd",
    "block_name",
    "column_index_zero_based",
    "weight_within_intersected_area"
]

missing = [
    c for c in required_weight_columns
    if c not in weights.columns
]

if missing:
    raise ValueError(f"Missing weight columns: {missing}")

weights["block_lgd"] = weights["block_lgd"].astype(int)
weights["column_index_zero_based"] = (
    weights["column_index_zero_based"].astype(int)
)

weights["block_name"] = weights["block_name"].astype(str).str.strip()

# Verify weights sum to approximately one for every block
weight_sums = weights.groupby("block_lgd")[
    "weight_within_intersected_area"
].sum()

print("Spatial weight-sum check:")
print(weight_sums)

if not np.allclose(weight_sums.values, 1.0, atol=1e-5):
    raise ValueError(
        "Some block spatial weights do not sum to 1. "
        "Check the spatial weights CSV."
    )

# ------------------------------------------------
# 4. Define the six extreme anomaly cases
# ------------------------------------------------

cases = [
    {
        "model": "BCC-CSM2-MR",
        "variable": "TMaxData",
        "date": "2027-12-28",
        "block_lgd": 577
    },
    {
        "model": "BCC-CSM2-MR",
        "variable": "TMaxData",
        "date": "2027-12-28",
        "block_lgd": 578
    },
    {
        "model": "BCC-CSM2-MR",
        "variable": "TMaxData",
        "date": "2027-12-28",
        "block_lgd": 580
    },
    {
        "model": "BCC-CSM2-MR",
        "variable": "TMaxData",
        "date": "2027-12-28",
        "block_lgd": 586
    },
    {
        "model": "EC-Earth3",
        "variable": "TMaxData",
        "date": "2016-01-07",
        "block_lgd": 578
    },
    {
        "model": "EC-Earth3",
        "variable": "TMaxData",
        "date": "2016-01-07",
        "block_lgd": 586
    }
]

# ------------------------------------------------
# 5. Function to read intermediate grid files
# ------------------------------------------------

def read_grid_file(filepath):

    if not os.path.exists(filepath):
        raise FileNotFoundError(filepath)

    df = pd.read_csv(filepath)

    required = ["Year", "Month", "Day"]

    for col in required:
        if col not in df.columns:
            raise ValueError(
                f"{col} missing from {filepath}. "
                f"Columns: {df.columns.tolist()[:10]}"
            )

    df["Date"] = pd.to_datetime(
        dict(
            year=df["Year"],
            month=df["Month"],
            day=df["Day"]
        ),
        errors="raise"
    )

    if df["Date"].duplicated().any():
        raise ValueError(f"Duplicate dates in {filepath}")

    return df


# ------------------------------------------------
# 6. Trace individual grid-cell values
# ------------------------------------------------

grid_records = []
block_records = []

for case in cases:

    model = case["model"]
    variable = case["variable"]
    date = pd.Timestamp(case["date"])
    lgd = case["block_lgd"]

    block_weight = weights[
        weights["block_lgd"] == lgd
    ].copy()

    if block_weight.empty:
        raise ValueError(f"No spatial weights found for LGD {lgd}")

    block_name = block_weight["block_name"].iloc[0]

    # Final block column includes LGD prefix
    final_column = f"{lgd}_{block_name}"

    # File names
    filename = f"{model}_ssp245_{variable}_Alwar.csv"

    extracted_path = os.path.join(
        EXTRACTED_DIR, filename
    )

    corrected_path = os.path.join(
        CORRECTED_DIR, filename
    )

    final_path = os.path.join(
        BLOCK_DIR,
        f"{model}_ssp245_{variable}_AlwarBlocks.csv"
    )

    # Read all three processing stages
    extracted = read_grid_file(extracted_path)
    corrected = read_grid_file(corrected_path)
    final = pd.read_csv(final_path, parse_dates=["Date"])

    # Select the target date
    extracted_row = extracted.loc[extracted["Date"] == date]
    corrected_row = corrected.loc[corrected["Date"] == date]
    final_row = final.loc[final["Date"] == date]

    if len(extracted_row) != 1:
        raise ValueError(
            f"Expected one extracted row for {model}, {date}; "
            f"found {len(extracted_row)}"
        )

    if len(corrected_row) != 1:
        raise ValueError(
            f"Expected one corrected row for {model}, {date}; "
            f"found {len(corrected_row)}"
        )

    if len(final_row) != 1:
        raise ValueError(
            f"Expected one final row for {model}, {date}; "
            f"found {len(final_row)}"
        )

    if final_column not in final.columns:
        raise KeyError(
            f"Expected final column '{final_column}' not found.\n"
            f"Available block columns:\n"
            f"{final.columns.tolist()}"
        )

    # Read each contributing grid cell
    extracted_values = []
    corrected_values = []
    cell_weights = []

    for _, wrow in block_weight.iterrows():

        grid_index = int(wrow["column_index_zero_based"])
        grid_column = f"grid_col_{grid_index}"

        if grid_column not in extracted.columns:
            raise KeyError(
                f"{grid_column} not found in {extracted_path}"
            )

        if grid_column not in corrected.columns:
            raise KeyError(
                f"{grid_column} not found in {corrected_path}"
            )

        source_value = float(
            extracted_row.iloc[0][grid_column]
        )

        corrected_value = float(
            corrected_row.iloc[0][grid_column]
        )

        weight = float(
            wrow["weight_within_intersected_area"]
        )

        extracted_values.append(source_value)
        corrected_values.append(corrected_value)
        cell_weights.append(weight)

        grid_records.append({
            "Model": model,
            "Date": date,
            "Variable": variable,
            "Block_LGD": lgd,
            "Block_Name": block_name,
            "Grid_Index": grid_index,
            "Grid_Column": grid_column,
            "Weight": weight,
            "Original_Extracted_Value": source_value,
            "Leapday_Corrected_Value": corrected_value
        })

    # Weighted means, without silently ignoring missing cells
    extracted_values = np.array(extracted_values, dtype=float)
    corrected_values = np.array(corrected_values, dtype=float)
    cell_weights = np.array(cell_weights, dtype=float)

    if not np.isfinite(extracted_values).all():
        extracted_mean = np.nan
    else:
        extracted_mean = np.sum(
            extracted_values * cell_weights
        )

    if not np.isfinite(corrected_values).all():
        corrected_mean = np.nan
    else:
        corrected_mean = np.sum(
            corrected_values * cell_weights
        )

    final_value = float(
        final_row.iloc[0][final_column]
    )

    block_records.append({
        "Model": model,
        "Date": date,
        "Variable": variable,
        "Block_LGD": lgd,
        "Block_Name": block_name,
        "Final_Block_Column": final_column,
        "Extracted_Weighted_Mean": extracted_mean,
        "Corrected_Weighted_Mean": corrected_mean,
        "Final_Block_Value": final_value,
        "Difference_Final_vs_Corrected": (
            final_value - corrected_mean
            if np.isfinite(corrected_mean)
            else np.nan
        )
    })


# ------------------------------------------------
# 7. Create reports
# ------------------------------------------------

grid_report = pd.DataFrame(grid_records)
block_report = pd.DataFrame(block_records)

grid_output = os.path.join(
    OUTPUT_DIR, "Anomaly_Grid_Cell_Values.csv"
)

block_output = os.path.join(
    OUTPUT_DIR, "Anomaly_Weighted_Block_Comparison.csv"
)

grid_report.to_csv(grid_output, index=False)
block_report.to_csv(block_output, index=False)

# ------------------------------------------------
# 8. Display results
# ------------------------------------------------

print("\n" + "=" * 80)
print("GRID-CELL VALUES")
print("=" * 80)

display(
    grid_report.sort_values(
        ["Model", "Date", "Block_LGD", "Grid_Index"]
    ).round(5)
)

print("\n" + "=" * 80)
print("WEIGHTED BLOCK COMPARISON")
print("=" * 80)

display(block_report.round(5))

print("\nReports saved successfully:")
print(grid_output)
print(block_output)


**[orig #213]** archived — failed (EmptyDataError)  
<sub>⚠ last run ended with EmptyDataError</sub>

In [ ]:
# ================================================================
# AUDIT ALL 39 EXTRACTED FILES AGAINST REQUIRED SPATIAL GRID CELLS
# ================================================================

import os
import re
import glob
import pandas as pd
import numpy as np

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

ANALYSIS_DIR = os.path.join(BASE_DIR, "Alwar_grid_analysis")

WEIGHT_FILE = os.path.join(
    ANALYSIS_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

EXTRACTED_DIR = os.path.join(
    ANALYSIS_DIR,
    "Alwar_daily_extracted"
)

OUTPUT_DIR = os.path.join(
    ANALYSIS_DIR,
    "Climate_Indicators"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Read required grid indices
weights = pd.read_csv(WEIGHT_FILE)

required_indices = sorted(
    weights["column_index_zero_based"].astype(int).unique()
)

required_columns = {
    f"grid_col_{i}" for i in required_indices
}

# Find all extracted files
files = sorted(
    glob.glob(os.path.join(EXTRACTED_DIR, "*.csv"))
)

print("Number of extracted files:", len(files))
print("Required unique grid cells:", len(required_indices))

audit_records = []

for filepath in files:

    filename = os.path.basename(filepath)

    # Read only the header
    header = pd.read_csv(filepath, nrows=0)

    actual_columns = set(header.columns)

    actual_grid_columns = {
        col for col in actual_columns
        if re.fullmatch(r"grid_col_\d+", str(col))
    }

    missing = sorted(
        required_columns - actual_grid_columns
    )

    present = sorted(
        required_columns.intersection(actual_grid_columns)
    )

    audit_records.append({
        "File": filename,
        "Required_Cells": len(required_columns),
        "Present_Cells": len(present),
        "Missing_Cells": len(missing),
        "Missing_Grid_Columns": ", ".join(missing),
        "Status": (
            "PASS" if len(missing) == 0 else "INCOMPLETE"
        )
    })

audit = pd.DataFrame(audit_records)

print("\nFILE-BY-FILE GRID CELL AUDIT")
display(audit)

print("\nSUMMARY")
print(audit["Status"].value_counts())

output_file = os.path.join(
    OUTPUT_DIR,
    "All_39_Files_Grid_Cell_Audit.csv"
)

audit.to_csv(output_file, index=False)

print("\nAudit saved:")
print(output_file)

**[orig #216]** archived — failed; superseded by original cell 217  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:
# ================================================================
# REBUILD ALWAR GRID-CELL DATASETS FROM ORIGINAL CMIP6 SOURCE FILES
# 13 MODELS x 3 VARIABLES = 39 FILES
# 22 REQUIRED GRID CELLS
#
# Original source files are not modified.
# Previous extracted files are not overwritten.
# ================================================================

import os
import glob
import numpy as np
import pandas as pd

# ------------------------------------------------
# 1. PATHS
# ------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted"
)

SOURCE_DIR = os.path.join(BASE_DIR, "India")

ANALYSIS_DIR = os.path.join(
    BASE_DIR, "Alwar_grid_analysis"
)

WEIGHT_FILE = os.path.join(
    ANALYSIS_DIR,
    "Area_weighted_spatial_selection",
    "Alwar_block_grid_intersection_weights.csv"
)

OUTPUT_DIR = os.path.join(
    ANALYSIS_DIR,
    "Alwar_daily_extracted_REBUILT"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------
# 2. MODEL AND VARIABLE DEFINITIONS
# ------------------------------------------------

models = [
    "ACCESS-CM2",
    "ACCESS-ESM1-5",
    "BCC-CSM2-MR",
    "CanESM5",
    "EC-Earth3",
    "EC-Earth3-Veg",
    "INM-CM4-8",
    "INM-CM5-0",
    "MPI-ESM1-2-HR",
    "MPI-ESM1-2-LR",
    "MRI-ESM2-0",
    "NorESM2-LM",
    "NorESM2-MM"
]

variables = [
    "PrecipData",
    "TMaxData",
    "TMinData"
]

# ------------------------------------------------
# 3. READ REQUIRED GRID INDICES
# ------------------------------------------------

weights = pd.read_csv(WEIGHT_FILE)

required_indices = sorted(
    weights["column_index_zero_based"]
    .astype(int)
    .unique()
)

print("Required grid cells:", len(required_indices))
print("Grid indices:", required_indices)

if len(required_indices) != 22:
    raise ValueError(
        f"Expected 22 grid cells, found {len(required_indices)}"
    )

# ------------------------------------------------
# 4. LOCATE ORIGINAL SOURCE FILES
# ------------------------------------------------

all_source_files = glob.glob(
    os.path.join(SOURCE_DIR, "**", "*"),
    recursive=True
)

all_source_files = [
    f for f in all_source_files
    if os.path.isfile(f)
]

print("\nOriginal files found:", len(all_source_files))

# Locate each source file using its full path and model name.
source_map = {}

for model in models:

    for variable in variables:

        matches = [
            f for f in all_source_files
            if model in f
            and os.path.basename(f) == variable
        ]

        if len(matches) != 1:
            raise FileNotFoundError(
                f"Expected exactly one source file for "
                f"{model} / {variable}; found {len(matches)}:\n"
                f"{matches}"
            )

        source_map[(model, variable)] = matches[0]

if len(source_map) != 39:
    raise ValueError(
        f"Expected 39 source files; found {len(source_map)}"
    )

# ------------------------------------------------
# 5. EXTRACT GRID CELLS
# ------------------------------------------------

inventory = []

for (model, variable), source_path in source_map.items():

    print(f"\nProcessing: {model} | {variable}")

    # Read source data.
    # The original file has two metadata rows:
    # row 1 = longitude
    # row 2 = latitude
    # row 3 onward = Year, Month, Day, grid values

    raw = pd.read_csv(
        source_path,
        sep=r"\s+",
        header=None,
        skiprows=2,
        dtype=np.float64
    )

    expected_columns = 3 + 4641

    if raw.shape[1] != expected_columns:
        raise ValueError(
            f"Unexpected number of columns in {source_path}: "
            f"{raw.shape[1]}; expected {expected_columns}"
        )

    # Date columns
    date_data = raw.iloc[:, :3].copy()

    date_data.columns = [
        "Year", "Month", "Day"
    ]

    date_data["Year"] = date_data["Year"].astype(int)
    date_data["Month"] = date_data["Month"].astype(int)
    date_data["Day"] = date_data["Day"].astype(int)

    dates = pd.to_datetime(
        dict(
            year=date_data["Year"],
            month=date_data["Month"],
            day=date_data["Day"]
        ),
        errors="raise"
    )

    # Extract the required original grid columns.
    selected_data = raw.iloc[
        :,
        [3 + i for i in required_indices]
    ].copy()

    selected_data.columns = [
        f"grid_col_{i}"
        for i in required_indices
    ]

    # Combine date and grid data
    output = pd.concat(
        [
            date_data.reset_index(drop=True),
            selected_data.reset_index(drop=True)
        ],
        axis=1
    )

    # Validate date sequence
    if dates.duplicated().any():
        raise ValueError(
            f"Duplicate dates in {model} {variable}"
        )

    if not dates.is_monotonic_increasing:
        raise ValueError(
            f"Dates are not chronological in {model} {variable}"
        )

    output.insert(0, "Date", dates)

    # Validate output dimensions
    if output.shape[1] != 4 + len(required_indices):
        raise ValueError(
            f"Unexpected output columns for {model} {variable}"
        )

    # Validate numerical values
    grid_values = output[
        [f"grid_col_{i}" for i in required_indices]
    ].to_numpy(dtype=float)

    if not np.isfinite(grid_values).all():
        raise ValueError(
            f"Non-finite values found in {model} {variable}"
        )

    # Save rebuilt extraction
    output_filename = (
        f"{model}_ssp245_{variable}_Alwar_REBUILT.csv"
    )

    output_path = os.path.join(
        OUTPUT_DIR,
        output_filename
    )

    output.to_csv(
        output_path,
        index=False
    )

    inventory.append({
        "Model": model,
        "Variable": variable,
        "Source_File": source_path,
        "Output_File": output_filename,
        "Rows": len(output),
        "Grid_Cells": len(required_indices),
        "Columns": output.shape[1],
        "First_Date": output["Date"].min(),
        "Last_Date": output["Date"].max(),
        "Status": "PASS"
    })

# ------------------------------------------------
# 6. SAVE EXTRACTION INVENTORY
# ------------------------------------------------

inventory_df = pd.DataFrame(inventory)

inventory_path = os.path.join(
    OUTPUT_DIR,
    "Rebuilt_Extraction_Inventory.csv"
)

inventory_df.to_csv(
    inventory_path,
    index=False
)

# ------------------------------------------------
# 7. FINAL SUMMARY
# ------------------------------------------------

print("\n" + "=" * 70)
print("REBUILT EXTRACTION SUMMARY")
print("=" * 70)

display(inventory_df)

print("\nFiles successfully rebuilt:", len(inventory_df))

print(
    "Total extracted grid cells per file:",
    len(required_indices)
)

print(
    "Rows per file:",
    inventory_df["Rows"].unique()
)

print(
    "Number of PASS files:",
    (inventory_df["Status"] == "PASS").sum()
)

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nInventory:")
print(inventory_path)

if len(inventory_df) != 39:
    raise RuntimeError("Not all 39 files were rebuilt.")

if not (inventory_df["Status"] == "PASS").all():
    raise RuntimeError("Some files failed extraction validation.")

print("\nSUCCESS: All 39 source files rebuilt.")

**[orig #221]** archived — Drive search probe  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import os
import glob

ROOT = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6"
)

print("Checking root directory:")
print(ROOT)
print("Exists:", os.path.exists(ROOT))

# Search recursively for all files belonging to BCC-CSM2-MR
all_files = glob.glob(
    os.path.join(ROOT, "**", "*"),
    recursive=True
)

bcc_files = [
    f for f in all_files
    if os.path.isfile(f)
    and "BCC-CSM2-MR" in os.path.basename(f)
]

print("\nBCC-CSM2-MR files found:", len(bcc_files))

for f in bcc_files:
    print(f)

print("\n" + "=" * 65)
print("TMAX/TMIN CANDIDATES")
print("=" * 65)

for f in bcc_files:
    name = os.path.basename(f).lower()

    if "tmax" in name or "tmin" in name:
        print(f)

    elif "precip" not in name:
        # Show other possible naming conventions
        print("CHECK:", f)


**[orig #222]** archived — Drive search probe  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import os
import glob

ROOT = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6"
)

print("=" * 70)
print("SEARCHING FOR ORIGINAL CMIP6 SOURCE FILES")
print("=" * 70)

# Search for directories named India
india_dirs = []

for path in glob.glob(
    os.path.join(ROOT, "**", "India"),
    recursive=True
):
    if os.path.isdir(path):
        india_dirs.append(path)

print("\nIndia directories found:")

for path in india_dirs:
    print(path)

# Search for original source files using model and variable terms
models = [
    "BCC-CSM2-MR",
    "CanESM5",
    "INM-CM4-8",
    "INM-CM5-0",
    "EC-Earth3"
]

print("\n" + "=" * 70)
print("POTENTIAL ORIGINAL SOURCE FILES")
print("=" * 70)

for model in models:

    print(f"\nMODEL: {model}")

    matches = glob.glob(
        os.path.join(ROOT, "**", f"*{model}*"),
        recursive=True
    )

    for f in matches:
        if not os.path.isfile(f):
            continue

        # Ignore intermediate Alwar CSV files
        if "Alwar_grid_analysis" in f:
            continue

        print(f)

# Check archive
archives = glob.glob(
    os.path.join(ROOT, "**", "*.tar*"),
    recursive=True
)

print("\n" + "=" * 70)
print("ARCHIVES FOUND")
print("=" * 70)

for f in archives:
    print(f, "| Size (GB):", round(os.path.getsize(f)/1e9, 3))


**[orig #223]** archived — folder inspection probe  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import os

INDIA_DIR = (
    "/content/drive/MyDrive/gw_framework/climate_forcing/"
    "Vimal_Mishra_CMIP6/SSP245_extracted/India"
)

print("=" * 75)
print("INDIA DIRECTORY INSPECTION")
print("=" * 75)

if not os.path.isdir(INDIA_DIR):
    raise FileNotFoundError(INDIA_DIR)

total_files = 0
total_size = 0
directories = []

for root, dirs, files in os.walk(INDIA_DIR):

    relative = os.path.relpath(root, INDIA_DIR)

    if dirs:
        directories.append((relative, dirs.copy()))

    for filename in files:
        filepath = os.path.join(root, filename)
        total_files += 1
        total_size += os.path.getsize(filepath)

print("\nTotal files:", total_files)
print("Total size (GB):", round(total_size / 1e9, 3))

print("\n" + "=" * 75)
print("DIRECTORY STRUCTURE")
print("=" * 75)

for relative, dirs in directories[:40]:
    print(relative, "->", dirs[:15])

print("\n" + "=" * 75)
print("FIRST 60 FILES")
print("=" * 75)

count = 0

for root, dirs, files in os.walk(INDIA_DIR):

    for filename in sorted(files):

        filepath = os.path.join(root, filename)

        print(
            os.path.relpath(filepath, INDIA_DIR),
            "|",
            round(os.path.getsize(filepath) / 1e6, 2),
            "MB"
        )

        count += 1

        if count >= 60:
            break

    if count >= 60:
        break

print("\n" + "=" * 75)
print("INSPECTION COMPLETE")
print("=" * 75)


**[orig #231]** archived — folder check  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "Alwar_grid_analysis/Climate_Indicators"
)

print("Climate Indicators folder exists:", os.path.exists(BASE_DIR))

if os.path.exists(BASE_DIR):
    for root, dirs, files in os.walk(BASE_DIR):
        for file in files:
            if "Temperature_Pair_Uniqueness" in file:
                print(os.path.join(root, file))

**[orig #235]** archived — Drive search probe  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

ROOT = "/content/drive/MyDrive/gw_framework"

print("GW framework folder exists:", os.path.exists(ROOT))

if not os.path.exists(ROOT):
    raise FileNotFoundError(
        "gw_framework folder not found. Check your Google Drive location."
    )

print("\nSearching for existing QC files...\n")

keywords = [
    "All_NonSentinel_Reversal_Events",
    "Temperature_Pair_Uniqueness",
    "Reversal_Temporal_Magnitude",
    "Cross_Model_Synchronisation",
    "Consecutive_Day_Event_Runs",
    "Same_Date_Clustering"
]

found = []

for current_root, dirs, files in os.walk(ROOT):
    # Avoid searching large raw climate-data directories
    dirs[:] = [
        d for d in dirs
        if d not in [
            "SSP245_extracted",
            "Alwar_daily_extracted_REBUILT",
            "Alwar_block_daily_weighted_REBUILT",
            "Alwar_block_daily_forcing_REBUILT"
        ]
    ]

    for filename in files:
        if any(k.lower() in filename.lower() for k in keywords):
            full_path = os.path.join(current_root, filename)
            found.append(full_path)

if found:
    for path in found:
        print(path)
else:
    print("No matching files found.")

**[orig #236]** archived — folder listing  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os

ROOT = "/content/drive/MyDrive/gw_framework"

print("FOLDERS INSIDE gw_framework:\n")

for item in os.listdir(ROOT):
    path = os.path.join(ROOT, item)
    if os.path.isdir(path):
        print("[DIR] ", item)
    else:
        print("[FILE]", item)

print("\nSearching for folders containing 'Climate' or 'Alwar':\n")

for current_root, dirs, files in os.walk(ROOT):
    depth = current_root.replace(ROOT, "").count(os.sep)

    if depth > 5:
        dirs[:] = []
        continue

    for folder in dirs:
        if "climate" in folder.lower() or "alwar" in folder.lower():
            print(os.path.join(current_root, folder))

**[orig #237]** archived — Drive search probe  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import os

ROOT = "/content/drive/MyDrive/gw_framework"

targets = [
    "Alwar_block_daily_forcing_REBUILT",
    "Alwar_block_daily_weighted_REBUILT",
    "Alwar_daily_extracted_REBUILT",
    "All_NonSentinel_Reversal_Events.csv"
]

for target in targets:
    matches = []

    for current_root, dirs, files in os.walk(ROOT):
        if target in dirs:
            matches.append(os.path.join(current_root, target))

        if target in files:
            matches.append(os.path.join(current_root, target))

    print("\n", target)

    if matches:
        for m in matches:
            print("FOUND:", m)
    else:
        print("NOT FOUND")

**[orig #245]** archived — superseded by the corrected-file-structure version (orig 247)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

# ============================================================
# ORIGINAL CMIP6 SOURCE VALIDATION
# Verify all 2,804 non-sentinel Tmax < Tmin events
# Direct comparison: original source files vs event inventory
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

# ------------------------------------------------------------
# 1. PATHS AND SETTINGS
# ------------------------------------------------------------

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

SOURCE_BASE = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/India"
)

EVENT_FILE = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Reversal_Temporal_Magnitude_QC",
    "All_NonSentinel_Reversal_Events.csv"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Original_Source_Validation"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

MODELS = [
    "CanESM5",
    "INM-CM4-8",
    "INM-CM5-0"
]

TOLERANCE = 1e-6

# Original source folders
VARIABLE_FOLDERS = {
    "Tmax": "TMaxData",
    "Tmin": "TMinData"
}

# ------------------------------------------------------------
# 2. LOAD EVENT INVENTORY
# ------------------------------------------------------------

if not os.path.exists(EVENT_FILE):
    raise FileNotFoundError(f"Event inventory not found: {EVENT_FILE}")

events = pd.read_csv(EVENT_FILE)

print("Event inventory columns:")
print(events.columns.tolist())

# Flexible column identification
def find_column(df, candidates, description):
    normalized = {
        str(c).lower().replace("_", "").replace(" ", ""): c
        for c in df.columns
    }

    for candidate in candidates:
        key = candidate.lower().replace("_", "").replace(" ", "")
        if key in normalized:
            return normalized[key]

    raise ValueError(
        f"Could not identify {description} column. "
        f"Available columns: {df.columns.tolist()}"
    )

model_col = find_column(
    events, ["model"], "model"
)

date_col = find_column(
    events, ["date", "datetime"], "date"
)

grid_col = find_column(
    events, ["grid_index", "column_index_zero_based",
             "grid_column_index"], "grid index"
)

tmax_col = find_column(
    events, ["Tmax", "Tmax_C", "tmax_value",
             "Tmax_value"], "Tmax"
)

tmin_col = find_column(
    events, ["Tmin", "Tmin_C", "tmin_value",
             "Tmin_value"], "Tmin"
)

events[date_col] = pd.to_datetime(
    events[date_col], errors="raise"
).dt.normalize()

events[grid_col] = pd.to_numeric(
    events[grid_col], errors="raise"
).astype(int)

events[tmax_col] = pd.to_numeric(
    events[tmax_col], errors="raise"
)

events[tmin_col] = pd.to_numeric(
    events[tmin_col], errors="raise"
)

events[model_col] = events[model_col].astype(str)

events = events[
    events[model_col].isin(MODELS)
].copy()

print("\nEvents to verify:", len(events))
print("\nEvents by model:")
print(events.groupby(model_col).size())

if len(events) != 2804:
    raise ValueError(
        f"Expected 2804 events, but found {len(events)}. "
        "Check the event inventory and model filtering."
    )

# ------------------------------------------------------------
# 3. LOCATE ORIGINAL SOURCE FILES
# ------------------------------------------------------------

def locate_source_file(model, variable_folder):

    search_dir = os.path.join(
        SOURCE_BASE,
        model,
        "ssp245",
        variable_folder
    )

    if not os.path.isdir(search_dir):
        raise FileNotFoundError(
            f"Original source directory not found: {search_dir}"
        )

    candidates = [
        p for p in Path(search_dir).rglob("*")
        if p.is_file()
        and not p.name.startswith(".")
    ]

    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No original source files found in {search_dir}"
        )

    if len(candidates) > 1:
        print("\nMultiple files found:")
        for p in candidates:
            print(p)

        raise RuntimeError(
            f"Multiple files found for {model}/{variable_folder}. "
            "Please identify the correct original file before proceeding."
        )

    return str(candidates[0])


# ------------------------------------------------------------
# 4. PREPARE EVENT LOOKUP
# ------------------------------------------------------------

# For each model and date, retain only the grid indices
# needed for the event-level source comparison.

event_lookup = {}

for model in MODELS:

    model_events = events[
        events[model_col] == model
    ]

    lookup = defaultdict(set)

    for _, row in model_events.iterrows():
        lookup[row[date_col]].add(int(row[grid_col]))

    event_lookup[model] = lookup


# ------------------------------------------------------------
# 5. READ ORIGINAL SOURCE FILE
# ------------------------------------------------------------

def read_original_event_values(file_path, required_dates, required_grids):

    """
    Original source layout:
    Line 1: metadata / longitude information
    Line 2: metadata / latitude information
    Remaining lines: Year Month Day followed by grid values.

    Grid index 0 corresponds to source field position 3.
    """

    required_dates = set(required_dates)
    required_grids = set(required_grids)

    results = {}
    duplicate_dates = set()
    malformed_rows = []

    with open(
        file_path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:

        # Skip two metadata rows
        next(f, None)
        next(f, None)

        for line_number, line in enumerate(f, start=3):

            if not line.strip():
                continue

            parts = line.split()

            if len(parts) < 4:
                continue

            try:
                year = int(float(parts[0]))
                month = int(float(parts[1]))
                day = int(float(parts[2]))

                current_date = pd.Timestamp(
                    year=year,
                    month=month,
                    day=day
                ).normalize()

            except Exception:
                continue

            # Ignore dates that do not contain any required event
            if current_date not in required_dates:
                continue

            if current_date in results:
                duplicate_dates.add(current_date)

            values = {}

            for grid_index in required_grids:

                source_position = 3 + grid_index

                if source_position >= len(parts):
                    malformed_rows.append({
                        "date": current_date,
                        "line_number": line_number,
                        "grid_index": grid_index,
                        "source_position": source_position,
                        "available_fields": len(parts)
                    })
                    continue

                try:
                    values[grid_index] = float(
                        parts[source_position]
                    )
                except Exception:
                    malformed_rows.append({
                        "date": current_date,
                        "line_number": line_number,
                        "grid_index": grid_index,
                        "source_position": source_position,
                        "available_fields": len(parts)
                    })

            results[current_date] = values

    return results, duplicate_dates, malformed_rows


# ------------------------------------------------------------
# 6. VERIFY ALL EVENTS AGAINST ORIGINAL FILES
# ------------------------------------------------------------

all_results = []
source_file_inventory = []
source_row_issues = []

for model in MODELS:

    print("\n" + "=" * 65)
    print("PROCESSING MODEL:", model)
    print("=" * 65)

    model_events = events[
        events[model_col] == model
    ].copy()

    required_dates = set(model_events[date_col])

    required_grids = set(
        model_events[grid_col].astype(int)
    )

    # Locate original Tmax and Tmin source files
    tmax_file = locate_source_file(
        model, "TMaxData"
    )

    tmin_file = locate_source_file(
        model, "TMinData"
    )

    print("Original Tmax file:", tmax_file)
    print("Original Tmin file:", tmin_file)

    source_file_inventory.extend([
        {
            "model": model,
            "variable": "Tmax",
            "source_file": tmax_file
        },
        {
            "model": model,
            "variable": "Tmin",
            "source_file": tmin_file
        }
    ])

    # Read only the required dates and grid columns
    tmax_data, tmax_duplicates, tmax_issues = (
        read_original_event_values(
            tmax_file,
            required_dates,
            required_grids
        )
    )

    tmin_data, tmin_duplicates, tmin_issues = (
        read_original_event_values(
            tmin_file,
            required_dates,
            required_grids
        )
    )

    print("Required event dates:", len(required_dates))
    print("Tmax source dates found:", len(tmax_data))
    print("Tmin source dates found:", len(tmin_data))

    for issue in tmax_issues:
        source_row_issues.append({
            "model": model,
            "variable": "Tmax",
            **issue
        })

    for issue in tmin_issues:
        source_row_issues.append({
            "model": model,
            "variable": "Tmin",
            **issue
        })

    # Compare event by event
    for _, event in model_events.iterrows():

        event_date = event[date_col]
        grid_index = int(event[grid_col])

        expected_tmax = float(event[tmax_col])
        expected_tmin = float(event[tmin_col])

        source_tmax = np.nan
        source_tmin = np.nan

        if (
            event_date in tmax_data
            and grid_index in tmax_data[event_date]
        ):
            source_tmax = tmax_data[event_date][grid_index]

        if (
            event_date in tmin_data
            and grid_index in tmin_data[event_date]
        ):
            source_tmin = tmin_data[event_date][grid_index]

        tmax_found = np.isfinite(source_tmax)
        tmin_found = np.isfinite(source_tmin)

        tmax_match = (
            tmax_found
            and abs(source_tmax - expected_tmax) <= TOLERANCE
        )

        tmin_match = (
            tmin_found
            and abs(source_tmin - expected_tmin) <= TOLERANCE
        )

        complete_match = tmax_match and tmin_match

        source_reversal = (
            tmax_found
            and tmin_found
            and source_tmax < source_tmin
        )

        all_reversal = (
            expected_tmax < expected_tmin
        )

        reversal_match = (
            source_reversal == all_reversal
            if tmax_found and tmin_found
            else False
        )

        all_results.append({
            "model": model,
            "date": event_date,
            "grid_index": grid_index,

            "inventory_Tmax_C": expected_tmax,
            "inventory_Tmin_C": expected_tmin,

            "source_Tmax_C": source_tmax,
            "source_Tmin_C": source_tmin,

            "Tmax_found": tmax_found,
            "Tmin_found": tmin_found,

            "Tmax_match": tmax_match,
            "Tmin_match": tmin_match,
            "complete_value_match": complete_match,

            "inventory_reversal": all_reversal,
            "source_reversal": source_reversal,
            "reversal_condition_match": reversal_match,

            "Tmax_difference_C": (
                source_tmax - expected_tmax
                if tmax_found else np.nan
            ),

            "Tmin_difference_C": (
                source_tmin - expected_tmin
                if tmin_found else np.nan
            )
        })

    print("Model events processed:", len(model_events))
    print(
        "Duplicate Tmax dates:",
        len(tmax_duplicates)
    )
    print(
        "Duplicate Tmin dates:",
        len(tmin_duplicates)
    )


# ------------------------------------------------------------
# 7. CREATE DETAILED AUDIT REPORT
# ------------------------------------------------------------

detail = pd.DataFrame(all_results)

detail_file = os.path.join(
    OUTPUT_DIR,
    "Original_Source_All_2804_Event_Verification.csv"
)

detail.to_csv(detail_file, index=False)

# ------------------------------------------------------------
# 8. MODEL-LEVEL SUMMARY
# ------------------------------------------------------------

summary_rows = []

for model in MODELS:

    subset = detail[
        detail["model"] == model
    ].copy()

    summary_rows.append({

        "model": model,

        "events_checked": len(subset),

        "Tmax_found": int(
            subset["Tmax_found"].sum()
        ),

        "Tmin_found": int(
            subset["Tmin_found"].sum()
        ),

        "Tmax_matches": int(
            subset["Tmax_match"].sum()
        ),

        "Tmin_matches": int(
            subset["Tmin_match"].sum()
        ),

        "complete_value_matches": int(
            subset["complete_value_match"].sum()
        ),

        "reversal_condition_matches": int(
            subset["reversal_condition_match"].sum()
        ),

        "Tmax_mismatches": int(
            (~subset["Tmax_match"]).sum()
        ),

        "Tmin_mismatches": int(
            (~subset["Tmin_match"]).sum()
        ),

        "maximum_Tmax_difference_C": (
            subset["Tmax_difference_C"].abs().max()
        ),

        "maximum_Tmin_difference_C": (
            subset["Tmin_difference_C"].abs().max()
        ),

        "matching_percentage": (
            100 * subset["complete_value_match"].mean()
            if len(subset) else np.nan
        )
    })

summary = pd.DataFrame(summary_rows)

summary_file = os.path.join(
    OUTPUT_DIR,
    "Original_Source_All_2804_Event_Summary.csv"
)

summary.to_csv(summary_file, index=False)

# ------------------------------------------------------------
# 9. SOURCE FILE INVENTORY AND PARSING ISSUES
# ------------------------------------------------------------

pd.DataFrame(source_file_inventory).to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Original_Source_File_Inventory.csv"
    ),
    index=False
)

issues_df = pd.DataFrame(source_row_issues)

issues_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "Original_Source_Row_Parsing_Issues.csv"
    ),
    index=False
)

# ------------------------------------------------------------
# 10. FINAL AUDIT REPORT
# ------------------------------------------------------------

print("\n")
print("=" * 75)
print("ORIGINAL CMIP6 SOURCE VALIDATION COMPLETED")
print("=" * 75)

print("\nMODEL-WISE SUMMARY")
display(summary)

print("\nTOTAL EVENTS:", len(detail))

print(
    "Complete source-value matches:",
    int(detail["complete_value_match"].sum())
)

print(
    "Complete value mismatches:",
    int((~detail["complete_value_match"]).sum())
)

print(
    "Reversal-condition matches:",
    int(detail["reversal_condition_match"].sum())
)

print(
    "Reversal-condition mismatches:",
    int((~detail["reversal_condition_match"]).sum())
)

print(
    "Events with missing Tmax:",
    int((~detail["Tmax_found"]).sum())
)

print(
    "Events with missing Tmin:",
    int((~detail["Tmin_found"]).sum())
)

print(
    "Source parsing issues:",
    len(issues_df)
)

print("\nDETAILED REPORT:")
print(detail_file)

print("\nSUMMARY REPORT:")
print(summary_file)

print("\nOUTPUT DIRECTORY:")
print(OUTPUT_DIR)

# Conservative pass/fail assessment
audit_pass = (
    len(detail) == 2804
    and detail["complete_value_match"].all()
    and detail["reversal_condition_match"].all()
    and detail["Tmax_found"].all()
    and detail["Tmin_found"].all()
    and len(issues_df) == 0
)

print("\nFINAL SOURCE AUDIT STATUS:")

if audit_pass:
    print("PASS: All 2,804 events match original source values.")
else:
    print("REVIEW REQUIRED: One or more source checks did not pass.")


**[orig #246]** archived — folder listing  
<sub>• output not saved in the source notebook</sub>

In [ ]:

import os
from pathlib import Path

ROOT = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6"
)

print("ROOT EXISTS:", os.path.exists(ROOT))
print("\nTop-level directories:")

if os.path.exists(ROOT):
    for item in os.listdir(ROOT):
        path = os.path.join(ROOT, item)
        print("DIR " if os.path.isdir(path) else "FILE", item)

print("\n" + "=" * 70)
print("SEARCHING FOR ORIGINAL SOURCE FILES")
print("=" * 70)

target_models = ["CanESM5", "INM-CM4-8", "INM-CM5-0"]
found = []

for current_root, dirs, files in os.walk(ROOT):
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    path_lower = current_root.lower()

    if any(model.lower() in path_lower for model in target_models):
        relevant_files = [
            f for f in files
            if not f.startswith(".")
        ]

        if relevant_files:
            print("\nDIRECTORY:", current_root)
            for f in relevant_files[:10]:
                print("  FILE:", f)
                found.append(os.path.join(current_root, f))

print("\nTotal files found in matching model directories:", len(found))

print("\nPossible Tmax/Tmin directories:")
for current_root, dirs, files in os.walk(ROOT):
    folder = os.path.basename(current_root).lower()

    if "tmax" in folder or "tmin" in folder:
        print(current_root)


**[orig #249]** archived — Drive remount and folder check  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

BASE_DIR = (
    "/content/drive/MyDrive/gw_framework/"
    "climate_forcing/Vimal_Mishra_CMIP6/"
    "SSP245_extracted/Alwar_grid_analysis"
)

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))
print("Research directory exists:", os.path.exists(BASE_DIR))
print("Climate indicators directory exists:",
      os.path.exists(os.path.join(BASE_DIR, "Climate_Indicators")))


**[orig #251]** archived — file listing (needs `BASE_DIR` from orig 249)  
<sub>• output not saved in the source notebook</sub>

In [ ]:

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Climate_Indicators",
    "Original_Source_Validation"
)

for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        print(os.path.join(root, file))


**[orig #268]** archived — Step 4H automatic 1 m boundary correction — replaced by the source-preserving audit (orig 272)  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ======================================================
# STEP 4H: LITHOLOGY BOUNDARY CORRECTION
# Maximum automatic correction = 1.00 m
# ======================================================

STEP4 = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/Step4_Source_Verification"
)

INPUT = STEP4 / "11_Parsed_Original_Lithology_Intervals.csv"

OUTPUT = STEP4 / "12_Lithology_Intervals_Corrected_1m.csv"
LOG = STEP4 / "12_Lithology_Correction_Log_1m.csv"
REVIEW = STEP4 / "12_Lithology_Manual_Review_1m.csv"

TOLERANCE = 1.00

# ------------------------------------------------------
# 1. LOAD SOURCE DATA
# ------------------------------------------------------

df = pd.read_csv(INPUT)

df = df.sort_values(
    ["Borehole_ID", "Source_From_m", "Source_To_m"]
).reset_index(drop=True)

# Preserve original source values
df["Original_From_m"] = df["Source_From_m"]
df["Original_To_m"] = df["Source_To_m"]
df["Original_Thickness_m"] = df["Source_Thickness_m"]

df["Corrected_From_m"] = df["Source_From_m"]
df["Corrected_To_m"] = df["Source_To_m"]

df["Correction_Status"] = "UNCHANGED"
df["Correction_Reason"] = ""

# ------------------------------------------------------
# 2. PROCESS EACH BOREHOLE
# ------------------------------------------------------

correction_log = []

for borehole, indices in df.groupby("Borehole_ID").groups.items():

    indices = list(indices)

    previous_bottom = None

    for idx in indices:

        top = float(df.loc[idx, "Original_From_m"])
        bottom = float(df.loc[idx, "Original_To_m"])

        if not np.isfinite(top) or not np.isfinite(bottom):
            df.loc[idx, "Correction_Status"] = "MANUAL_REVIEW"
            df.loc[idx, "Correction_Reason"] = "Missing depth value"
            continue

        if bottom <= top:
            df.loc[idx, "Correction_Status"] = "MANUAL_REVIEW"
            df.loc[idx, "Correction_Reason"] = "Invalid interval"
            continue

        if previous_bottom is not None:

            difference = top - previous_bottom

            # Small gap or overlap up to 1 metre
            if 0 < abs(difference) <= TOLERANCE:

                corrected_top = previous_bottom

                df.loc[idx, "Corrected_From_m"] = corrected_top
                df.loc[idx, "Correction_Status"] = "AUTO_CORRECTED"

                if difference > 0:
                    reason = "Gap closed within 1 m tolerance"
                else:
                    reason = "Overlap removed within 1 m tolerance"

                df.loc[idx, "Correction_Reason"] = reason

                correction_log.append({
                    "Borehole_ID": borehole,
                    "Original_From_m": top,
                    "Original_To_m": bottom,
                    "Previous_Interval_Bottom_m": previous_bottom,
                    "Corrected_From_m": corrected_top,
                    "Difference_m": difference,
                    "Absolute_Adjustment_m": abs(difference),
                    "Reason": reason
                })

            # Large gap or overlap
            elif abs(difference) > TOLERANCE:

                df.loc[idx, "Correction_Status"] = "MANUAL_REVIEW"

                if difference > 0:
                    reason = "Gap greater than 1 m"
                else:
                    reason = "Overlap greater than 1 m"

                df.loc[idx, "Correction_Reason"] = reason

        # Keep original bottom unchanged
        df.loc[idx, "Corrected_To_m"] = bottom

        previous_bottom = bottom

# ------------------------------------------------------
# 3. RECALCULATE CORRECTED THICKNESS
# ------------------------------------------------------

df["Corrected_Thickness_m"] = (
    df["Corrected_To_m"] - df["Corrected_From_m"]
)

df["Thickness_Difference_m"] = (
    df["Original_Thickness_m"] - df["Corrected_Thickness_m"]
)

df["Thickness_Check"] = np.where(
    df["Original_Thickness_m"].isna(),
    "REPORTED_THICKNESS_MISSING",
    np.where(
        np.abs(df["Thickness_Difference_m"]) <= TOLERANCE,
        "WITHIN_1M",
        "REVIEW_THICKNESS"
    )
)

# ------------------------------------------------------
# 4. CREATE LOG AND MANUAL REVIEW TABLE
# ------------------------------------------------------

correction_log = pd.DataFrame(correction_log)

manual_review = df[
    df["Correction_Status"] == "MANUAL_REVIEW"
].copy()

# ------------------------------------------------------
# 5. SAVE ALL OUTPUTS
# ------------------------------------------------------

df.to_csv(OUTPUT, index=False)
correction_log.to_csv(LOG, index=False)
manual_review.to_csv(REVIEW, index=False)

# ------------------------------------------------------
# 6. SUMMARY
# ------------------------------------------------------

print("\n========== CORRECTION SUMMARY ==========")

print("Total intervals:", len(df))

print("\nCorrection status:")
print(df["Correction_Status"].value_counts())

print("\nThickness check:")
print(df["Thickness_Check"].value_counts())

print("\nAutomatic corrections:")
display(correction_log)

print("\nManual review records:")
display(
    manual_review[
        [
            "Borehole_ID",
            "Original_From_m",
            "Original_To_m",
            "Correction_Reason"
        ]
    ]
)

print("\nSaved files:")
print(OUTPUT)
print(LOG)
print(REVIEW)

**[orig #269]** archived — Drive mount only  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**[orig #275]** archived — Drive mount only  
<sub>• output not saved in the source notebook</sub>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**[orig #281]** archived — exact duplicate of orig 280  
<sub>• output not saved in the source notebook</sub>

In [ ]:
import pandas as pd
from pathlib import Path

MODEL = Path(
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar/"
    "Alwar_BDR_Model_Ready/MODFLOW_Model_Input"
)

ELEV_FILE = (
    MODEL / "Step6_Elevation_Validation" /
    "Step6F_Borehole_Lithology_Elevation_Database.csv"
)

HS_FILE = MODEL / "Alwar_Hydrostratigraphic_Intervals.csv"

elev = pd.read_csv(ELEV_FILE)
hs = pd.read_csv(HS_FILE)

print("Elevation database columns:")
print(elev.columns.tolist())

print("\nHydrostratigraphic database columns:")
print(hs.columns.tolist())

# Inspect one record per borehole from elevation database
print("\nElevation database sample:")
display(elev.head(3))

# Summarise hydrostratigraphic depth coverage
depth_summary = (
    hs.groupby(["Borehole_ID", "Dataset"])
    .agg(
        Interval_Count=("Source_Row", "count"),
        Minimum_Depth_m=("Source_From_m", "min"),
        Maximum_Depth_m=("Source_To_m", "max"),
        HSG_Count=("Hydrostratigraphic_Group_ID", "nunique")
    )
    .reset_index()
)

# Identify coordinate/elevation fields available
candidate_fields = [
    c for c in elev.columns
    if any(term in c.lower() for term in [
        "latitude", "longitude", "elevation",
        "confidence", "source", "status"
    ])
]

print("\nCandidate coordinate/elevation fields:")
print(candidate_fields)

print("\nDepth coverage summary:")
display(depth_summary)

print("\nElevation records per borehole:")
display(
    elev.drop_duplicates("Borehole_ID")[
        ["Borehole_ID"] + candidate_fields
    ]
)

**[orig #298]** archived — Drive mount only  
<sub>• output not saved in the source notebook</sub>

In [ ]:

from google.colab import drive

drive.mount('/content/drive')


**[orig #332]** archived — failed (FileNotFoundError); image opened in the next cells  
<sub>⚠ last run ended with FileNotFoundError</sub>

In [ ]:
from PIL import Image
from IPython.display import display
from pathlib import Path

P = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference/"
    "G21F6_Detailed_Planning_Map_Extraction/"
    "NATMO_Alwar_Detailed_Planning_Map_Refined.png"
)

img = Image.open(P)

print("Dimensions:", img.size)
display(img)

**[orig #333]** archived — folder check  
<sub>✅ ran OK</sub>

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

BASE = Path(
    "/content/drive/MyDrive/gw_framework/MODFLOW_Alwar/"
    "Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/NATMO_Reference"
)

print("Folder exists:", BASE.exists())

if BASE.exists():
    print("\nImage files found:")
    for p in BASE.rglob("*"):
        if p.is_file() and p.suffix.lower() in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
            print(p)
else:
    print("NATMO reference folder not found. Check the Drive path.")

**[orig #357]** archived — Drive mount troubleshooting  
<sub>✅ ran OK</sub>

In [ ]:
import os
from google.colab import drive

# Check existing mount
drive_path = "/content/drive"

if os.path.isdir(os.path.join(drive_path, "MyDrive")):
    print("Google Drive is already accessible.")
    DRIVE_ROOT = os.path.join(drive_path, "MyDrive")

else:
    print("Existing /content/drive is not a usable Drive mount.")

    # Use a separate mountpoint; do not delete existing files
    alternate_path = "/content/gdrive"

    os.makedirs(alternate_path, exist_ok=True)

    if os.listdir(alternate_path):
        raise RuntimeError(
            "/content/gdrive is also not empty. "
            "Please share the directory listing."
        )

    drive.mount(alternate_path)
    DRIVE_ROOT = os.path.join(alternate_path, "MyDrive")

print("\nDrive root:", DRIVE_ROOT)
print("Drive accessible:", os.path.exists(DRIVE_ROOT))


**[orig #358]** archived — file check  
<sub>✅ ran OK</sub>

In [ ]:
import os

pdf_path = (
    "/content/drive/MyDrive/gw_framework/"
    "MODFLOW_Alwar/Step7ID4_Regional_Geological_Evidence/"
    "Official_Geological_Maps/"
    "G21F13_Rajasthan_Geological_Reference/"
    "Rajasthan_Aquifer_Mapping_Final_Report.pdf"
)

print("PDF exists:", os.path.isfile(pdf_path))

if os.path.isfile(pdf_path):
    print("File size (MB):", round(os.path.getsize(pdf_path)/1024**2, 2))
else:
    print("Check Drive mount and folder path.")

**[orig #359]** archived — Drive mount troubleshooting  
<sub>✅ ran OK</sub>

In [ ]:
import os

filename = "Rajasthan_Aquifer_Mapping_Final_Report.pdf"

roots = [
    "/content/drive",
    "/content/gdrive"
]

print("GOOGLE DRIVE MOUNT CHECK")
print("=" * 60)

for root in roots:
    print(f"\nChecking: {root}")

    if os.path.isdir(root):
        print("Directory exists")

        mydrive = os.path.join(root, "MyDrive")

        if os.path.isdir(mydrive):
            print("MyDrive exists:", mydrive)
        else:
            print("MyDrive not found")

    else:
        print("Directory does not exist")

# Search likely MyDrive locations
print("\nSEARCHING FOR PDF")
print("=" * 60)

matches = []

for root in roots:
    mydrive = os.path.join(root, "MyDrive")

    if not os.path.isdir(mydrive):
        continue

    for current, dirs, files in os.walk(mydrive):
        dirs[:] = [
            d for d in dirs
            if d not in [".ipynb_checkpoints", "__pycache__"]
        ]

        if filename in files:
            matches.append(os.path.join(current, filename))

print("Matches found:", len(matches))

for path in matches:
    print("\nEXACT PDF PATH:")
    print(path)
    print("Size MB:", round(os.path.getsize(path) / 1024**2, 2))

**[orig #360]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:
import os

filename = "Rajasthan_Aquifer_Mapping_Final_Report.pdf"
search_root = "/content/drive"

matches = []

print("Searching all mounted Google Drive locations...")

for current, dirs, files in os.walk(search_root):

    # Skip system/cache folders
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for f in files:
        if f.lower() == filename.lower():
            matches.append(os.path.join(current, f))

print("\nSearch completed.")
print("Matches:", len(matches))

for path in matches:
    print("\nFOUND:", path)
    print("Size (MB):", round(os.path.getsize(path) / 1024**2, 2))

# Also locate the target folder by name
print("\nChecking for G21F13 folders...")

for current, dirs, files in os.walk(search_root):
    for d in dirs:
        if "G21F13" in d:
            print(os.path.join(current, d))

**[orig #361]** archived — Drive search probe  
<sub>✅ ran OK</sub>

In [ ]:

import os

drive_root = "/content/drive/MyDrive"

target = "Step7ID_Interval_Conceptual_Interpretation.csv"

exact_matches = []
related_matches = []

print("Searching Alwar BDR directories...")

search_roots = [
    os.path.join(drive_root, "gw_framework/data/BDR"),
    os.path.join(drive_root, "gw_framework/MODFLOW_Alwar"),
]

for search_root in search_roots:

    if not os.path.isdir(search_root):
        print("Directory not found:", search_root)
        continue

    for current, dirs, files in os.walk(search_root):

        dirs[:] = [
            d for d in dirs
            if d not in ["__pycache__", ".ipynb_checkpoints"]
        ]

        for filename in files:

            lower = filename.lower()

            if filename == target:
                exact_matches.append(
                    os.path.join(current, filename)
                )

            elif (
                lower.endswith(".csv")
                and (
                    "conceptual" in lower
                    or "interval" in lower
                    or "borehole" in lower
                    or "lithology" in lower
                )
            ):
                related_matches.append(
                    os.path.join(current, filename)
                )

print("\nEXACT MATCHES:", len(exact_matches))

for path in exact_matches:
    print(path)

print("\nRELATED CSV FILES:", len(related_matches))

for path in related_matches[:100]:
    print(path)


**[orig #362]** archived — Drive listing  
<sub>✅ ran OK</sub>

In [ ]:

import os

root = "/content/drive/MyDrive"

print("=" * 65)
print("1. GOOGLE DRIVE ROOT")
print("=" * 65)

if not os.path.isdir(root):
    print("MyDrive is not accessible.")
else:
    for item in os.listdir(root):
        path = os.path.join(root, item)
        print("[DIR] " if os.path.isdir(path) else "[FILE]", item)

print("\n" + "=" * 65)
print("2. SEARCH FOR gw_framework")
print("=" * 65)

framework_paths = []

for current, dirs, files in os.walk("/content/drive"):
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    if "gw_framework" in dirs:
        framework_paths.append(os.path.join(current, "gw_framework"))

for path in framework_paths:
    print(path)

print("\n" + "=" * 65)
print("3. SEARCH FOR ALWAR BDR FOLDERS")
print("=" * 65)

for current, dirs, files in os.walk("/content/drive"):
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for folder in dirs:
        if "Alwar_BDR" in folder or "BDR_Model_Ready" in folder:
            print(os.path.join(current, folder))

print("\nDiagnostic completed.")



**[orig #363]** archived — Drive listing  
<sub>✅ ran OK</sub>

In [ ]:

import os

ROOT = "/content/drive/MyDrive/gw_framework"

print("=" * 65)
print("TOP-LEVEL RESEARCH DIRECTORIES")
print("=" * 65)

for item in sorted(os.listdir(ROOT)):
    path = os.path.join(ROOT, item)
    print("[DIR] " if os.path.isdir(path) else "[FILE]", item)

print("\n" + "=" * 65)
print("SEARCHING ALL CSV FILES")
print("=" * 65)

csv_matches = []

for current, dirs, files in os.walk(ROOT):
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for filename in files:
        lower = filename.lower()

        if lower.endswith(".csv") and any(
            term in lower for term in [
                "step7id",
                "conceptual",
                "interval",
                "lithology",
                "borehole",
                "hydrostratigraphic"
            ]
        ):
            path = os.path.join(current, filename)
            csv_matches.append(path)

for path in csv_matches:
    print(path)

print("\nMatching CSV files:", len(csv_matches))

print("\n" + "=" * 65)
print("SEARCHING FOR SOURCE WORKBOOKS")
print("=" * 65)

for current, dirs, files in os.walk(ROOT):
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for filename in files:
        lower = filename.lower()

        if lower.endswith((".xlsx", ".xls", ".xlsm")):
            if any(term in lower for term in ["alwar", "bdr", "borehole"]):
                print(os.path.join(current, filename))


**[orig #364]** archived — Drive listing  
<sub>✅ ran OK</sub>

In [ ]:

import os

ROOT = "/content/drive/MyDrive"

print("=" * 65)
print("GOOGLE DRIVE — MYDRIVE TOP LEVEL")
print("=" * 65)

for name in sorted(os.listdir(ROOT)):
    path = os.path.join(ROOT, name)
    print("[DIR] " if os.path.isdir(path) else "[FILE]", name)

print("\n" + "=" * 65)
print("MODFLOW_ALWAR CONTENTS")
print("=" * 65)

BASE = os.path.join(ROOT, "gw_framework", "MODFLOW_Alwar")

for current, dirs, files in os.walk(BASE):
    level = current.replace(BASE, "").count(os.sep)
    if level > 4:
        dirs[:] = []
        continue

    print("\n" + "  " * level + os.path.basename(current) + "/")

    for filename in sorted(files):
        print("  " * (level + 1) + filename)

print("\n" + "=" * 65)
print("SEARCH FOR BDR ARCHIVES AND DOCUMENTS")
print("=" * 65)

terms = ["bdr", "atal", "nhp", "alwar", "lithology"]

for current, dirs, files in os.walk(ROOT):
    dirs[:] = [
        d for d in dirs
        if d not in ["__pycache__", ".ipynb_checkpoints"]
    ]

    for filename in files:
        low = filename.lower()

        if low.endswith((".zip", ".docx", ".xlsx", ".csv")):
            if any(term in low for term in terms):
                print(os.path.join(current, filename))


**[orig #365]** archived — folder check  
<sub>✅ ran OK</sub>

In [ ]:
import os

paths = [
    "/content/drive/MyDrive/gw_framework/data",
    "/content/drive/MyDrive/gw_framework/data/BDR",
    "/content/drive/MyDrive/gw_framework/data/BDR/Alwar",
]

for path in paths:
    print("\n", path)
    print("Exists:", os.path.exists(path))
    print("Is directory:", os.path.isdir(path))

    if os.path.isdir(path):
        print("Contents:")
        for item in sorted(os.listdir(path))[:50]:
            print("  ", item)

**[orig #366]** archived — Drive mount troubleshooting  
<sub>✅ ran OK</sub>

In [ ]:

from google.colab import drive
import os

# Mount Google Drive at an alternative location
MOUNT = "/content/gdrive"

os.makedirs(MOUNT, exist_ok=True)

drive.mount(
    MOUNT,
    force_remount=True
)

# Check the actual contents
MYDRIVE = os.path.join(MOUNT, "MyDrive")

print("\nMyDrive accessible:", os.path.isdir(MYDRIVE))

framework = os.path.join(MYDRIVE, "gw_framework")

print("\ngw_framework exists:", os.path.isdir(framework))

if os.path.isdir(framework):
    print("\ngw_framework contents:")
    for item in sorted(os.listdir(framework)):
        print(item)

# Check BDR path
bdr_path = os.path.join(
    framework, "data", "BDR", "Alwar"
)

print("\nBDR Alwar folder exists:", os.path.isdir(bdr_path))

if os.path.isdir(bdr_path):
    print("\nBDR Alwar contents:")
    for item in sorted(os.listdir(bdr_path)):
        print(item)
